# casmi26-vh2-gnx11: private validation harness v2 (never submitted)

gnx11 (the 0.505 line; public sources credited in the cells below) run in full mode on 300 held-out training molecules, with the main engine on the fold-safe FPNet fold0 model (VH_CLEAN_MAIN): 150 natural products without any external reference spectrum (identity fold 0), the 35 enveda-np-examples without external reference, 115 enveda-180 controls, all from the fold that model never saw; every train row of those structures removed. Snapshots in /kaggle/working/vh/.

In [ ]:
# в”Ђв”Ђ Configuration в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
VARIANT = 'CUSTOM'
#   V0  baseline 0.465 answers, no neural term
#   V1  public 0.466 recipe: neural reciprocal-rank fusion, weight 0.25 everywhere
#   V2  leader-safe consensus: rank 1 as V1, ranks 2-25 at neural weight 0.50   (upstream default)
#   V3  V2 + tail rescue: outside candidates may take the last 3 places by neural score
#   V4  bold consensus: rank 1 as V1, ranks 2-25 at neural weight 1.00
#   'CUSTOM' uses the verified 0.477 optimal weights (w_top=0.25, w_mid=0.75, rescue=0)
CUSTOM = dict(w_top=0.25, w_mid=0.75, rescue=0)

RRF_OFFSET = 3.0              # k in 1/(k + rank), as in the public recipe
COMMIT_MODE = 'fast'         # 'smoke': commit run validates 12 visible molecules and asserts all stages;
                              #          the scoring rerun on hidden test executes in full
OUTSIDERS_PER_MOLECULE = 20   # candidate pool for the tail rescue (V3, or CUSTOM with rescue > 0)
TIME_GUARD_HOURS = 7.5        # optional stages are skipped once the session has run this long
QUIET_LOGS = True             # one line per pipeline stage here; full logs stay in /kaggle/working/*.log
ENABLE_TAIL_SLOT = True
PHYSICAL_POST_SECONDS = 1500

# Physical switches the fragment check; PROVENANCE the provenance reranker; neural model: Saner Turhaner v1
PHYSICAL = True
PROVENANCE = False
# Variant M: fragment check reads variants/submission_V2.csv;
# merge cell keeps its rank-1 moves unblocked on the CUSTOM order
FRAG_ON_V2 = True
# InChIKey-14 deduplication: removes redundant stereoisomers within candidate pool
ENABLE_K14_DEDUP = True

# в”Ђв”Ђ Final layer (baidalinadilzhan) в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
# Documentation-prior weights on ranks 2-25 (MID: 0.10 / 0.05 / 0.02), spectral-library bonus only for molecules whose
# best library match reaches LIB_HIT_GATE, the pipeline's leader pinned at rank 2 after a fragment-check leader move,
# metric-exact deduplication (the scorer's own key) and empty slots filled with scored outside candidates.
POP_W, SPEC_W, NP_W = 0.12, 0.12, 0.12
LIB_HIT_GATE = 0.78
PIN_OLD_LEADER = True
OUTSIDER_FILL = True

# ── v8 (romanrozen) ──
V8 = 'MASS'                   # 'MASS' | 'RR' | 'BOTH' | 'BASE' (the Version 22 rule exactly)
assert V8 in ('MASS', 'RR', 'BOTH', 'BASE'), V8
MASS_ON = V8 in ('MASS', 'BOTH')
RR8 = V8 in ('RR', 'BOTH')
MASS_CENTER, MASS_FREE, MASS_FULL, MASS_W = -1.5, 3.0, 8.0, 0.15   # ppm of (exact - neutral)/neutral; above NP_W 0.12
MASS_LEAD_OFF, MASS_LIB_TAU = 5.0, 0.9                              # rank-1 rule: leader beyond 5 ppm, library < 0.9
RR8_ALPHA = 1.0               # weight of the reranker against the list's own order (z-scores)
RR8_LIB_TAU = 0.9
RR8_BUDGET_SECONDS = 1800
RR8_LIMIT_SECONDS = 9 * 3600 - 1800   # never start a step past 8.5 h of session time
RR8_MIN_SECONDS = 600
MASS_PEN, MASS_PPM = {}, {}   # filled by the v8 mass cell: {molecule_id: {smiles: penalty <= 0}}, {..: {smiles: ppm}}


In [ ]:
# в”Ђв”Ђ Session: paths, run mode, design tokens and small display helpers в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
import os, sys, re, io, json, time, math, base64, shutil, hashlib, subprocess, collections, warnings
from pathlib import Path

NB_T0 = time.time()
warnings.filterwarnings('ignore')
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
CODE = ROOT / 'p2s_code'; CODE.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

# Design tokens: chart ink and a fixed categorical order (identity never follows rank).
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, SURFACE, CARD = '#e1e0d9', '#c3c2b7', '#fcfcfb', '#f7f8fb'
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948')
QUIET_GRAY = '#c9c8c1'
ROLE = {'main': BLUE, 'engine2': ORANGE, 'pubchem': AQUA, 'new': VIOLET}
FONT = "system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.size': 10.5, 'axes.titlesize': 12.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlecolor': INK, 'axes.titlepad': 10, 'axes.labelcolor': INK2, 'axes.edgecolor': AXIS,
    'axes.linewidth': 0.8, 'xtick.color': AXIS, 'ytick.color': AXIS, 'xtick.labelcolor': INK2,
    'ytick.labelcolor': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'grid.linestyle': '-', 'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.fontsize': 10, 'figure.dpi': 110, 'savefig.dpi': 160,
    'savefig.bbox': 'tight'})
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass


def tiles(items):
    """A row of stat tiles: items = [(label, value, note), ...]."""
    cells = ''.join(
        f'<div style="display:inline-block;vertical-align:top;min-width:150px;margin:0 10px 10px 0;'
        f'padding:12px 16px;border-radius:12px;background:#ffffff;border:1px solid rgba(11,11,11,0.10);">'
        f'<div style="font-size:12px;color:{INK2};">{label}</div>'
        f'<div style="font-size:25px;font-weight:650;color:{INK};line-height:1.3;">{value}</div>'
        f'<div style="font-size:11.5px;color:{MUTED};">{note}</div></div>' for label, value, note in items)
    display(HTML(f'<div style="font-family:{FONT};margin:4px 0 2px;">{cells}</div>'))


def note(title, text, color=VIOLET):
    display(HTML(f'<div style="font-family:{FONT};border-left:4px solid {color};background:{CARD};'
                 f'padding:10px 14px;border-radius:8px;margin:8px 0;font-size:13.5px;line-height:1.5;">'
                 f'<b style="color:{INK};">{title}</b> <span style="color:{INK2};">{text}</span></div>'))


def table_html(frame, highlight=None, numeric=()):
    """Plain, readable HTML table; numbers right-aligned; `highlight` = index label of a row to emphasise."""
    align = {c: 'right' if c in set(numeric) else 'left' for c in frame.columns}
    head = ''.join(f'<th style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {AXIS};'
                   f'color:{INK2};font-weight:600;background:transparent;">{c}</th>' for c in frame.columns)
    body = []
    for idx, row in frame.iterrows():
        on = highlight is not None and idx == highlight
        style = 'background:#efedfb;font-weight:600;' if on else 'background:transparent;'
        body.append(f'<tr style="{style}">' + ''.join(
            f'<td style="text-align:{align[c]};padding:7px 12px;border-bottom:1px solid {GRID};color:{INK};'
            f'font-variant-numeric:tabular-nums;">{v}</td>' for c, v in zip(frame.columns, row.values)) + '</tr>')
    display(HTML(f'<table style="font-family:{FONT};font-size:13px;border-collapse:collapse;margin:6px 0 10px;">'
                 f'<thead><tr>{head}</tr></thead><tbody>{"".join(body)}</tbody></table>'))


def minutes(seconds):
    return f'{seconds / 60:,.1f} min' if seconds >= 60 else f'{seconds:,.0f} s'


def gpu_name():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout.strip()
        return ' + '.join(x.strip() for x in out.splitlines() if x.strip()) or 'none'
    except Exception:
        return 'none'


# Run mode: the commit run sees the visible format example; the scoring rerun sees the hidden test.
COMP = next((p for p in (INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-CASMI26-molecule-id-mass-spectra',
                         INPUT / 'enveda-casmi26-molecule-id-mass-spectra')
             if all((p / f).is_file() for f in ('train.parquet', 'test.parquet', 'sample_submission.csv'))), None)
assert COMP is not None, 'Attach the competition data (enveda-CASMI26-molecule-id-mass-spectra)'
_ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
del _ids
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
SHOW = not IS_RERUN                      # figures are drawn for readers; the scoring rerun skips them
SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
assert VARIANT in ('V0', 'V1', 'V2', 'V3', 'V4', 'CUSTOM'), VARIANT

GPU = gpu_name()
MODE = 'Scoring rerun' if IS_RERUN else ('Commit В· smoke run' if RUN_PIPELINE else 'Commit В· fast path')
tiles([('Run mode', MODE, 'hidden test' if IS_RERUN else 'visible format example'),
       ('Variant', VARIANT, 'set in the configuration cell'),
       ('Accelerator', GPU.split(' + ')[0] if GPU != 'none' else 'none', GPU if ' + ' in GPU else 'GPU required'),
       ('Python', '.'.join(map(str, sys.version_info[:3])), 'pipeline expects 3.12'),
       ('Answer rows', f'{len(SAMPLE_IDS):,}', 'molecules in sample_submission')])

In [ ]:
# ===== VH (Arsokan private harness, never submitted): held-out synthetic test + reduced train, then redirect =====
# v2: three groups from VH_SEL['groups'] - nphard (natural products with no external reference spectrum, identity fold 0,
# TOF-type query spectra), npref0 (the 35 enveda-np-examples without external reference), e180 (150 controls).
# Inserted right after the session cell. Test = the 250 enveda-np-examples structures (all their np-examples spectra) plus 150
# enveda-180 structures (identity fold 0, seed 20261123; at most VH_CTRL_MAX_SPECTRA spectra in the visible test's adducts).
# Every train row of a held-out inchikey14 (and of plain keys sharing its metric key) leaves the train file the pipeline sees.
# np spectra: those in the visible test's adducts (all of a molecule's spectra if it has none). Redirect: COMP and the run-mode globals of the session cell are recomputed on the synthetic dir; inside this kernel process
# glob / pathlib searches that hit the real competition files return the synthetic ones; the frozen pipeline (subprocesses)
# reads CASMI_VH_COMP (three patched lines). Answers go to /kaggle/working/vh/answers.csv.
import json as _vj, os as _vos, time as _vt
from pathlib import Path as _VP
VH_SEL = _vj.loads(r'''{"max_target":1292.6554,"max_precursor":1291.6481,"mass":{"AEQVPPGEJJBFEE":418.236225,"AEUAVVIPDJTEFS":880.321238,"AIFVFVGRWAMEBM":598.168641,"AKARZXADDNXORH":472.158076,"ALZAZMCIBRHMFF":349.131408,"APCGLAMGCVBTAW":297.151098,"AQASRZOCERRGBL":624.319937,"ARTHABYUQAMJIY":338.163043,"ATBZZQPALSPNMF":285.100108,"AWOHLRJBHMHKEK":265.179027,"BBSPXZBAZKHYLF":458.230453,"BCKRWGIHQHCDDK":310.149284,"BHZCGCMWEVHUBA":429.266779,"BJBUTJQYZDYRMJ":314.079038,"BOYQFGIYXJASCJ":301.131408,"BPCUMWJFAYFREB":278.120132,"BQPRWZCEKZLBHL":246.052823,"BYYDEEAJCDGLER":354.095082,"BZVQHHHEUBCPCO":342.205576,"CBPNZQVSJQDFBE":1029.602486,"CCTOEXCQYMSSED":312.20893,"CEMXHAPUFJOOSV":645.248014,"CEWRXYDCERBVOI":342.205576,"CGTADGCBEXYWNE":965.572523,"CHQPDRGLGJNQTJ":856.37824,"CICPHGOQDCGVPX":291.033207,"CIGNQXROGJKDRV":350.199428,"CJHYKSSBQRABTM":650.402998,"CMDDZMINRWEFHE":260.116092,"CNQCBOHTKBVCFM":452.263485,"CNUMSWIHQSORIU":308.184841,"CNXVIBRFWKALQW":371.148121,"CPZBLNMUGSZIPR":296.188863,"CSHCXOALUXZRIY":345.216475,"CSNDCMIQYXOFFH":299.072449,"CSNLBEBTDFBRHJ":308.173607,"CUJYYVNWNPLKFR":1228.572427,"CWGIHUKKVIGOGZ":294.100442,"CXFDTBRNJQYDLK":360.105647,"DCVVBKWKHIYDMI":298.135114,"DEBDFZGNZTYPMF":234.16198,"DGRWRQNOCZODOH":380.184841,"DHUHPHPCHBYKKF":270.136828,"DKYFESWWIPJYTA":368.162374,"DOFJNFPSMUCECH":388.115818,"DRTKDQCMAUJQEZ":510.173726,"DXYPXVFLRRIOEH":245.177964,"DZBBOCDYINLWCH":518.178812,"DZPAERDUEHFZBZ":289.124883,"FBWWXAGANVJTLU":378.240624,"FGCFGDJTBPYORT":343.189592,"FGZJNOLHDQWUKC":340.07864,"FHKHGNFKBPFJCB":854.466386,"FIHGVQVGHLSXDO":372.135304,"FRJNEAVRLDVWHP":378.06951,"FUQGIYDZGLORRC":270.089209,"FVNMCSSOFLKTDZ":263.138225,"FWDOTPMBXZPVHK":506.084911,"FYJOVKRHRUYSLY":756.211294,"GAOSBIFUUKUHCO":316.133554,"GBGNHQGRWGJVAE":1074.50356,"GBOGMAARMMDZGR":479.267173,"GDSGARQXYUCOQO":342.070799,"GKMPLLYSAVOIMN":245.152812,"GLAAQZFBFGEBPS":330.073953,"GOHHRVCULPSXEU":466.147512,"GOZAGCBVCYGQHP":1292.655369,"GSAWBDXYMSQMRT":604.251977,"GVRNTWSGBWPJGS":334.214409,"HCEPYGIKCDFDPJ":451.220617,"HEURTYMJWQPWNN":292.131074,"HGJXAVROWQLCTP":954.097446,"HHJUWIANJFBDHT":753.410149,"HIQYFEBMHDFUID":682.346546,"HJGURBGBPIKRER":322.120509,"HKVGJQVJNQRJPO":526.168641,"HOACIBQKYRHBOW":159.089543,"HOJGTLHLTUSPKM":318.149204,"HPROWNGJLHZLKK":356.125988,"HPSPCMSCDNHZJM":284.068473,"ICEDBMJSTNTMIT":288.063388,"ICVCJSMZACAHCK":326.06299,"IFIQVSCCFRXSJV":504.345089,"IJXUKFJHONIXSN":334.191738,"IKVJAOUOLWYVJI":359.220892,"IMZOHRNGPZTCSG":384.23006,"IPMKTVRAUBQRFN":391.081362,"IQWXFMQSQNSHKI":312.099774,"IXEBKWWUUASQCF":290.163043,"JBFYFLXEJFQWMU":275.111735,"JCJQVBFSZWANOK":329.210327,"JEJFTTRHGBKKEI":416.219889,"JGXZVDAPLSTBGZ":548.277404,"JGZVRQHKBDUMST":304.158706,"JHJGKOOKRIUVPU":286.204513,"JJGYRVZPUJUFTR":320.235145,"JLKWLPLEMBUFOD":560.116605,"JLQSXXWTCJPCBC":152.058577,"JMBNFQSPXWSBBM":376.136527,"JNKDXWGGWNLKOD":514.168641,"JPOVIFSQRCDPCZ":280.215078,"JQCXWCOOWVGKMT":362.24571,"JQYSYAHSSVSRRA":281.119798,"JUGBMANIHJGWTI":356.196074,"JXDFAGABGALTRF":328.156912,"JYWSARFDLXXOHT":344.147118,"KBLILLDKTZXCRK":432.179755,"KHQMXKRETWUVJU":910.492601,"KHYZTWQWOWXPRD":538.089997,"KKRXDNYRUZGPFM":271.193614,"KOAZOLQZSZBRMW":320.144868,"KOLGVIJQEZBWIQ":288.193674,"KPULLBDIGSBVKN":361.236542,"KQBFNIVRWYGXRN":526.168641,"KQXDHUJYNAXLNZ":750.491813,"KTAAQCYIQXDMEJ":388.189926,"KTBAIBUBSGVINZ":408.23006,"LACFSJYWKBBIGU":364.156912,"LAEVWOPAFHFGGX":288.158626,"LARPFJIXBULVPK":650.402998,"LAUJMTGOOIBBOB":428.147118,"LISLGICORSJAKO":442.235539,"LKVIDBLGZHMPCV":726.200729,"LLQCRTZROWMVOL":262.120509,"LLVDUYLOIPGGSW":364.189926,"LPZUTEQMEUNESW":338.13972,"LTGVIEDOIDWOMN":339.158292,"LTIUKEBFENMBJC":324.158626,"MBIQENSCDNJOIY":118.062994,"MFCCTVWYKKSJFC":318.148061,"MJFCZLNKUOSQHM":329.185175,"MJHNTJSVTSWECS":610.209735,"MKMZAENVDZADSW":313.057605,"MONAOJUXJOXWQB":738.413169,"MRPJBTFHICBFNE":318.073953,"MTVWNRQFFJSBLZ":268.215078,"MXLIDYXKRCYRPL":580.158076,"NDDIECMNVNWXSH":724.358448,"NEEJGFSQMSAACJ":339.180758,"NEPRRZOAYAPZKI":264.158626,"NFMZRMNYRASNPY":1085.577059,"NGRRXMZEVSBATO":336.136159,"NGUSMSQYMDPBGF":622.18977,"NIQQWVLDIKEASA":386.195405,"NLBYRERHXBTBBR":328.26136,"NONWRJXAJGEWOR":904.321238,"NTBUBTCXACOEEC":268.073559,"NTENQBSIKKZPAE":304.189926,"NUFSMWWDLUDJEU":248.163711,"NVRUMGAPYRSOQF":764.234355,"NXSUIALRPVXVTA":238.084124,"NYFIMGDPCRCSAF":312.136159,"NZNHJGNBLPBNLU":680.377177,"ODOQERFSSZYGMQ":295.106925,"OIFNWHZBZDUYNO":318.137971,"OILROCFKEOHUJO":450.110338,"OJOLFAIGOXZBCI":119.988115,"OLDFOHABNQZMKB":378.240624,"OLNOTHHGCUVDPF":354.205576,"OLZAGZCCJJBKNZ":284.089603,"OMWSHVACUVHLON":375.252192,"OMZPVENDYBXXKL":331.153206,"OMZRMXULWNMRAE":322.084124,"OTHMMMVZCOHVOQ":272.077262,"OUWMONOZVGQKME":350.199428,"OXVXZNMLQWQCGM":322.097837,"PBPNOAHYDPHKFH":222.052823,"PEYQJGMLQXTXNP":309.122575,"PIFFQYJYNWXNGE":210.052823,"PLDZOHPIBZVDLQ":527.226765,"PMIVZFNCQVPSBC":341.139386,"PQTZDPVAHXDQQE":315.180758,"QAUDHOGPLBDVAX":424.136947,"QEFFZOVZUUFTOI":257.116427,"QFJCIRLUMZQUOT":913.555142,"QFVPFFXPLPVPJH":334.142976,"QJPCKAJTLHDNCS":390.240624,"QKDHVEHKHGYWMD":302.151809,"QKXGRPHIBLICSC":388.224974,"QPQHCRKTYXPSQG":386.136553,"QQMSZHORHNORLP":347.063084,"QQTZGKRXYRFKIF":430.235539,"QRRXKZXUFXJUSU":338.231791,"QSMRBPGSBKBADN":601.050292,"QUKDLMFYXITHBM":356.125988,"QVSFEOQJKPOOHP":348.23006,"RCFGIEPQSDGMJJ":432.214803,"RGXRCOZTKWBYSQ":359.130363,"RHFUOMFWUGWKKO":259.062677,"RISXUTCDCPHJFQ":386.100168,"RIZQUXKTIFSLTA":305.165188,"RJDSARTZFIBDAR":766.450342,"RLQUGMOZSOFTJC":387.204573,"RLTDMRMWVUCOAI":470.33961,"RNKMOGIPOMVCHO":636.096264,"RPHAAJGBADUATP":412.011658,"RPWYCEBUPDNVLU":350.090272,"RRPVXOQZFWFSKB":250.131742,"RTKCCBCUTAQRDO":325.131408,"RUBLIKRGQGISNL":401.220223,"RUUZDCZRFRLJFC":228.089878,"RUYJAWPWFZNOSA":327.071133,"RYJWSNUDETVRFF":275.115758,"RYWOCBZDEJHTGU":336.158626,"SNGHLUWTFLYPMT":484.173332,"STZYTFJPGGDRJD":517.32258,"SUQFMFHTLHMZNM":570.282883,"SZMGCGMODIEDBW":330.088353,"TUGAUFMQYWZJAB":318.25588,"TUUBGLDJKKCMRH":562.168641,"TVUDIEPQXOLFQE":334.142976,"TWDKBDSVUUKABK":395.126991,"TXCFBEVWILDJSM":282.103813,"TYVPXMVYBHUEFJ":382.161661,"TZRHLKRLEZJVIJ":370.098728,"UARFSULCZRBFHY":314.145284,"UCMRCJDJWPACEX":351.194677,"UDIXYHJHYVDNOG":450.116212,"UPALKCDWIDYZDP":258.029501,"UQKFHRCRUPZUFI":349.21139,"URRIQHWWGFUPHG":352.142307,"USODOIULHQCQHT":284.152478,"UULCYEQSLJMISY":317.148789,"UVBLDLGZDSGCSN":454.344695,"UYRHHBXYXSYGHA":243.162314,"VAKMLCWKWYZBJB":372.252526,"VDNIEQFSLSKIGT":312.195011,"VELFUJGIURLAQV":324.150764,"VEQGXPCBCYUHSO":337.137259,"VGRPJNIVIGBNNK":514.267902,"VJLLAJFALOGHNM":381.121237,"VLFCCQPDWDBSSM":344.017183,"VLUISNOGMYOSEQ":299.138225,"VLUQROFRFLIWCU":373.188923,"VNADFOGBKXRWGC":314.151809,"VQLHKFGYIJUROV":355.200825,"VSNFQQXVMPSASB":245.162708,"VWLSKMQIVFMYAS":355.189592,"WBMOUMRUWNKLGQ":502.22162,"WCNWARYSICKKGF":412.042255,"WEDIEMBZPFJDNC":298.141638,"WGLRJONCGNNMKL":388.115818,"WGMNBGJQXOIVGA":428.168247,"WHVUDFXTBURRCU":334.168128,"WIBZYKZFSQVEHW":371.184506,"WIONIXOBNMDJFJ":199.063329,"WMOWJHFPSNZXSK":330.110338,"WPMZATBVCMJOKV":328.189926,"WQMLFJWIKARBFW":520.303618,"WRALEXHONIJMHH":822.430275,"WRJSPKBWGMBBSI":553.210052,"WROKSYFOSDWFHL":520.230847,"WSBDCEJEINRUML":168.162649,"WSGPLSDARZNMCW":404.131862,"WYKUTTFFZMQCGO":446.084911,"XEVQXKKKAVVSMW":196.109944,"XEZSJABZBMQTAW":338.120132,"XJDJODWHDUVAGF":570.15847,"XJFGLAISLKVAAO":320.235145,"XJLATMLVMSFZBN":385.257671,"XMPDAVDYIOMTLX":248.141244,"XNERWVPQCYSMLC":146.036779,"XNXXZRQPTAQILV":477.262757,"XQPCUYBPEMQWSR":345.047674,"XRWQBDJPMXRDOQ":534.319268,"XSMYYYQVWPZWIZ":369.120382,"XUGSROZUUURBSW":520.209718,"XVMZDZFTCKLZTF":378.121572,"XWUKEZFFRGEMQU":342.132805,"XYPWCJWXFYYGPA":258.089209,"XYRRJNHGKVCNTB":358.200491,"XYYAFLHHHZVPRN":232.109944,"YBRVKGSUHFCMRG":398.192961,"YCDZBVXSGVWFFX":622.444469,"YCUVUDODLRLVIC":456.206245,"YDWLCFYLXHRFSE":379.13322,"YHKYFYCSTJGMID":373.225308,"YJYIPFSCVCVFMF":256.067034,"YLTPWCZXKJSORQ":438.204239,"YQNSSQXYEVZXJO":309.151098,"YRGOYVFPYBBRIV":380.221226,"YRXDKQKYWMHRSY":261.092042,"YTBYWBMZOGLBSF":299.126991,"YZMKFMIZNSOPSN":486.188983,"ZBHUUXLHDOUMKM":390.183109,"ZBXWGKPUSLRPHX":548.152991,"ZCEMXUNJITYHIZ":708.220677,"ZEEMKUSSEDGZFI":289.153875,"ZIANPMCMOVUTCH":319.108754,"ZIOCYJNRYIRTQD":268.167459,"ZJHMRGBCMWLATK":347.140055,"ZMUSCGJNJYXJBP":416.219889,"ZQXBVPNSSGEUCM":962.363103,"ZTHLHHDJRXJGRX":284.068473,"ZUBFQHPQWWXFAC":330.113506,"ZVVZLQBYQRVNQJ":167.094629,"ZWVHTXAYIKBMEE":136.052429,"ZXPRMRWMMPZQED":670.373035,"ZYKANGMCBCQZGW":312.158626,"ZZZVLJDXFSFWDF":374.192961},"mkey":{"AEQVPPGEJJBFEE":"AEQVPPGEJJBFEE","AEUAVVIPDJTEFS":"AEUAVVIPDJTEFS","AIFVFVGRWAMEBM":"AIFVFVGRWAMEBM","AKARZXADDNXORH":"AKARZXADDNXORH","ALZAZMCIBRHMFF":"ALZAZMCIBRHMFF","APCGLAMGCVBTAW":"APCGLAMGCVBTAW","AQASRZOCERRGBL":"AQASRZOCERRGBL","ARTHABYUQAMJIY":"ARTHABYUQAMJIY","ATBZZQPALSPNMF":"ATBZZQPALSPNMF","AWOHLRJBHMHKEK":"AWOHLRJBHMHKEK","BBSPXZBAZKHYLF":"BBSPXZBAZKHYLF","BCKRWGIHQHCDDK":"BCKRWGIHQHCDDK","BHZCGCMWEVHUBA":"BHZCGCMWEVHUBA","BJBUTJQYZDYRMJ":"BJBUTJQYZDYRMJ","BOYQFGIYXJASCJ":"BOYQFGIYXJASCJ","BPCUMWJFAYFREB":"BPCUMWJFAYFREB","BQPRWZCEKZLBHL":"BQPRWZCEKZLBHL","BYYDEEAJCDGLER":"BYYDEEAJCDGLER","BZVQHHHEUBCPCO":"BZVQHHHEUBCPCO","CBPNZQVSJQDFBE":"ZWYQUVBAZLHTHP","CCTOEXCQYMSSED":"CCTOEXCQYMSSED","CEMXHAPUFJOOSV":"QOOXKCQTBVDROE","CEWRXYDCERBVOI":"CEWRXYDCERBVOI","CGTADGCBEXYWNE":"UOZZDEYFDFKHQS","CHQPDRGLGJNQTJ":"FCLHABDDNCSZCO","CICPHGOQDCGVPX":"CICPHGOQDCGVPX","CIGNQXROGJKDRV":"CIGNQXROGJKDRV","CJHYKSSBQRABTM":"CJHYKSSBQRABTM","CMDDZMINRWEFHE":"CMDDZMINRWEFHE","CNQCBOHTKBVCFM":"CNQCBOHTKBVCFM","CNUMSWIHQSORIU":"CNUMSWIHQSORIU","CNXVIBRFWKALQW":"CNXVIBRFWKALQW","CPZBLNMUGSZIPR":"CPZBLNMUGSZIPR","CSHCXOALUXZRIY":"CSHCXOALUXZRIY","CSNDCMIQYXOFFH":"CSNDCMIQYXOFFH","CSNLBEBTDFBRHJ":"CSNLBEBTDFBRHJ","CUJYYVNWNPLKFR":"CUJYYVNWNPLKFR","CWGIHUKKVIGOGZ":"CWGIHUKKVIGOGZ","CXFDTBRNJQYDLK":"CXFDTBRNJQYDLK","DCVVBKWKHIYDMI":"DCVVBKWKHIYDMI","DEBDFZGNZTYPMF":"DEBDFZGNZTYPMF","DGRWRQNOCZODOH":"DGRWRQNOCZODOH","DHUHPHPCHBYKKF":"DHUHPHPCHBYKKF","DKYFESWWIPJYTA":"DKYFESWWIPJYTA","DOFJNFPSMUCECH":"DOFJNFPSMUCECH","DRTKDQCMAUJQEZ":"DRTKDQCMAUJQEZ","DXYPXVFLRRIOEH":"DXYPXVFLRRIOEH","DZBBOCDYINLWCH":"DZBBOCDYINLWCH","DZPAERDUEHFZBZ":"DZPAERDUEHFZBZ","FBWWXAGANVJTLU":"FBWWXAGANVJTLU","FGCFGDJTBPYORT":"FGCFGDJTBPYORT","FGZJNOLHDQWUKC":"FGZJNOLHDQWUKC","FHKHGNFKBPFJCB":"FHKHGNFKBPFJCB","FIHGVQVGHLSXDO":"FIHGVQVGHLSXDO","FRJNEAVRLDVWHP":"FRJNEAVRLDVWHP","FUQGIYDZGLORRC":"FUQGIYDZGLORRC","FVNMCSSOFLKTDZ":"FVNMCSSOFLKTDZ","FWDOTPMBXZPVHK":"FWDOTPMBXZPVHK","FYJOVKRHRUYSLY":"FYJOVKRHRUYSLY","GAOSBIFUUKUHCO":"GAOSBIFUUKUHCO","GBGNHQGRWGJVAE":"GBGNHQGRWGJVAE","GBOGMAARMMDZGR":"GBOGMAARMMDZGR","GDSGARQXYUCOQO":"GDSGARQXYUCOQO","GKMPLLYSAVOIMN":"GKMPLLYSAVOIMN","GLAAQZFBFGEBPS":"GLAAQZFBFGEBPS","GOHHRVCULPSXEU":"GOHHRVCULPSXEU","GOZAGCBVCYGQHP":"GOZAGCBVCYGQHP","GSAWBDXYMSQMRT":"GSAWBDXYMSQMRT","GVRNTWSGBWPJGS":"GVRNTWSGBWPJGS","HCEPYGIKCDFDPJ":"HCEPYGIKCDFDPJ","HEURTYMJWQPWNN":"HEURTYMJWQPWNN","HGJXAVROWQLCTP":"HGJXAVROWQLCTP","HHJUWIANJFBDHT":"HHJUWIANJFBDHT","HIQYFEBMHDFUID":"HIQYFEBMHDFUID","HJGURBGBPIKRER":"HJGURBGBPIKRER","HKVGJQVJNQRJPO":"HKVGJQVJNQRJPO","HOACIBQKYRHBOW":"HOACIBQKYRHBOW","HOJGTLHLTUSPKM":"HOJGTLHLTUSPKM","HPROWNGJLHZLKK":"HPROWNGJLHZLKK","HPSPCMSCDNHZJM":"HPSPCMSCDNHZJM","ICEDBMJSTNTMIT":"ICEDBMJSTNTMIT","ICVCJSMZACAHCK":"ICVCJSMZACAHCK","IFIQVSCCFRXSJV":"IFIQVSCCFRXSJV","IJXUKFJHONIXSN":"IJXUKFJHONIXSN","IKVJAOUOLWYVJI":"IKVJAOUOLWYVJI","IMZOHRNGPZTCSG":"IMZOHRNGPZTCSG","IPMKTVRAUBQRFN":"IPMKTVRAUBQRFN","IQWXFMQSQNSHKI":"IQWXFMQSQNSHKI","IXEBKWWUUASQCF":"IXEBKWWUUASQCF","JBFYFLXEJFQWMU":"JBFYFLXEJFQWMU","JCJQVBFSZWANOK":"JCJQVBFSZWANOK","JEJFTTRHGBKKEI":"JEJFTTRHGBKKEI","JGXZVDAPLSTBGZ":"JGXZVDAPLSTBGZ","JGZVRQHKBDUMST":"JGZVRQHKBDUMST","JHJGKOOKRIUVPU":"JHJGKOOKRIUVPU","JJGYRVZPUJUFTR":"XJFGLAISLKVAAO","JLKWLPLEMBUFOD":"JLKWLPLEMBUFOD","JLQSXXWTCJPCBC":"JLQSXXWTCJPCBC","JMBNFQSPXWSBBM":"JMBNFQSPXWSBBM","JNKDXWGGWNLKOD":"JNKDXWGGWNLKOD","JPOVIFSQRCDPCZ":"JPOVIFSQRCDPCZ","JQCXWCOOWVGKMT":"JQCXWCOOWVGKMT","JQYSYAHSSVSRRA":"JQYSYAHSSVSRRA","JUGBMANIHJGWTI":"JUGBMANIHJGWTI","JXDFAGABGALTRF":"JXDFAGABGALTRF","JYWSARFDLXXOHT":"JYWSARFDLXXOHT","KBLILLDKTZXCRK":"KBLILLDKTZXCRK","KHQMXKRETWUVJU":"ZLOTZUGMLIVSJQ","KHYZTWQWOWXPRD":"KHYZTWQWOWXPRD","KKRXDNYRUZGPFM":"KKRXDNYRUZGPFM","KOAZOLQZSZBRMW":"KOAZOLQZSZBRMW","KOLGVIJQEZBWIQ":"KOLGVIJQEZBWIQ","KPULLBDIGSBVKN":"KPULLBDIGSBVKN","KQBFNIVRWYGXRN":"KQBFNIVRWYGXRN","KQXDHUJYNAXLNZ":"KQXDHUJYNAXLNZ","KTAAQCYIQXDMEJ":"KTAAQCYIQXDMEJ","KTBAIBUBSGVINZ":"KTBAIBUBSGVINZ","LACFSJYWKBBIGU":"LACFSJYWKBBIGU","LAEVWOPAFHFGGX":"LAEVWOPAFHFGGX","LARPFJIXBULVPK":"LARPFJIXBULVPK","LAUJMTGOOIBBOB":"LAUJMTGOOIBBOB","LISLGICORSJAKO":"PSTABKZVTHLQRD","LKVIDBLGZHMPCV":"LKVIDBLGZHMPCV","LLQCRTZROWMVOL":"LLQCRTZROWMVOL","LLVDUYLOIPGGSW":"LLVDUYLOIPGGSW","LPZUTEQMEUNESW":"LPZUTEQMEUNESW","LTGVIEDOIDWOMN":"LTGVIEDOIDWOMN","LTIUKEBFENMBJC":"LTIUKEBFENMBJC","MBIQENSCDNJOIY":"MBIQENSCDNJOIY","MFCCTVWYKKSJFC":"MFCCTVWYKKSJFC","MJFCZLNKUOSQHM":"MJFCZLNKUOSQHM","MJHNTJSVTSWECS":"MJHNTJSVTSWECS","MKMZAENVDZADSW":"MKMZAENVDZADSW","MONAOJUXJOXWQB":"MONAOJUXJOXWQB","MRPJBTFHICBFNE":"GTXZTKMMFNOOMW","MTVWNRQFFJSBLZ":"MTVWNRQFFJSBLZ","MXLIDYXKRCYRPL":"MXLIDYXKRCYRPL","NDDIECMNVNWXSH":"NDDIECMNVNWXSH","NEEJGFSQMSAACJ":"NEEJGFSQMSAACJ","NEPRRZOAYAPZKI":"NEPRRZOAYAPZKI","NFMZRMNYRASNPY":"NFMZRMNYRASNPY","NGRRXMZEVSBATO":"NGRRXMZEVSBATO","NGUSMSQYMDPBGF":"NGUSMSQYMDPBGF","NIQQWVLDIKEASA":"NIQQWVLDIKEASA","NLBYRERHXBTBBR":"NLBYRERHXBTBBR","NONWRJXAJGEWOR":"NONWRJXAJGEWOR","NTBUBTCXACOEEC":"NTBUBTCXACOEEC","NTENQBSIKKZPAE":"NTENQBSIKKZPAE","NUFSMWWDLUDJEU":"NUFSMWWDLUDJEU","NVRUMGAPYRSOQF":"NVRUMGAPYRSOQF","NXSUIALRPVXVTA":"NXSUIALRPVXVTA","NYFIMGDPCRCSAF":"NYFIMGDPCRCSAF","NZNHJGNBLPBNLU":"NZNHJGNBLPBNLU","ODOQERFSSZYGMQ":"ODOQERFSSZYGMQ","OIFNWHZBZDUYNO":"OIFNWHZBZDUYNO","OILROCFKEOHUJO":"OILROCFKEOHUJO","OJOLFAIGOXZBCI":"OJOLFAIGOXZBCI","OLDFOHABNQZMKB":"OLDFOHABNQZMKB","OLNOTHHGCUVDPF":"OLNOTHHGCUVDPF","OLZAGZCCJJBKNZ":"OLZAGZCCJJBKNZ","OMWSHVACUVHLON":"OMWSHVACUVHLON","OMZPVENDYBXXKL":"OMZPVENDYBXXKL","OMZRMXULWNMRAE":"JOFTYVPERBCMPJ","OTHMMMVZCOHVOQ":"OTHMMMVZCOHVOQ","OUWMONOZVGQKME":"OUWMONOZVGQKME","OXVXZNMLQWQCGM":"OXVXZNMLQWQCGM","PBPNOAHYDPHKFH":"PBPNOAHYDPHKFH","PEYQJGMLQXTXNP":"PEYQJGMLQXTXNP","PIFFQYJYNWXNGE":"PIFFQYJYNWXNGE","PLDZOHPIBZVDLQ":"PLDZOHPIBZVDLQ","PMIVZFNCQVPSBC":"PMIVZFNCQVPSBC","PQTZDPVAHXDQQE":"PQTZDPVAHXDQQE","QAUDHOGPLBDVAX":"QAUDHOGPLBDVAX","QEFFZOVZUUFTOI":"QEFFZOVZUUFTOI","QFJCIRLUMZQUOT":"DVSZTYJRXBBIFA","QFVPFFXPLPVPJH":"QFVPFFXPLPVPJH","QJPCKAJTLHDNCS":"QJPCKAJTLHDNCS","QKDHVEHKHGYWMD":"QKDHVEHKHGYWMD","QKXGRPHIBLICSC":"QKXGRPHIBLICSC","QPQHCRKTYXPSQG":"QPQHCRKTYXPSQG","QQMSZHORHNORLP":"QQMSZHORHNORLP","QQTZGKRXYRFKIF":"QQTZGKRXYRFKIF","QRRXKZXUFXJUSU":"QRRXKZXUFXJUSU","QSMRBPGSBKBADN":"JNDZMLHNKOCNMX","QUKDLMFYXITHBM":"QUKDLMFYXITHBM","QVSFEOQJKPOOHP":"QVSFEOQJKPOOHP","RCFGIEPQSDGMJJ":"RCFGIEPQSDGMJJ","RGXRCOZTKWBYSQ":"RGXRCOZTKWBYSQ","RHFUOMFWUGWKKO":"RHFUOMFWUGWKKO","RISXUTCDCPHJFQ":"RISXUTCDCPHJFQ","RIZQUXKTIFSLTA":"RIZQUXKTIFSLTA","RJDSARTZFIBDAR":"RJDSARTZFIBDAR","RLQUGMOZSOFTJC":"RLQUGMOZSOFTJC","RLTDMRMWVUCOAI":"RLTDMRMWVUCOAI","RNKMOGIPOMVCHO":"RNKMOGIPOMVCHO","RPHAAJGBADUATP":"RPHAAJGBADUATP","RPWYCEBUPDNVLU":"RPWYCEBUPDNVLU","RRPVXOQZFWFSKB":"RRPVXOQZFWFSKB","RTKCCBCUTAQRDO":"RTKCCBCUTAQRDO","RUBLIKRGQGISNL":"RUBLIKRGQGISNL","RUUZDCZRFRLJFC":"RUUZDCZRFRLJFC","RUYJAWPWFZNOSA":"RUYJAWPWFZNOSA","RYJWSNUDETVRFF":"RYJWSNUDETVRFF","RYWOCBZDEJHTGU":"RYWOCBZDEJHTGU","SNGHLUWTFLYPMT":"KLWSYWJRTXQLGN","STZYTFJPGGDRJD":"STZYTFJPGGDRJD","SUQFMFHTLHMZNM":"SUQFMFHTLHMZNM","SZMGCGMODIEDBW":"SZMGCGMODIEDBW","TUGAUFMQYWZJAB":"TUGAUFMQYWZJAB","TUUBGLDJKKCMRH":"TUUBGLDJKKCMRH","TVUDIEPQXOLFQE":"TVUDIEPQXOLFQE","TWDKBDSVUUKABK":"TWDKBDSVUUKABK","TXCFBEVWILDJSM":"TXCFBEVWILDJSM","TYVPXMVYBHUEFJ":"TYVPXMVYBHUEFJ","TZRHLKRLEZJVIJ":"TZRHLKRLEZJVIJ","UARFSULCZRBFHY":"UARFSULCZRBFHY","UCMRCJDJWPACEX":"UCMRCJDJWPACEX","UDIXYHJHYVDNOG":"OFQMBMOYAGVFSW","UPALKCDWIDYZDP":"UPALKCDWIDYZDP","UQKFHRCRUPZUFI":"UQKFHRCRUPZUFI","URRIQHWWGFUPHG":"URRIQHWWGFUPHG","USODOIULHQCQHT":"USODOIULHQCQHT","UULCYEQSLJMISY":"UULCYEQSLJMISY","UVBLDLGZDSGCSN":"UVBLDLGZDSGCSN","UYRHHBXYXSYGHA":"UYRHHBXYXSYGHA","VAKMLCWKWYZBJB":"VAKMLCWKWYZBJB","VDNIEQFSLSKIGT":"VDNIEQFSLSKIGT","VELFUJGIURLAQV":"VELFUJGIURLAQV","VEQGXPCBCYUHSO":"VEQGXPCBCYUHSO","VGRPJNIVIGBNNK":"VGRPJNIVIGBNNK","VJLLAJFALOGHNM":"VJLLAJFALOGHNM","VLFCCQPDWDBSSM":"VLFCCQPDWDBSSM","VLUISNOGMYOSEQ":"VLUISNOGMYOSEQ","VLUQROFRFLIWCU":"VLUQROFRFLIWCU","VNADFOGBKXRWGC":"VNADFOGBKXRWGC","VQLHKFGYIJUROV":"VQLHKFGYIJUROV","VSNFQQXVMPSASB":"VSNFQQXVMPSASB","VWLSKMQIVFMYAS":"VWLSKMQIVFMYAS","WBMOUMRUWNKLGQ":"WBMOUMRUWNKLGQ","WCNWARYSICKKGF":"WCNWARYSICKKGF","WEDIEMBZPFJDNC":"WEDIEMBZPFJDNC","WGLRJONCGNNMKL":"WGLRJONCGNNMKL","WGMNBGJQXOIVGA":"WGMNBGJQXOIVGA","WHVUDFXTBURRCU":"WHVUDFXTBURRCU","WIBZYKZFSQVEHW":"WIBZYKZFSQVEHW","WIONIXOBNMDJFJ":"WIONIXOBNMDJFJ","WMOWJHFPSNZXSK":"WMOWJHFPSNZXSK","WPMZATBVCMJOKV":"WPMZATBVCMJOKV","WQMLFJWIKARBFW":"WQMLFJWIKARBFW","WRALEXHONIJMHH":"HDWHBDOGVXCYAX","WRJSPKBWGMBBSI":"WRJSPKBWGMBBSI","WROKSYFOSDWFHL":"WROKSYFOSDWFHL","WSBDCEJEINRUML":"WSBDCEJEINRUML","WSGPLSDARZNMCW":"WSGPLSDARZNMCW","WYKUTTFFZMQCGO":"WYKUTTFFZMQCGO","XEVQXKKKAVVSMW":"XEVQXKKKAVVSMW","XEZSJABZBMQTAW":"XEZSJABZBMQTAW","XJDJODWHDUVAGF":"XJDJODWHDUVAGF","XJFGLAISLKVAAO":"XJFGLAISLKVAAO","XJLATMLVMSFZBN":"XJLATMLVMSFZBN","XMPDAVDYIOMTLX":"XMPDAVDYIOMTLX","XNERWVPQCYSMLC":"XNERWVPQCYSMLC","XNXXZRQPTAQILV":"XNXXZRQPTAQILV","XQPCUYBPEMQWSR":"XQPCUYBPEMQWSR","XRWQBDJPMXRDOQ":"XRWQBDJPMXRDOQ","XSMYYYQVWPZWIZ":"XSMYYYQVWPZWIZ","XUGSROZUUURBSW":"XUGSROZUUURBSW","XVMZDZFTCKLZTF":"XVMZDZFTCKLZTF","XWUKEZFFRGEMQU":"XWUKEZFFRGEMQU","XYPWCJWXFYYGPA":"XYPWCJWXFYYGPA","XYRRJNHGKVCNTB":"XYRRJNHGKVCNTB","XYYAFLHHHZVPRN":"XYYAFLHHHZVPRN","YBRVKGSUHFCMRG":"YBRVKGSUHFCMRG","YCDZBVXSGVWFFX":"YCDZBVXSGVWFFX","YCUVUDODLRLVIC":"YCUVUDODLRLVIC","YDWLCFYLXHRFSE":"YDWLCFYLXHRFSE","YHKYFYCSTJGMID":"MSUDDHFTWMZEIJ","YJYIPFSCVCVFMF":"YJYIPFSCVCVFMF","YLTPWCZXKJSORQ":"YLTPWCZXKJSORQ","YQNSSQXYEVZXJO":"YQNSSQXYEVZXJO","YRGOYVFPYBBRIV":"YRGOYVFPYBBRIV","YRXDKQKYWMHRSY":"YRXDKQKYWMHRSY","YTBYWBMZOGLBSF":"YTBYWBMZOGLBSF","YZMKFMIZNSOPSN":"YZMKFMIZNSOPSN","ZBHUUXLHDOUMKM":"ZBHUUXLHDOUMKM","ZBXWGKPUSLRPHX":"ZBXWGKPUSLRPHX","ZCEMXUNJITYHIZ":"ZCEMXUNJITYHIZ","ZEEMKUSSEDGZFI":"ZEEMKUSSEDGZFI","ZIANPMCMOVUTCH":"ZIANPMCMOVUTCH","ZIOCYJNRYIRTQD":"ZIOCYJNRYIRTQD","ZJHMRGBCMWLATK":"ZJHMRGBCMWLATK","ZMUSCGJNJYXJBP":"ZMUSCGJNJYXJBP","ZQXBVPNSSGEUCM":"ZQXBVPNSSGEUCM","ZTHLHHDJRXJGRX":"ZTHLHHDJRXJGRX","ZUBFQHPQWWXFAC":"ZUBFQHPQWWXFAC","ZVVZLQBYQRVNQJ":"XNABSBUEFUKYDV","ZWVHTXAYIKBMEE":"ZWVHTXAYIKBMEE","ZXPRMRWMMPZQED":"ZXPRMRWMMPZQED","ZYKANGMCBCQZGW":"ZYKANGMCBCQZGW","ZZZVLJDXFSFWDF":"ZZZVLJDXFSFWDF"},"seed":20261124,"n_train_rows":2539608,"vis_adducts":["[M+H]+","[M-H]-","[M+CH2O2-H]-","[M+Na]+","[M+NH4]+","[M+K]+","[M+Cl]-"],"twin_keys":["DVSZTYJRXBBIFA","GTXZTKMMFNOOMW","HMJIYCCIJYRONP","IPWPEUJWMOPJDG","JGMOKGBVKVMRFX","JOFTYVPERBCMPJ","JPMRHWLJLNKRTJ","KLWSYWJRTXQLGN","LGIBRQVYYFOBSF","MSKAFRMBWXTPHY","MSUDDHFTWMZEIJ","OFQMBMOYAGVFSW","PADQINQHPQKXNL","PSTABKZVTHLQRD","QNQBPPQLRODXET","QOOXKCQTBVDROE","RQZAXGRLVPAYTJ","VKHAHZOOUSRJNA","WBGKWQHBNHJJPZ","XLTANAWLDBYGFU","XNABSBUEFUKYDV","YHDPWEWZPGTMMU","ZLOTZUGMLIVSJQ"],"groups":[{"name":"nphard","keys":["AEQVPPGEJJBFEE","AEUAVVIPDJTEFS","AIFVFVGRWAMEBM","AKARZXADDNXORH","ALZAZMCIBRHMFF","AQASRZOCERRGBL","BBSPXZBAZKHYLF","BHZCGCMWEVHUBA","BJBUTJQYZDYRMJ","BQPRWZCEKZLBHL","BYYDEEAJCDGLER","CCTOEXCQYMSSED","CEMXHAPUFJOOSV","CHQPDRGLGJNQTJ","CNQCBOHTKBVCFM","CNXVIBRFWKALQW","CPZBLNMUGSZIPR","CUJYYVNWNPLKFR","CWGIHUKKVIGOGZ","CXFDTBRNJQYDLK","DEBDFZGNZTYPMF","DKYFESWWIPJYTA","DRTKDQCMAUJQEZ","DZBBOCDYINLWCH","FHKHGNFKBPFJCB","FUQGIYDZGLORRC","FWDOTPMBXZPVHK","FYJOVKRHRUYSLY","GBGNHQGRWGJVAE","GLAAQZFBFGEBPS","GOHHRVCULPSXEU","GOZAGCBVCYGQHP","GSAWBDXYMSQMRT","GVRNTWSGBWPJGS","HCEPYGIKCDFDPJ","HEURTYMJWQPWNN","HIQYFEBMHDFUID","HJGURBGBPIKRER","HKVGJQVJNQRJPO","HOACIBQKYRHBOW","HPROWNGJLHZLKK","HPSPCMSCDNHZJM","ICEDBMJSTNTMIT","IMZOHRNGPZTCSG","IQWXFMQSQNSHKI","JBFYFLXEJFQWMU","JEJFTTRHGBKKEI","JGXZVDAPLSTBGZ","JJGYRVZPUJUFTR","JLKWLPLEMBUFOD","JLQSXXWTCJPCBC","JNKDXWGGWNLKOD","JQCXWCOOWVGKMT","JYWSARFDLXXOHT","KBLILLDKTZXCRK","KHQMXKRETWUVJU","KHYZTWQWOWXPRD","KOLGVIJQEZBWIQ","KQBFNIVRWYGXRN","KQXDHUJYNAXLNZ","KTBAIBUBSGVINZ","LARPFJIXBULVPK","LAUJMTGOOIBBOB","LISLGICORSJAKO","LKVIDBLGZHMPCV","MBIQENSCDNJOIY","MJHNTJSVTSWECS","MKMZAENVDZADSW","MONAOJUXJOXWQB","MRPJBTFHICBFNE","MXLIDYXKRCYRPL","NDDIECMNVNWXSH","NFMZRMNYRASNPY","NGRRXMZEVSBATO","NGUSMSQYMDPBGF","NLBYRERHXBTBBR","NONWRJXAJGEWOR","NVRUMGAPYRSOQF","NXSUIALRPVXVTA","NYFIMGDPCRCSAF","NZNHJGNBLPBNLU","OILROCFKEOHUJO","OJOLFAIGOXZBCI","OLDFOHABNQZMKB","OLZAGZCCJJBKNZ","OMZRMXULWNMRAE","PBPNOAHYDPHKFH","PIFFQYJYNWXNGE","PLDZOHPIBZVDLQ","QAUDHOGPLBDVAX","QKDHVEHKHGYWMD","QPQHCRKTYXPSQG","QQMSZHORHNORLP","QSMRBPGSBKBADN","QUKDLMFYXITHBM","QVSFEOQJKPOOHP","RHFUOMFWUGWKKO","RISXUTCDCPHJFQ","RJDSARTZFIBDAR","RLQUGMOZSOFTJC","RLTDMRMWVUCOAI","RPHAAJGBADUATP","RPWYCEBUPDNVLU","RUBLIKRGQGISNL","RUUZDCZRFRLJFC","RYJWSNUDETVRFF","SUQFMFHTLHMZNM","TUGAUFMQYWZJAB","TUUBGLDJKKCMRH","TZRHLKRLEZJVIJ","UDIXYHJHYVDNOG","URRIQHWWGFUPHG","UVBLDLGZDSGCSN","VGRPJNIVIGBNNK","VJLLAJFALOGHNM","VLUQROFRFLIWCU","VNADFOGBKXRWGC","VSNFQQXVMPSASB","WBMOUMRUWNKLGQ","WCNWARYSICKKGF","WEDIEMBZPFJDNC","WGLRJONCGNNMKL","WGMNBGJQXOIVGA","WMOWJHFPSNZXSK","WRALEXHONIJMHH","WRJSPKBWGMBBSI","WROKSYFOSDWFHL","WSBDCEJEINRUML","WSGPLSDARZNMCW","WYKUTTFFZMQCGO","XJDJODWHDUVAGF","XJFGLAISLKVAAO","XMPDAVDYIOMTLX","XNERWVPQCYSMLC","XSMYYYQVWPZWIZ","XUGSROZUUURBSW","XVMZDZFTCKLZTF","XYPWCJWXFYYGPA","YCDZBVXSGVWFFX","YCUVUDODLRLVIC","YLTPWCZXKJSORQ","YRXDKQKYWMHRSY","ZBXWGKPUSLRPHX","ZCEMXUNJITYHIZ","ZIOCYJNRYIRTQD","ZMUSCGJNJYXJBP","ZQXBVPNSSGEUCM","ZVVZLQBYQRVNQJ","ZWVHTXAYIKBMEE","ZXPRMRWMMPZQED"],"libs":null,"inst_re":"(?i)(q-?tof|timstof|maxis|^esi-tof$|^lc-esi-tof$|lc-q-tof)","ce":[20.0,60.0],"min_peaks":5,"cap":4,"max_ppm":5.0},{"name":"npref0","keys":["ATBZZQPALSPNMF","CBPNZQVSJQDFBE","CGTADGCBEXYWNE","CJHYKSSBQRABTM","DOFJNFPSMUCECH","FBWWXAGANVJTLU","GBOGMAARMMDZGR","HGJXAVROWQLCTP","HHJUWIANJFBDHT","IFIQVSCCFRXSJV","KKRXDNYRUZGPFM","LLQCRTZROWMVOL","NTBUBTCXACOEEC","QFJCIRLUMZQUOT","QJPCKAJTLHDNCS","QKXGRPHIBLICSC","QQTZGKRXYRFKIF","RCFGIEPQSDGMJJ","RNKMOGIPOMVCHO","SNGHLUWTFLYPMT","STZYTFJPGGDRJD","TWDKBDSVUUKABK","UPALKCDWIDYZDP","UYRHHBXYXSYGHA","WIONIXOBNMDJFJ","WQMLFJWIKARBFW","XEVQXKKKAVVSMW","XJLATMLVMSFZBN","XNXXZRQPTAQILV","XRWQBDJPMXRDOQ","XYYAFLHHHZVPRN","YHKYFYCSTJGMID","YZMKFMIZNSOPSN","ZBHUUXLHDOUMKM","ZTHLHHDJRXJGRX"],"libs":["enveda-np-examples"],"inst_re":null,"ce":null,"min_peaks":0,"cap":null},{"name":"e180","keys":["APCGLAMGCVBTAW","ARTHABYUQAMJIY","AWOHLRJBHMHKEK","BCKRWGIHQHCDDK","BOYQFGIYXJASCJ","BPCUMWJFAYFREB","BZVQHHHEUBCPCO","CEWRXYDCERBVOI","CICPHGOQDCGVPX","CIGNQXROGJKDRV","CMDDZMINRWEFHE","CNUMSWIHQSORIU","CSHCXOALUXZRIY","CSNDCMIQYXOFFH","CSNLBEBTDFBRHJ","DCVVBKWKHIYDMI","DGRWRQNOCZODOH","DHUHPHPCHBYKKF","DXYPXVFLRRIOEH","DZPAERDUEHFZBZ","FGCFGDJTBPYORT","FGZJNOLHDQWUKC","FIHGVQVGHLSXDO","FRJNEAVRLDVWHP","FVNMCSSOFLKTDZ","GAOSBIFUUKUHCO","GDSGARQXYUCOQO","GKMPLLYSAVOIMN","HOJGTLHLTUSPKM","ICVCJSMZACAHCK","IJXUKFJHONIXSN","IKVJAOUOLWYVJI","IPMKTVRAUBQRFN","IXEBKWWUUASQCF","JCJQVBFSZWANOK","JGZVRQHKBDUMST","JHJGKOOKRIUVPU","JMBNFQSPXWSBBM","JPOVIFSQRCDPCZ","JQYSYAHSSVSRRA","JUGBMANIHJGWTI","JXDFAGABGALTRF","KOAZOLQZSZBRMW","KPULLBDIGSBVKN","KTAAQCYIQXDMEJ","LACFSJYWKBBIGU","LAEVWOPAFHFGGX","LLVDUYLOIPGGSW","LPZUTEQMEUNESW","LTGVIEDOIDWOMN","LTIUKEBFENMBJC","MFCCTVWYKKSJFC","MJFCZLNKUOSQHM","MTVWNRQFFJSBLZ","NEEJGFSQMSAACJ","NEPRRZOAYAPZKI","NIQQWVLDIKEASA","NTENQBSIKKZPAE","NUFSMWWDLUDJEU","ODOQERFSSZYGMQ","OIFNWHZBZDUYNO","OLNOTHHGCUVDPF","OMWSHVACUVHLON","OMZPVENDYBXXKL","OTHMMMVZCOHVOQ","OUWMONOZVGQKME","OXVXZNMLQWQCGM","PEYQJGMLQXTXNP","PMIVZFNCQVPSBC","PQTZDPVAHXDQQE","QEFFZOVZUUFTOI","QFVPFFXPLPVPJH","QRRXKZXUFXJUSU","RGXRCOZTKWBYSQ","RIZQUXKTIFSLTA","RRPVXOQZFWFSKB","RTKCCBCUTAQRDO","RUYJAWPWFZNOSA","RYWOCBZDEJHTGU","SZMGCGMODIEDBW","TVUDIEPQXOLFQE","TXCFBEVWILDJSM","TYVPXMVYBHUEFJ","UARFSULCZRBFHY","UCMRCJDJWPACEX","UQKFHRCRUPZUFI","USODOIULHQCQHT","UULCYEQSLJMISY","VAKMLCWKWYZBJB","VDNIEQFSLSKIGT","VELFUJGIURLAQV","VEQGXPCBCYUHSO","VLFCCQPDWDBSSM","VLUISNOGMYOSEQ","VQLHKFGYIJUROV","VWLSKMQIVFMYAS","WHVUDFXTBURRCU","WIBZYKZFSQVEHW","WPMZATBVCMJOKV","XEZSJABZBMQTAW","XQPCUYBPEMQWSR","XWUKEZFFRGEMQU","XYRRJNHGKVCNTB","YBRVKGSUHFCMRG","YDWLCFYLXHRFSE","YJYIPFSCVCVFMF","YQNSSQXYEVZXJO","YRGOYVFPYBBRIV","YTBYWBMZOGLBSF","ZEEMKUSSEDGZFI","ZIANPMCMOVUTCH","ZJHMRGBCMWLATK","ZUBFQHPQWWXFAC","ZYKANGMCBCQZGW","ZZZVLJDXFSFWDF"],"libs":["enveda-180"],"inst_re":null,"ce":null,"min_peaks":0,"cap":4,"max_ppm":5.0}],"expect":{"rows_removed":13411},"fr_val_in_nphard":25}''')
# engine 2's neutral-mass deltas (pv.ADDUCTS, n = z = 1) for the visible-test adducts
VH_ADDUCT_DELTA = {'[M+H]+': 1.00727645, '[M-H]-': -1.00727645, '[M+CH2O2-H]-': 44.99820285, '[M+Na]+': 22.9892207,
                   '[M+NH4]+': 18.03382555, '[M+K]+': 38.9631581, '[M+Cl]-': 34.96940126}
VH_CLEAN_MAIN = True              # main engine (frozen cell 37) loads fpnet_fold0.pt (never saw fold 0) instead of fpnet_full1
VH_COMP = _VP('/tmp/enveda-casmi26-vh2-mass-spectra')
VH_OUT = _VP('/kaggle/working/vh')
TEST_COLS = ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode',
             'instrument_type', 'precursor_mz', 'collision_energy_orig', 'collision_energy_ev', 'collision_energy_orig_units']


def vh_build(src_dir, comp_dir, out_dir, sel):
    """Write comp_dir/{train,test}.parquet + sample_submission.csv and out_dir/{answers.csv, test.parquet, audit.json}."""
    import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
    t0 = _vt.time()
    comp_dir, out_dir = _VP(comp_dir), _VP(out_dir)
    comp_dir.mkdir(parents=True, exist_ok=True); out_dir.mkdir(parents=True, exist_ok=True)
    groups = sel['groups']
    drop = set().union(*[set(g['keys']) for g in groups]) | set(sel['twin_keys'])
    pf = pq.ParquetFile(str(_VP(src_dir) / 'train.parquet'))
    assert pf.metadata.num_rows == sel['n_train_rows'], (pf.metadata.num_rows, sel['n_train_rows'])
    dset = pa.array(sorted(drop))
    w = pq.ParquetWriter(str(comp_dir / 'train.parquet'), pf.schema_arrow, compression='zstd')
    parts, removed, kept = [], 0, 0
    for g in range(pf.num_row_groups):
        tb = pf.read_row_group(g)
        m = pc.is_in(tb.column('inchikey14'), value_set=dset)
        q = tb.filter(m)
        if q.num_rows:
            parts.append(q.to_pandas())
        removed += q.num_rows
        rest = tb.filter(pc.invert(m))
        kept += rest.num_rows
        if rest.num_rows:
            w.write_table(rest)
        del tb, q, rest
    w.close()
    assert removed == sel['expect']['rows_removed'], (removed, sel['expect'])
    q = pd.concat(parts, ignore_index=True)
    rng = np.random.default_rng(sel['seed'])
    import re as _re

    def cap(df, n):
        if n is None:
            return df
        return df.groupby('inchikey14', group_keys=False).apply(
            lambda x: x.iloc[np.sort(rng.permutation(len(x))[:n])] if len(x) > n else x)
    ce = q.collision_energy_ev.map(lambda v: float(np.mean(v)) if v is not None and len(v) else np.nan)
    _ex = sel['mass']                    # answers' exact masses, computed with RDKit at build time (no RDKit in the image yet)
    ppm = (q.precursor_mz - q.adduct.map(VH_ADDUCT_DELTA) - q.inchikey14.map(_ex)) / q.inchikey14.map(_ex) * 1e6
    parts_t, n_spec = [], {}
    for g in groups:
        x = q[q.inchikey14.isin(set(g['keys']))]
        if g['libs']:
            x = x[x.ingest_lib.isin(g['libs'])]
        m = x.adduct.isin(sel['vis_adducts']) & (x.num_peaks.fillna(0) >= g['min_peaks'])
        if g['inst_re']:
            m &= x.instrument_type.fillna('').str.contains(g['inst_re'], regex=True)
        if g['ce']:
            m &= ce[x.index].between(g['ce'][0], g['ce'][1])
        if g.get('max_ppm') is not None:
            m &= ppm[x.index].abs() <= g['max_ppm']
        x = x[m | (~x.inchikey14.isin(set(x.loc[m, 'inchikey14'])) & (g['name'] == 'npref0'))]
        x = cap(x.sort_values(['inchikey14', 'precursor_mz']).reset_index(drop=True), g['cap'])
        n_spec[g['name']] = int(len(x))
        parts_t.append(x.assign(_grp=g['name']))
    tq = pd.concat(parts_t, ignore_index=True)
    keys = sorted(set(tq.inchikey14))
    assert set(keys) == drop - set(sel['twin_keys']), (len(keys), len(drop))
    hexes = rng.choice(16 ** 6, size=len(keys), replace=False)
    mid = {k: 'm_%06x' % h for k, h in zip(rng.permutation(keys), hexes)}
    tq['molecule_id'] = tq.inchikey14.map(mid)
    sh = rng.choice(16 ** 8, size=len(tq), replace=False)
    tq['spectrum_id'] = ['s_%08x' % h for h in sh]
    tq = tq.sort_values(['molecule_id', 'spectrum_id']).reset_index(drop=True)
    _ls, _ld = pa.large_string(), pa.list_(pa.float64())
    vis_schema = pa.schema([(c, _ld if c in ('ms2_mzs', 'ms2_normalized_intensities', 'collision_energy_ev') else
                             pa.float64() if c in ('base_peak_intensity', 'precursor_mz') else _ls) for c in TEST_COLS])
    test = pa.Table.from_pandas(tq[TEST_COLS], schema=vis_schema, preserve_index=False)
    pq.write_table(test, str(comp_dir / 'test.parquet'))
    ids = sorted(set(tq.molecule_id))
    pd.DataFrame({'molecule_id': ids, 'smiles': ';'.join(['CCO'] * 25)}).to_csv(comp_dir / 'sample_submission.csv', index=False)
    ans = tq.drop_duplicates('molecule_id')[['molecule_id', 'inchikey14', '_grp']].rename(columns={'_grp': 'group'})
    smi = q.drop_duplicates('inchikey14').set_index('inchikey14').normalized_smiles
    ans.insert(2, 'smiles', ans.inchikey14.map(smi).values)
    ans.sort_values('molecule_id').to_csv(out_dir / 'answers.csv', index=False)
    left = set(pq.read_table(str(comp_dir / 'train.parquet'), columns=['inchikey14']).column(0).to_pylist())
    audit = dict(molecules=len(ids), groups=ans.group.value_counts().to_dict(), spectra=n_spec, test_spectra=int(len(tq)),
                 rows_removed=int(removed), rows_kept=int(kept), held_out_keys_left_in_train=len(drop & left), seconds=round(_vt.time() - t0, 1))
    assert audit['held_out_keys_left_in_train'] == 0 and all(audit['groups'].get(g['name']) == len(g['keys']) for g in groups), audit
    (out_dir / 'audit.json').write_text(_vj.dumps(audit, indent=1))
    pq.write_table(test, str(out_dir / 'test.parquet'))
    return audit


def vh_install_redirect(real_comp, vh_comp):
    """In-process: glob.glob / glob.iglob / Path.glob / Path.rglob hits on the real competition files -> the synthetic files."""
    import glob as _g, pathlib as _pl
    real = _vos.path.realpath(str(real_comp)); names = ('train.parquet', 'test.parquet', 'sample_submission.csv')

    def m(p):
        s = str(p)
        if _vos.path.basename(s) in names and _vos.path.realpath(_vos.path.dirname(s)) == real:
            return _vos.path.join(str(vh_comp), _vos.path.basename(s))
        return None

    def remap(items):
        out, seen = [], set()
        for p in items:
            r = m(p)
            q = (r if isinstance(p, str) else type(p)(r)) if r else p
            if str(q) not in seen:
                seen.add(str(q)); out.append(q)
        return out
    if not getattr(_g, '_vh', False):
        _glob, _iglob = _g.glob, _g.iglob
        _g.glob = lambda *a, **k: remap(_glob(*a, **k))
        _g.iglob = lambda *a, **k: iter(remap(_iglob(*a, **k)))
        _g._vh = True
    if not getattr(_pl.Path, '_vh', False):
        _pglob, _prglob = _pl.Path.glob, _pl.Path.rglob
        _pl.Path.glob = lambda self, *a, **k: iter(remap(_pglob(self, *a, **k)))
        _pl.Path.rglob = lambda self, *a, **k: iter(remap(_prglob(self, *a, **k)))
        _pl.Path._vh = True


def vh_snap(tag):
    """Copy of submission.csv at a stage boundary (missing file: nothing)."""
    import shutil
    src = _VP('/kaggle/working/submission.csv')
    VH_OUT.mkdir(parents=True, exist_ok=True)
    if src.is_file():
        shutil.copy2(src, VH_OUT / f'sub_{tag}.csv')
    print('[vh] snapshot', tag, src.is_file(), flush=True)


if 'COMP' in globals() and 'SAMPLE_IDS' in globals():
    VH_REAL_COMP = COMP
    VH_AUDIT = vh_build(VH_REAL_COMP, VH_COMP, VH_OUT, VH_SEL)
    print('[vh] AUDIT', _vj.dumps(VH_AUDIT), flush=True)
    COMP = VH_COMP
    _vos.environ['CASMI_VH_COMP'] = str(VH_COMP)
    _vos.environ['VH_CLEAN_MAIN'] = '1' if VH_CLEAN_MAIN else '0'
    vh_install_redirect(VH_REAL_COMP, VH_COMP)
    _ids = pd.read_parquet(COMP / 'test.parquet', columns=['spectrum_id']).spectrum_id.astype(str)
    TEST_SIGNATURE = hashlib.md5(','.join(sorted(_ids)).encode()).hexdigest()
    del _ids
    IS_RERUN = bool(_vos.getenv('KAGGLE_IS_COMPETITION_RERUN')) or TEST_SIGNATURE != '654e030e4173780e945252b4a5adf70b'
    assert IS_RERUN, 'VH: synthetic test must run in full mode'
    RUN_PIPELINE = IS_RERUN or COMMIT_MODE != 'fast'
    SHOW = not IS_RERUN
    SAMPLE_IDS = pd.read_csv(COMP / 'sample_submission.csv', dtype=str).molecule_id.tolist()
    MODE = 'VH harness (full pipeline on the synthetic test)'
    import glob as _vg
    assert sorted(INPUT.rglob('train.parquet'), key=lambda p: ('mass-spectra' not in str(p).lower(), len(str(p))))[0] == VH_COMP / 'train.parquet'
    assert sorted(_vg.glob(str(INPUT) + '/**/test.parquet', recursive=True), key=lambda p: ('mass-spectra' not in p.lower(), len(p)))[0] == str(VH_COMP / 'test.parquet')
    print('[vh] COMP', COMP, '| IS_RERUN', IS_RERUN, '| molecules', len(SAMPLE_IDS), flush=True)


In [ ]:
# ===== VH pre-flight: abort in the first minutes if any synthetic molecule would be skipped by an engine =====
# No RDKit here (installed later by the pipeline): masses and metric keys come embedded in VH_SEL.
# Run 1 of this harness died after 3 h in frozen cell 48 ("Engine 2 did not cover all active molecules"): engine 2 (and engine 1)
# drop a molecule whose candidate window is empty, i.e. whose neutral-mass target (median over its spectra, engine 2's own
# adduct deltas) misses every pool structure by more than 30 ppm. Checked here, per molecule of the synthetic test:
#   ids of test.parquet == ids of sample_submission.csv; every spectrum's adduct has a neutral-mass rule; >= 1 peak per spectrum;
#   the target lies within 10 ppm of the answer's exact mass (answers.csv; selection used 5 ppm for nphard / e180); the answer is in the COCONUT part of the pool;
#   >= 1 v2-pool structure within 10 ppm of the target.
if 'VH_COMP' in globals() and 'VH_OUT' in globals():
    import numpy as _pf_np, pandas as _pf_pd
    _te = _pf_pd.read_parquet(VH_COMP / 'test.parquet'); _ss = _pf_pd.read_csv(VH_COMP / 'sample_submission.csv', dtype=str)
    _an = _pf_pd.read_csv(VH_OUT / 'answers.csv', dtype=str).set_index('molecule_id')
    _pm_path = sorted(INPUT.rglob('pool_meta.parquet'), key=lambda p: len(str(p)))[0]
    _pm = _pf_pd.read_parquet(_pm_path, columns=['key', 'mass', 'src'])
    _pmass = _pf_np.sort(_pm['mass'].to_numpy(dtype=float)); _coco = set(_pm.loc[_pm['src'] >= 2, 'key'])
    # engine 2's own pool masses (COCONUT / PubChemLite capped at 1,300 Da, ChEBI / LIPID MAPS): >= 1 within 30 ppm is required
    _e2 = []
    for _n in ('coco_mass.npy', 'bio_mass.npy', 'pubchemlite_mass.npy'):
        _h = sorted(INPUT.rglob(_n), key=lambda p: len(str(p)))
        assert _h, 'VH pre-flight: ' + _n + ' not found'
        _e2.append(_pf_np.load(_h[0]))
    _e2 = _pf_np.sort(_pf_np.concatenate(_e2)); _e2 = _e2[_pf_np.isfinite(_e2)]
    assert _te.precursor_mz.max() <= VH_SEL['max_precursor'], ('precursor above the passing envelope', _te.precursor_mz.max())
    _bad = []
    assert set(_te.molecule_id.astype(str)) == set(_ss.molecule_id) == set(_an.index), 'VH pre-flight: id sets differ'
    assert _te.adduct.isin(list(VH_ADDUCT_DELTA)).all(), sorted(set(_te.adduct) - set(VH_ADDUCT_DELTA))
    assert (_te.ms2_mzs.map(len) > 0).all(), 'VH pre-flight: empty spectrum'
    for _mid, _sub in _te.groupby('molecule_id'):
        _nm = (_sub.precursor_mz - _sub.adduct.map(VH_ADDUCT_DELTA)).to_numpy(dtype=float)
        _t = float(_pf_np.median(_nm)); _ik = _an.inchikey14[_mid]; _true = float(VH_SEL['mass'][_ik])
        _ppm = abs(_t - _true) / _true * 1e6
        _n10 = int(_pf_np.searchsorted(_pmass, _t * (1 + 1e-5)) - _pf_np.searchsorted(_pmass, _t * (1 - 1e-5)))
        _key = VH_SEL['mkey'].get(_ik)    # metric key, embedded at build time
        _w30 = int(_pf_np.searchsorted(_e2, _t * (1 + 3e-5)) - _pf_np.searchsorted(_e2, _t * (1 - 3e-5)))
        _np_need = _an.group[_mid] == 'nphard'
        if not (_ppm <= 10.0 and _n10 >= 1 and _w30 >= 1 and _t <= VH_SEL['max_target'] and (not _np_need or _key in _coco or _an.inchikey14[_mid] in _coco)):
            _bad.append(dict(mid=_mid, group=_an.group[_mid], ppm=round(_ppm, 1), pool10=_n10, eng2_w30=_w30, target=round(_t, 3), coconut=_key in _coco or _an.inchikey14[_mid] in _coco))
    print('[vh] pre-flight:', len(_te.molecule_id.unique()), 'molecules,', len(_te), 'spectra, failures', len(_bad), flush=True)
    assert not _bad, f'VH pre-flight failed: {_bad[:10]}'


In [ ]:
# House style for our own figures (engine/33_style.py at repo commit 4ebb73e9, without its ink and grid
# colours, so the upstream tiles and tables keep theirs). A later cell imports another `apply`; ours stays as style_apply.
ACCENT = "#d9480f"      # the one colour that means "this is the point"
SECOND = "#1c7ed6"      # used only when two things must be told apart
PALETTE = [ACCENT, SECOND, "#495057", "#adb5bd", "#2b8a3e", "#862e9c"]


def apply() -> None:
    """Set matplotlib defaults. Safe to call more than once."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (9, 4.6), "figure.dpi": 110, "savefig.dpi": 160,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
        "axes.labelsize": 11, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 10, "ytick.labelsize": 10,
        "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 10,
    })


def label_point(ax, x, y, text: str, color: str = ACCENT) -> None:
    """Print the number that matters on the chart, next to the point it describes."""
    ax.annotate(text, (x, y), xytext=(6, 6), textcoords="offset points", color=color,
                fontsize=11, fontweight="semibold")


def first_screen(claim: str, number: str, beats: str | None = None, runtime: str | None = None,
                 accel: str | None = None) -> str:
    """The markdown every busyaprime notebook opens with. Returns it so a builder can write it into cell 0.

    claim   one sentence: what this notebook shows that the reader did not know
    number  the result, with its metric and where it was measured (public LB, CV on stated folds)
    beats   the public work it beats, with that work's number and a link
    runtime wall clock on the stated accelerator
    """
    rows = [f"| result | **{number}** |"]
    if beats:
        rows.append(f"| beats | {beats} |")
    if runtime:
        rows.append(f"| runtime | {runtime}{' on ' + accel if accel else ''} |")
    return f"{claim}\n\n| | |\n|---|---|\n" + "\n".join(rows) + "\n"
style_apply = apply

In [ ]:
# в”Ђв”Ђ Pinned sources (generated; hidden in the viewer) в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
# MODULES: the six helper modules of the public 0.466 notebooks, unchanged, plus this notebook's
# consensus.py and outsiders.py. FROZEN_B64: the public 0.465 pipeline notebook, byte for byte.
MODULES = {}
MODULES['runner.py'] = '"""Execute the pinned corrected baseline once, then export its final candidates."""\nimport os,sys,json,hashlib\nfrom pathlib import Path\nfrom IPython.display import display\nif __name__==\'__main__\':\n    source=Path(sys.argv[1]); expected=sys.argv[2]; root=Path(sys.argv[3]).resolve()\n    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected, \'Source hash differs\'\n    assert not os.environ.get(\'INC_PANEL\'), \'Replay input is not allowed\'\n    root.mkdir(parents=True,exist_ok=True); os.chdir(root)\n    assert not (root/\'work/spec_meta.parquet\').exists(), \'Fresh pipeline directory required\'\n    assert not (root/\'submission.csv\').exists(), \'Fresh pipeline directory required\'\n    scope={\'__name__\':\'__main__\',\'display\':display}\n    for i,c in enumerate(json.loads(source.read_text())[\'cells\']):\n        if c[\'cell_type\']!=\'code\': continue\n        print(f\'PIPELINE CELL {i} START\',flush=True)\n        exec(compile(\'\'.join(c[\'source\']),f\'pipeline_cell_{i}\',\'exec\'),scope)\n        print(f\'PIPELINE CELL {i} DONE\',flush=True)\n    assert \'candidate_promotion_evidence\' in scope, \'Candidate-specific promotion fix missing\'\n    from export_evidence import export\n    export(scope,root)\n    (root/\'execution_receipt.json\').write_text(json.dumps(dict(source_sha256=expected,\n        mode=\'submission\',source_adaptations=[],candidate_promotion_fix=True,\n        is_full_run=bool(scope[\'IS_RERUN\']),processed_molecules=int(scope[\'te\'].molecule_id.nunique())),indent=2))\n'
MODULES['export_evidence.py'] = '"""Executed in the frozen pipeline process AFTER every original final rule."""\nimport math,json,shutil\nfrom pathlib import Path\nimport pandas as pd\n\ndef export(scope, root):\n    root=Path(root);frame=pd.read_csv(root/\'submission.csv\',dtype=str,keep_default_na=False)\n    shutil.copy2(root/\'submission.csv\',root/\'submission_baseline0465.csv\')\n    processed=set(scope[\'te\'].molecule_id.astype(str));identity=scope[\'metric_identity\']\n    def rank_map(smis):\n        out={}\n        for rank,s in enumerate(smis,1):\n            key=identity(s)\n            if key is not None:out.setdefault(key,rank)\n        return out\n    def finite(v):\n        try:return float(v) if v is not None and math.isfinite(float(v)) else None\n        except (ValueError,TypeError):return None\n    rows=[]\n    for mid,answer in frame.itertuples(index=False):\n        candidates=answer.split(\';\')\n        assert 1<=len(candidates)<=25 and len(candidates)==len(set(candidates)),mid\n        active=mid in processed\n        base=scope[\'BASE\'].get(mid,([],[],0.))\n        maps={\n            \'main\':rank_map(base[0]),\n            \'prefusion\':rank_map(scope.get(\'PREFUSION_LISTS\',{}).get(mid,[])),\n            \'engine2\':rank_map(scope.get(\'ENG\',{}).get(mid,{}).get(\'smiles\',[])),\n            \'pubchem\':rank_map(scope.get(\'PC\',{}).get(mid,{}).get(\'pc\',[]))}\n        ice=scope.get(\'ICE_SCORES\',{}).get(mid,{}) or {}\n        gl=scope[\'gl_of\'](mid) or {}\n        for rank,s in enumerate(candidates,1):\n            ident=identity(s);key=scope[\'chem\'].score_key(s)\n            row=dict(molecule_id=mid,smiles=s,metric_key=ident[0] if ident else None,baseline_rank=rank,\n                processed=active,library_max=finite(base[2]) or 0.,proxy_gl=mid in scope.get(\'N1_GL_MIDS\',set()),\n                ice=finite(ice.get(s,ice.get(key))),gl=finite(gl.get(s,gl.get(key))))\n            row.update({name+\'_rank\':mapping.get(ident,0) for name,mapping in maps.items()});rows.append(row)\n    pd.DataFrame(rows).to_parquet(root/\'frozen_candidate_evidence.parquet\',index=False)\n    # Save the effective query spectra used by the pipeline (12 rows\' molecules in visible smoke, all in rerun).\n    scope[\'te\'].to_parquet(root/\'effective_query_spectra.parquet\',index=False)\n    (root/\'frozen_run_info.json\').write_text(json.dumps(dict(processed_molecules=len(processed),\n        output_molecules=len(frame),is_full_run=bool(scope[\'IS_RERUN\']),\n        note=\'Baseline file saved after the original Engine2 rank-2 insertion; candidate-specific PubChem promotion fix included.\'),indent=2))\n    print(\'EXPORTED FROZEN FINAL25\',len(rows),\'candidate rows\',flush=True)\n'
MODULES['neural_scores.py'] = '"""Runs after the public pipeline process exits, freeing its GPU memory."""\nimport json,sys,importlib.util,types,hashlib\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\n\ndef load_model(bundle,checkpoint,device):\n    # Private module namespace prevents collision with public engines\' model/features modules.\n    package=\'_casmi_own\';p=types.ModuleType(package);p.__path__=[str(bundle)];sys.modules[package]=p\n    for name in [\'features\',\'model\']:\n        spec=importlib.util.spec_from_file_location(package+\'.\'+name,Path(bundle)/(name+\'.py\'))\n        mod=importlib.util.module_from_spec(spec);sys.modules[spec.name]=mod;spec.loader.exec_module(mod)\n    features=sys.modules[package+\'.features\'];cls=sys.modules[package+\'.model\'].SpectrumCandidateModel\n    cfg=json.loads((Path(bundle)/\'config.json\').read_text());model=cls(cfg).to(device)\n    ck=torch.load(Path(bundle)/checkpoint,map_location=device,weights_only=True)\n    model.load_state_dict(ck[\'state_dict\']);model.eval()\n    return model,features,cfg\n\n@torch.inference_mode()\ndef score(bundle,checkpoint,evidence,queries,output,device=\'cuda\'):\n    torch.set_num_threads(4)\n    if device==\'cuda\':assert torch.cuda.is_available(),\'GPU required for neural scoring\'\n    model,feat,cfg=load_model(bundle,checkpoint,device)\n    frame=pd.read_parquet(evidence);q=pd.read_parquet(queries);q[\'molecule_id\']=q.molecule_id.astype(str)\n    query={mid:g for mid,g in q.groupby(\'molecule_id\',sort=False)};cache={};parts=[];errors=[]\n    for index,(mid,g) in enumerate(frame.groupby(\'molecule_id\',sort=False)):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\').copy()\n        g[\'neural_ok\']=False\n        if not g.processed.iloc[0]:\n            for c in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[c]=0.\n            parts.append(g);continue\n        try:\n            assert mid in query,mid\n            records=[]\n            # Use EVERY observed query spectrum; no adduct merging and no new spectrum selection rule.\n            for row in query[mid].itertuples(index=False):\n                try:\n                    pk,n=feat.peaks(row.ms2_mzs,row.ms2_normalized_intensities,row.precursor_mz,cfg[\'max_peaks\'])\n                    if n<cfg[\'min_peaks\']:continue\n                    m,a=feat.metadata(row.precursor_mz,row.adduct,row.ionization_mode,row.collision_energy_ev)\n                    records.append((pk,n,m,a))\n                except (ValueError,TypeError):continue\n            if not records:raise ValueError(\'No usable spectra after the training preprocessing filters\')\n            fp=[]\n            for s in g.smiles:\n                if s not in cache:cache[s]=feat.structure(s,cfg[\'fp_bits\'])[-1]\n                fp.append(cache[s])\n            F=torch.tensor(np.stack(fp),device=device,dtype=torch.float32);M=model.encode_molecules(F)\n            cos_all=[];fp_all=[]\n            for st in range(0,len(records),64):\n                b=records[st:st+64]\n                x=torch.tensor(np.stack([v[0] for v in b]),device=device)\n                lengths=torch.tensor([v[1] for v in b],device=device,dtype=torch.long)\n                meta=torch.tensor(np.stack([v[2] for v in b]),device=device)\n                adduct=torch.tensor([v[3] for v in b],device=device,dtype=torch.long)\n                Z,P=model.encode_spectra(x,lengths,meta,adduct)\n                cos_all.append((Z@M.T).cpu().numpy());fp_all.append((P@F.T/cfg[\'fp_bits\']).cpu().numpy())\n            cos=np.concatenate(cos_all);fps=np.concatenate(fp_all)\n            combined=cos/cfg[\'temperature\']+cfg[\'fp_score_weight\']*fps\n            assert np.isfinite(combined).all(),mid\n            g[\'neural_cos\']=cos.mean(0);g[\'neural_fp\']=fps.mean(0);g[\'neural_score\']=combined.mean(0)\n            g[\'neural_std\']=combined.std(0);g[\'n_spectra\']=len(records);g[\'n_adducts\']=query[mid].adduct.nunique()\n            g[\'neural_ok\']=True\n            parts.append(g)\n        except (ValueError,TypeError) as exc:\n            errors.append(dict(molecule_id=mid,error=str(exc)))\n            for col in [\'neural_cos\',\'neural_fp\',\'neural_score\',\'neural_std\',\'n_spectra\',\'n_adducts\']:g[col]=0.\n            parts.append(g)\n        if (index+1)%25==0:print(\'NEURAL SCORED\',index+1,\'molecules\',flush=True)\n    result=pd.concat(parts,ignore_index=True)\n    assert len(result)==len(frame)\n    result.to_parquet(output,index=False)\n    Path(str(output)+\'.errors.json\').write_text(json.dumps(errors,indent=2))\n    print(\'NEURAL input failures kept at baseline:\',len(errors),flush=True)\n    print(\'NEURAL FEATURES\',result.shape,checkpoint,flush=True)\n\nif __name__==\'__main__\':score(*sys.argv[1:])\n'
MODULES['submission.py'] = 'import json,hashlib\nfrom pathlib import Path\nimport pandas as pd,numpy as np\nfrom ranking import rrf_order\n\ndef apply(evidence,baseline,output,weight=.25,offset=3.):\n    f=pd.read_parquet(evidence);base=pd.read_csv(baseline,dtype=str,keep_default_na=False)\n    groups={mid:g for mid,g in f.groupby(\'molecule_id\',sort=False)};rows=[];audit=[]\n    assert base.molecule_id.is_unique and set(groups)==set(base.molecule_id)\n    for mid,text in base.itertuples(index=False):\n        g=groups[mid].sort_values(\'baseline_rank\',kind=\'stable\');original=text.split(\';\')\n        assert g.smiles.tolist()==original,mid\n        active=bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n        order=rrf_order(g.neural_score.values,weight,offset) if active else np.arange(len(g))\n        new=g.smiles.iloc[order].tolist()\n        assert len(new)==len(original) and set(new)==set(original),mid\n        assert len(new)<=25 and len(set(new))==len(new)\n        rows.append((mid,\';\'.join(new)))\n        audit.append(dict(molecule_id=mid,processed=active,changed=new!=original,\n            changed_top1=new[0]!=original[0],candidate_count=len(new),weight=weight))\n    result=pd.DataFrame(rows,columns=[\'molecule_id\',\'smiles\'])\n    assert result.molecule_id.tolist()==base.molecule_id.tolist()\n    result.to_csv(output,index=False)\n    out=Path(output).parent;pd.DataFrame(audit).to_csv(out/\'neural_rerank_audit.csv\',index=False)\n    report=dict(method=\'baseline rank + neural rank reciprocal-rank fusion AFTER every baseline rule\',\n        neural_weight=weight,rrf_offset=offset,weight_is_validated=False, validation_scope="Fixed-weight submission experiment; no validation performed",\n        candidate_set_preserved=True,changed_rows=sum(x[\'changed\'] for x in audit),\n        changed_top1=sum(x[\'changed_top1\'] for x in audit),rows=len(rows),\n        frozen_sha256=hashlib.sha256(Path(baseline).read_bytes()).hexdigest(),\n        experimental_sha256=hashlib.sha256(Path(output).read_bytes()).hexdigest())\n    (out/\'neural_submission_report.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    return report\n'
MODULES['ranking.py'] = '"""Pure deterministic ranking utilities shared by both notebooks."""\nimport numpy as np\nimport pandas as pd\n\ndef stable_rank(values):\n    values=np.asarray(values,dtype=float)\n    assert np.isfinite(values).all(), \'Nonfinite candidate scores\'\n    order=np.argsort(-values,kind=\'stable\');ranks=np.empty(len(values),int);ranks[order]=np.arange(1,len(values)+1)\n    return ranks\n\ndef rrf_order(neural_scores,weight=0.15,offset=3.0):\n    assert weight>=0 and offset>=0\n    n=len(neural_scores);baseline=np.arange(1,n+1)\n    # Zero weight guarantees exact baseline order even when neural scores are unavailable.\n    if weight==0:return np.arange(n)\n    ranks=stable_rank(neural_scores)\n    return np.argsort(-(1/(offset+baseline)+weight/(offset+ranks)),kind=\'stable\')\n\ndef metrics(frame,score=\'prediction\'):\n    out=[]\n    for mid,g in frame.groupby(\'molecule_id\',sort=False):\n        g=g.sort_values(\'baseline_rank\',kind=\'stable\')\n        order=np.argsort(-g[score].to_numpy(),kind=\'stable\')\n        labels=g.label.to_numpy()[order];where=np.flatnonzero(labels)\n        r=int(where[0]+1) if len(where) else None\n        out.append(dict(molecule_id=mid,mrr25=1/r if r and r<=25 else 0.,top1=int(r==1),\n                        candidate_recall=int(bool(len(where))),rank=r))\n    return pd.DataFrame(out)\n\nBASE_FEATURES=[\'baseline_rr\',\'main_rr\',\'prefusion_rr\',\'engine2_rr\',\'pubchem_rr\',\'engine_agreement\',\n    \'library_max\',\'proxy_gl\',\'ice_missing\',\'gl_missing\',\'ice_z\',\'gl_z\',\n    \'neural_cos_z\',\'neural_fp_z\',\'neural_score_z\',\'neural_rr\',\'neural_std\',\n    \'n_spectra_log\',\'n_adducts\',\'neural_top_gap\',\'neural_agrees_top1\',\'engine2_agrees_top1\',\n    \'neural_delta_rank\',\'neural_x_library\',\'neural_x_agreement\',\'neural_x_sim_missing\',\n    \'main_missing\',\'engine2_missing\',\'pubchem_missing\']\n\ndef make_features(frame):\n    f=frame.copy()\n    for src in [\'baseline\',\'main\',\'prefusion\',\'engine2\',\'pubchem\']:\n        r=pd.to_numeric(f[src+\'_rank\'],errors=\'coerce\');f[src+\'_rr\']=np.where(r>0,1/(3+r),0)\n        if src in [\'main\',\'engine2\',\'pubchem\']:f[src+\'_missing\']=(~(r>0)).astype(float)\n    for src in [\'ice\',\'gl\',\'neural_cos\',\'neural_fp\',\'neural_score\']:\n        v=pd.to_numeric(f[src],errors=\'coerce\');v=v.where(np.isfinite(v))\n        if src in [\'ice\',\'gl\']:f[src+\'_missing\']=v.isna().astype(float)\n        grp=v.groupby(f.molecule_id,sort=False)\n        f[src+\'_z\']=((v-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    f[\'neural_rank\']=0.;f[\'neural_top_gap\']=0.;f[\'neural_agrees_top1\']=0.;f[\'engine2_agrees_top1\']=0.\n    for _,g in f.groupby(\'molecule_id\',sort=False):\n        idx=g.index;rank=stable_rank(g.neural_score.values);f.loc[idx,\'neural_rank\']=rank\n        vals=np.sort(g.neural_score_z.values)[::-1]\n        f.loc[idx,\'neural_top_gap\']=float(vals[0]-vals[1]) if len(vals)>1 else 0.\n        f.loc[idx,\'neural_agrees_top1\']=float(g.iloc[np.argmin(rank)].baseline_rank==1)\n        f.loc[idx,\'engine2_agrees_top1\']=float(((g.baseline_rank==1)&(g.engine2_rank==1)).any())\n    f[\'neural_rr\']=1/(3+f.neural_rank)\n    f[\'engine_agreement\']=((f.main_rank>0).astype(int)+(f.engine2_rank>0).astype(int)+(f.pubchem_rank>0).astype(int))\n    f[\'n_spectra_log\']=np.log1p(f.n_spectra)\n    f[\'neural_delta_rank\']=(f.baseline_rank-f.neural_rank)/25\n    f[\'neural_x_library\']=f.neural_score_z*f.library_max\n    f[\'neural_x_agreement\']=f.neural_score_z*f.engine_agreement\n    f[\'neural_x_sim_missing\']=f.neural_score_z*f.ice_missing*f.gl_missing\n    X=f[BASE_FEATURES].astype(float).replace([np.inf,-np.inf],np.nan).fillna(0.)\n    assert np.isfinite(X.values).all()\n    return X\n\ndef residual_scores(frame,probability,beta):\n    p=np.clip(np.asarray(probability,float),1e-5,1-1e-5)\n    raw=pd.Series(np.log(p/(1-p)),index=frame.index)\n    grp=raw.groupby(frame.molecule_id,sort=False)\n    z=((raw-grp.transform(\'mean\'))/grp.transform(lambda s:s.std(ddof=0)).clip(lower=1e-6)).fillna(0).clip(-5,5)\n    correction=float(beta)*z.values\n    if \'neural_ok\' in frame:correction=np.where(frame.neural_ok.values,correction,0.)\n    return -np.log(frame.baseline_rank.values.astype(float))+correction\n'
MODULES['bundle.py'] = "import hashlib,json,zipfile\nfrom pathlib import Path\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for b in iter(lambda:f.read(8<<20),b''):h.update(b)\n    return h.hexdigest()\n\ndef find_bundle(input_root,output_root,pins):\n    roots=[]\n    for path in Path(input_root).rglob('final_model.pt'):\n        if (path.parent/'config.json').is_file():roots.append(path.parent)\n    if not roots:\n        archives=list(Path(input_root).rglob('casmi_spectrum_model_v1.zip'))\n        assert len(archives)==1, 'Attach the supplied casmi_spectrum_model_v1.zip as a Kaggle dataset (zipped or extracted)'\n        dest=Path(output_root)/'uploaded_model';dest.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for item in z.infolist():\n                target=(dest/item.filename).resolve()\n                assert target.is_relative_to(dest.resolve()),'Unsafe ZIP member'\n            z.extractall(dest)\n        roots=[p.parent for p in dest.rglob('final_model.pt')]\n    matching=[r for r in roots if all((r/n).is_file() and sha(r/n)==h for n,h in pins.items())]\n    assert len(matching)==1, f'Expected exactly one matching model bundle, found {len(matching)}. Use the attached v1 artifacts.'\n    return matching[0]\n"
MODULES['consensus.py'] = '"""Leader-safe consensus re-ranking for CASMI26 answer lists.\n\nRuns after the frozen 0.465 pipeline and after the neural scorer. It never reruns a\nmodel: it reorders the candidates of each molecule and, only when asked, lets a few\ncandidates that the pipeline left outside its list compete for the last places.\n\nOrder rule\n    rank 1   = leader of   1/(k + b) + w_top / (k + n)\n    ranks 2+ = the others, ordered by   1/(k + b) + w_mid / (k + n)\nwith b the baseline rank, n the neural rank among the same candidates and k the\noffset. With w_top == w_mid this is exactly the fixed-weight reciprocal-rank fusion\nof the public 0.466 notebook (``ranking.rrf_order``), which build_variants re-checks.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom ranking import rrf_order, stable_rank\n\nPRESETS = {\n    \'V0\': dict(w_top=0.00, w_mid=0.00, rescue=0, label=\'Baseline 0.465 - no neural term\'),\n    \'V1\': dict(w_top=0.25, w_mid=0.25, rescue=0, label=\'Neural RRF 0.25 - public 0.466 recipe\'),\n    \'V2\': dict(w_top=0.25, w_mid=0.50, rescue=0, label=\'Leader-safe consensus 0.25 / 0.50\'),\n    \'V3\': dict(w_top=0.25, w_mid=0.50, rescue=3, label=\'Leader-safe consensus + 3-slot tail rescue\'),\n    \'V4\': dict(w_top=0.25, w_mid=1.00, rescue=0, label=\'Bold consensus 0.25 / 1.00\'),\n}\n\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef leader_safe_order(neural_scores, w_top, w_mid, offset=3.0):\n    """Indices of the new order (baseline order = 0..n-1)."""\n    s = np.asarray(neural_scores, dtype=float)\n    if len(s) == 0:\n        return np.zeros(0, dtype=int)\n    first = np.asarray(rrf_order(s, w_top, offset), dtype=int)\n    if w_mid == w_top:\n        return first\n    lead = int(first[0])\n    rest = [int(i) for i in rrf_order(s, w_mid, offset) if int(i) != lead]\n    return np.asarray([lead] + rest, dtype=int)\n\n\ndef rescue_tail(smiles, scores, outsiders, slots, limit=25):\n    """Let outside candidates compete for the last `slots` places by neural score.\n\n    smiles, scores: the reordered list and its neural scores.\n    outsiders: [(smiles, neural_score)] whose metric identities differ from every\n    listed candidate and from each other (guaranteed by outsiders.py).\n    Free places of a list shorter than `limit` are offered as well.\n    Returns (new_list, number_of_outsiders_admitted).\n    """\n    smiles = list(smiles)\n    scores = [float(v) for v in scores]\n    if slots <= 0 or not outsiders or not smiles:\n        return smiles, 0\n    k = min(int(slots), len(smiles))\n    free = max(0, int(limit) - len(smiles))\n    head = smiles[:len(smiles) - k]\n    pool = [(s, v, 0) for s, v in zip(smiles[len(smiles) - k:], scores[len(scores) - k:])]\n    pool += [(str(s), float(v), 1) for s, v in outsiders if np.isfinite(float(v)) and str(s) not in smiles]\n    pool.sort(key=lambda t: (-t[1], t[2]))  # stable; on ties a listed candidate keeps its place\n    chosen = pool[:k + free]\n    out = (head + [s for s, _, _ in chosen])[:limit]\n    return out, int(sum(t[2] for t in chosen))\n\n\ndef read_outsiders(path):\n    out = {}\n    if not path or not Path(path).is_file():\n        return out\n    frame = pd.read_parquet(path)\n    if frame.empty or \'neural_ok\' not in frame.columns:\n        return out\n    score = pd.to_numeric(frame.neural_score, errors=\'coerce\')\n    frame = frame[frame.neural_ok.astype(bool) & np.isfinite(score)]\n    for mid, g in frame.groupby(\'molecule_id\', sort=False):\n        g = g.sort_values(\'baseline_rank\', kind=\'stable\')\n        out[str(mid)] = list(zip(g.smiles.astype(str), g.neural_score.astype(float)))\n    return out\n\n\ndef build_variants(evidence, baseline, outdir, presets=None, offset=3.0, outsiders=None):\n    """Write submission_<name>.csv for every preset; return (report, moves)."""\n    presets = dict(PRESETS if presets is None else presets)\n    frame = pd.read_parquet(evidence)\n    base = pd.read_csv(baseline, dtype=str, keep_default_na=False)\n    assert list(base.columns) == [\'molecule_id\', \'smiles\'] and base.molecule_id.is_unique\n    groups = {str(m): g.sort_values(\'baseline_rank\', kind=\'stable\') for m, g in frame.groupby(\'molecule_id\', sort=False)}\n    assert set(groups) == set(base.molecule_id), \'evidence and baseline cover different molecules\'\n    outs = read_outsiders(outsiders)\n    outdir = Path(outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    report, moves = {}, []\n    for name, cfg in presets.items():\n        rows = []\n        stats = dict(active_rows=0, changed_rows=0, changed_top1=0, rescued=0)\n        for mid, text in base.itertuples(index=False):\n            original = text.split(\';\')\n            g = groups[mid]\n            assert g.smiles.tolist() == original, f\'evidence order differs from the baseline: {mid}\'\n            active = bool(g.processed.iloc[0]) and bool(g.neural_ok.all())\n            if active:\n                order = leader_safe_order(g.neural_score.values, cfg[\'w_top\'], cfg[\'w_mid\'], offset)\n            else:\n                order = np.arange(len(g))\n            new = g.smiles.iloc[order].tolist()\n            new_scores = g.neural_score.astype(float).values[order].tolist()\n            rescued = 0\n            if active and int(cfg.get(\'rescue\', 0)) > 0 and outs.get(mid):\n                new, rescued = rescue_tail(new, new_scores, outs[mid], int(cfg[\'rescue\']))\n            assert 1 <= len(new) <= 25 and len(set(new)) == len(new), mid\n            if not rescued:\n                assert sorted(new) == sorted(original), f\'candidate set changed: {mid}\'\n            stats[\'active_rows\'] += int(active)\n            stats[\'changed_rows\'] += int(new != original)\n            stats[\'changed_top1\'] += int(new[0] != original[0])\n            stats[\'rescued\'] += int(rescued)\n            rows.append((mid, \';\'.join(new)))\n            if active:\n                b_rank = {s: i + 1 for i, s in enumerate(original)}\n                n_rank = dict(zip(g.smiles, stable_rank(g.neural_score.values)))\n                for pos, s in enumerate(new, 1):\n                    moves.append((name, mid, s, pos, b_rank.get(s, 0), int(n_rank.get(s, 0))))\n        out = pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\'])\n        assert out.molecule_id.tolist() == base.molecule_id.tolist()\n        path = outdir / f\'submission_{name}.csv\'\n        out.to_csv(path, index=False)\n        report[name] = dict(cfg, offset=offset, file=path.name, sha256=sha256(path), **stats)\n    moves = pd.DataFrame(moves, columns=[\'variant\', \'molecule_id\', \'smiles\', \'final_rank\', \'baseline_rank\', \'neural_rank\'])\n    (outdir / \'consensus_report.json\').write_text(json.dumps(report, indent=2))\n    return report, moves\n\n\ndef validate(path, sample_ids, baseline_path=None, allow_new=False, identity_of=None):\n    """Raise unless `path` is a well-formed answer file for the expected molecules."""\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    assert list(frame.columns) == [\'molecule_id\', \'smiles\'], \'columns\'\n    assert frame.molecule_id.tolist() == list(sample_ids), \'molecule order\'\n    lists = frame.smiles.str.split(\';\')\n    assert lists.map(len).between(1, 25).all(), \'list length\'\n    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), \'empty or repeated SMILES\'\n    if baseline_path is not None:\n        base = pd.read_csv(baseline_path, dtype=str, keep_default_na=False)\n        assert base.molecule_id.tolist() == frame.molecule_id.tolist(), \'baseline order\'\n        for before, after in zip(base.smiles, frame.smiles):\n            b, a = before.split(\';\'), after.split(\';\')\n            if allow_new:\n                assert len(a) >= min(len(b), 25), \'list got shorter\'\n            else:\n                assert sorted(b) == sorted(a), \'candidate set changed\'\n    if identity_of is not None:\n        for answer in frame.smiles:\n            keys = [identity_of.get(s) for s in answer.split(\';\')]\n            keys = [k for k in keys if k]\n            assert len(keys) == len(set(keys)), \'duplicate metric identity\'\n    return frame\n'
MODULES['outsiders.py'] = '"""Collect candidates that the frozen pipeline considered but left outside its final list.\n\nReads the pipeline\'s own record of its candidate lists (e004_candidate_inputs.json),\nwalks the fusion overflow, Engine 2, the main-engine tail and the PubChem channel in\nround-robin rank order, and keeps structures whose official metric identity (the frozen\nnotebook\'s own metric_identity, executed from its source) is new for the molecule.\nOutput rows follow the evidence schema the neural scorer reads; baseline_rank 26, 27, ...\nonly fixes their order.\n\nusage: python outsiders.py <frozen.ipynb> <root> <evidence.parquet> <out.parquet> [per_molecule] [max_checks]\n"""\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOLUMNS = [\'molecule_id\', \'smiles\', \'metric_key\', \'source\', \'source_rank\', \'baseline_rank\', \'processed\']\n\n\ndef load_identity(frozen):\n    cells = json.loads(Path(frozen).read_text())[\'cells\']\n    hits = [\'\'.join(c[\'source\']) for c in cells\n            if c[\'cell_type\'] == \'code\' and \'def metric_identity\' in \'\'.join(c[\'source\'])]\n    assert len(hits) == 1, \'metric identity cell not found in the frozen source\'\n    scope = {\'__name__\': \'frozen_metric_identity\'}\n    exec(compile(hits[0], \'frozen_metric_identity\', \'exec\'), scope)\n    return scope[\'metric_identity\']\n\n\ndef source_lists(payload, mid):\n    eng = payload.get(\'engine2\', {}).get(mid) or {}\n    pc = payload.get(\'pubchem\', {}).get(mid) or {}\n    return [(\'fusion_overflow\', list(payload.get(\'fusion_overflow\', {}).get(mid) or [])),\n            (\'engine2\', list(eng.get(\'smiles\') or [])),\n            (\'main_tail\', list(payload.get(\'base_tail\', {}).get(mid) or [])),\n            (\'pubchem\', list(pc.get(\'pc\') or []))]\n\n\ndef collect(payload, evidence, identity, per_molecule=12, max_checks=30):\n    rows = []\n    active = evidence[evidence.processed.astype(bool)]\n    for mid, g in active.groupby(\'molecule_id\', sort=False):\n        mid = str(mid)\n        if mid not in payload.get(\'prefusion\', {}):\n            continue\n        listed = g.sort_values(\'baseline_rank\', kind=\'stable\').smiles.astype(str).tolist()\n        taken = {k for k in g.metric_key.tolist() if isinstance(k, str) and k}\n        seen = set(listed)\n        lists = source_lists(payload, mid)\n        depth = max((len(v) for _, v in lists), default=0)\n        picked = checks = 0\n        for r in range(depth):\n            if picked >= per_molecule or checks >= max_checks:\n                break\n            for name, values in lists:\n                if r >= len(values) or picked >= per_molecule or checks >= max_checks:\n                    continue\n                s = str(values[r]).strip()\n                if not s or s == \'CCO\' or s in seen:\n                    continue\n                seen.add(s)\n                checks += 1\n                key = identity(s)\n                if key is None or key[0] in taken:\n                    continue\n                taken.add(key[0])\n                picked += 1\n                rows.append((mid, s, key[0], name, r + 1, 25 + picked, True))\n    return pd.DataFrame(rows, columns=COLUMNS)\n\n\nif __name__ == \'__main__\':\n    frozen, root, evidence_path, out = sys.argv[1:5]\n    per = int(sys.argv[5]) if len(sys.argv) > 5 else 12\n    checks = int(sys.argv[6]) if len(sys.argv) > 6 else 30\n    t0 = time.time()\n    identity = load_identity(frozen)\n    payload = json.loads((Path(root) / \'e004_candidate_inputs.json\').read_text())\n    evidence = pd.read_parquet(evidence_path)\n    frame = collect(payload, evidence, identity, per, checks)\n    frame.to_parquet(out, index=False)\n    by_source = frame.source.value_counts().to_dict() if len(frame) else {}\n    print(\'OUTSIDERS\', len(frame), \'candidates for\', frame.molecule_id.nunique(), \'molecules\',\n          json.dumps(by_source), f\'{time.time() - t0:.0f}s\', flush=True)\n'
MODULE_SHA256 = {
 "runner.py": "86b74f3b4cd5767a2cbe1d291735c292b66b4fbf286a705c6cb777a1865883db",
 "export_evidence.py": "d4ec36a9f6216d2607fff4f481bc465dbc8fbc616ec4b24f12932e4c5e546e19",
 "neural_scores.py": "cb9e4d6732992e3310e951533fd0c644d3b515134ad03768c35e5a86551267d2",
 "submission.py": "d0027a893b1bfefe8066ede4f625c584fe0d4fa16bfe4bee32265ef34a7f007c",
 "ranking.py": "6f909cd7e41e88387c23109b93dd590885d2283509804279a58c624a758987d7",
 "bundle.py": "8595ef16ead9c9e6ac34a3398bff4b748f5668cdc517c396be18100a5b745cea",
 "consensus.py": "9e67ab49a852404587e62ca84e0fc551d70ce68ec8f7a47739aada6ca08d3989",
 "outsiders.py": "4dcf6c2b9c9b7be3ac8f7a616683495430fcb4e0c25b449684d132daaa8e7b5c"
}
MODEL_PINS = {
 "features.py": "a97ad9b3ab3711517864d56e497e1672cb29bb05683c72b52db9f9b0a17b1455",
 "model.py": "2547418857386d1c2513fd1ab6c70961e6a96e73d9d495bbfe3052197e69dc39",
 "config.json": "563fb2a2a81ca48ea094f9ab78c157a48a6e9a5503e85e6ce8f0fb3615c1cc03",
 "final_model.pt": "44436b2daec64373e66f9120dd2389c24fc83dd9430ebeeac18e3cb49dcb9642"
}
FROZEN_SHA256 = '59c72af49c54b5cfc9ce2b92cef0d1eb2a7e9773d9e24815a2cd3fd3b6ce9401'
FROZEN_B64 = '...'
print(f'{len(MODULES)} modules and the pinned pipeline source ready ({len(FROZEN_B64) * 3 // 4 / 1e6:.2f} MB)')

In [ ]:
# в”Ђв”Ђ Stage machinery: process runner, compact trace, and the four stage functions в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
import contextlib, importlib

STAGES = []           # (stage, seconds, status) for the run summary
STAGE_MARKS = {}      # stage -> [(frozen cell, START/DONE, seconds since stage start)]
FROZEN_STAGE = {
    2: 'Official RDKit 2026.03.3', 6: 'Competition files', 9: 'Asset inventory', 12: 'Documentation prior table',
    14: 'Prior benchmark table', 16: 'Natural-product panel', 18: 'Natural-product knowledge table',
    26: 'Datasets, priors and wheels', 27: 'Determinism patch', 28: 'Preflight', 30: 'Commit or rerun switch',
    32: 'Engine 2 В· two rankers', 34: 'PubChem channel', 36: 'Engine 1 В· library, pool, spectrum cache',
    37: 'Engine 1 В· models and rankers', 39: 'Engine 1 В· candidate lists', 41: 'GLACIER + ICEBERG forward models',
    43: 'Gated PubChem merge', 45: 'Two-engine fusion', 48: 'Metric audit and safe tail', 49: 'Metric-identity fusion',
    50: 'Postflight record', 51: 'Fragment score gap', 52: 'Window-40 overlay', 53: 'Engine 2 rank-2 placement',
    54: 'Execution receipt', 56: 'Example answer table'}
CELL_MARK = re.compile(r'^PIPELINE CELL (\d+) (START|DONE)')
KEY_LINE = re.compile(r'FAILED|ERROR|Traceback|merge stats|RANK2_OVERLAY|EXPORTED FROZEN|NEURAL FEATURES|'
                      r'NEURAL input failures|OUTSIDERS|engine lists \d+ |pubchem lists \d+|errors \d+ \|')
BASELINE = ROOT / 'submission_baseline0465.csv'
VARIANTS_DIR = ROOT / 'variants'
RESCUE_SLOTS = {'V3': 3, 'CUSTOM': int(CUSTOM.get('rescue', 0))}.get(VARIANT, 0)
PIPELINE_DONE = NEURAL_OK = OUTSIDERS_OK = CONSENSUS_OK = False
REPORT, MOVES, SUBMITTED, consensus = {}, pd.DataFrame(), None, None


def run_stage(args, log, name, quiet=None):
    """Run one stage in its own process; keep the full log on disk and print a compact trace."""
    quiet = QUIET_LOGS if quiet is None else quiet
    env = {**os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    t0, marks, tail = time.time(), [], collections.deque(maxlen=150)
    print(f'в–¶ {name}', flush=True)
    with open(log, 'w') as handle:
        proc = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            handle.write(line); handle.flush()
            t = time.time() - t0
            tail.append(line)
            m = CELL_MARK.match(line)
            if m:
                cell, kind = int(m.group(1)), m.group(2)
                marks.append((cell, kind, t))
                if quiet and kind == 'DONE' and cell in FROZEN_STAGE:
                    start = next((s for c, k, s in reversed(marks) if c == cell and k == 'START'), t)
                    if t - start >= 1:
                        print(f'  вњ“ {t / 60:6.1f} min   {FROZEN_STAGE[cell]:<40s} {t - start:8,.0f} s', flush=True)
                continue
            if not quiet:
                if len(line) < 5000:
                    print(line, end='', flush=True)
            elif KEY_LINE.search(line) and len(line) < 800:
                print('      ' + line.strip()[:180], flush=True)
        status = proc.wait()
    seconds = time.time() - t0
    STAGE_MARKS[name] = marks
    STAGES.append((name, seconds, 'ok' if status == 0 else f'exit {status}'))
    if status:
        print(''.join(tail))
        raise RuntimeError(f'{name} failed with exit code {status}; full log: {log}')
    print(f'  done in {minutes(seconds)}', flush=True)
    return seconds


def materialise_sources():
    """Stage 0: write and verify the pinned sources, find the neural bundle, install the official RDKit."""
    global FROZEN
    t0 = time.time()
    for name, text in MODULES.items():
        (CODE / name).write_text(text)
        assert hashlib.sha256((CODE / name).read_bytes()).hexdigest() == MODULE_SHA256[name], name
    FROZEN = CODE / 'frozen_0465.ipynb'
    FROZEN.write_bytes(base64.b64decode(FROZEN_B64))
    assert hashlib.sha256(FROZEN.read_bytes()).hexdigest() == FROZEN_SHA256, 'Pinned pipeline source differs'
    assert sys.version_info[:2] in ((3, 12), (3, 13)), 'Python 3.12 or 3.13 is required'
    if str(CODE) not in sys.path:
        sys.path.insert(0, str(CODE))
    from bundle import find_bundle
    from pathlib import Path as _BPath
    def _find_bundle_safe():
        try:
            return find_bundle(str(INPUT), ROOT, MODEL_PINS)
        except AssertionError:
            _cands = [p.parent for p in _BPath('/kaggle/input').rglob('final_model.pt')
                      if (p.parent / 'config.json').is_file() and 'casmi-spectrum-model-v1' in str(p)]
            print('BUNDLE FALLBACK candidates', len(_cands), flush=True)
            assert _cands, 'public model bundle not found'
            return str(sorted(_cands, key=lambda p: len(str(p)))[0])
    bundle = _find_bundle_safe()
    (CODE / 'bootstrap.py').write_text(''.join(json.loads(FROZEN.read_text())['cells'][2]['source']))
    run_stage([sys.executable, CODE / 'bootstrap.py'], ROOT / 'bootstrap.log', 'Official RDKit wheel', quiet=False)
    STAGES[-1] = ('Sources and RDKit wheel', time.time() - t0, 'ok')
    tiles([('Pipeline source', FROZEN_SHA256[:12] + 'вЂ¦', 'SHA-256 verified'),
           ('Helper modules', f'{len(MODULES)}', '6 public + 2 new, SHA-256 verified'),
           ('Neural bundle', Path(bundle).name, '4 files, SHA-256 verified'),
           ('Stage 0', minutes(time.time() - t0), 'including the wheel install')])
    return bundle


def run_public_pipeline():
    """Stage 1: every cell of the pinned pipeline in one process, then the export of its final lists."""
    if not RUN_PIPELINE:
        note('Fast commit.', 'The pipeline is skipped in this commit run; the scoring rerun executes every stage.', BLUE)
        return False
    assert not (ROOT / 'work/spec_meta.parquet').exists() and not (ROOT / 'submission.csv').exists(), \
        'Start from a fresh session (Run All after a restart)'
    seconds = run_stage([sys.executable, CODE / 'runner.py', FROZEN, FROZEN_SHA256, ROOT],
                        ROOT / 'submission_pipeline.log', 'Public 0.465 pipeline')
    receipt = json.loads((ROOT / 'execution_receipt.json').read_text())
    full = receipt['is_full_run']
    tiles([('Molecules ranked', f"{receipt['processed_molecules']:,}", 'hidden test' if full else 'visible smoke subset'),
           ('Answer rows', f'{len(SAMPLE_IDS):,}', 'all ranked' if full else 'placeholders elsewhere'),
           ('Pipeline time', minutes(seconds), 'one process, every cell'),
           ('Baseline file', BASELINE.name, 'kept for reference')])
    return True


def score_final_lists():
    """Stage 2a: the public scorer on exactly the exported candidates."""
    try:
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'frozen_candidate_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'neural_candidate_features.parquet', 'cuda'],
                  ROOT / 'submission_neural.log', 'Neural scorer В· final lists')
        return True
    except Exception as exc:
        note('Neural scorer failed.', f'The 0.465 answers will be submitted. {type(exc).__name__}: {str(exc)[:300]}', RED)
        return False


def score_outside_candidates(needed):
    """Stage 2b (optional): collect and score candidates ranked just outside the final lists."""
    elapsed_h = (time.time() - NB_T0) / 3600
    if not needed:
        return False
    if elapsed_h >= TIME_GUARD_HOURS:
        note('Tail rescue skipped.', f'The session has run {elapsed_h:.1f} h (guard {TIME_GUARD_HOURS} h).', YELLOW)
        return False
    try:
        run_stage([sys.executable, CODE / 'outsiders.py', FROZEN, ROOT, ROOT / 'frozen_candidate_evidence.parquet',
                   ROOT / 'outsider_evidence.parquet', OUTSIDERS_PER_MOLECULE],
                  ROOT / 'outsiders.log', 'Outside candidates В· collect')
        if not len(pd.read_parquet(ROOT / 'outsider_evidence.parquet')):
            return False
        run_stage([sys.executable, CODE / 'neural_scores.py', BUNDLE, 'final_model.pt',
                   ROOT / 'outsider_evidence.parquet', ROOT / 'effective_query_spectra.parquet',
                   ROOT / 'outsider_neural_features.parquet', 'cuda'],
                  ROOT / 'outsiders_neural.log', 'Neural scorer В· outside candidates')
        return True
    except Exception as exc:
        note('Tail rescue disabled.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
        return False


def show_neural_summary():
    if not NEURAL_OK:
        return
    nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet')
    act = nf[nf.processed.astype(bool)]
    n_out = len(pd.read_parquet(ROOT / 'outsider_neural_features.parquet')) if OUTSIDERS_OK else 0
    tiles([('Candidates scored', f'{len(act):,}', f'{act.molecule_id.nunique():,} molecules'),
           ('Scored molecules', f'{int(act.groupby("molecule_id").neural_ok.all().sum()):,}', 'all spectra usable'),
           ('Spectra per molecule', f'{act.groupby("molecule_id").n_spectra.first().median():.0f}', 'median used by the scorer'),
           ('Outside candidates', f'{n_out:,}', 'scored for the tail rescue' if n_out else 'not needed for this variant')])


def build_consensus():
    """Stage 3a: the public 0.466 file, its exact rebuild, then every variant."""
    global consensus
    try:
        consensus = importlib.import_module('consensus')
        from submission import apply as public_apply
        evidence = ROOT / 'neural_candidate_features.parquet'
        with contextlib.redirect_stdout(io.StringIO()):      # the public helper prints its own report
            public_apply(evidence, BASELINE, ROOT / 'submission_public0466.csv', weight=0.25, offset=3.0)
        consensus.build_variants(evidence, BASELINE, ROOT / 'variants_check', {'V1': consensus.PRESETS['V1']}, 3.0)
        mine = pd.read_csv(ROOT / 'variants_check/submission_V1.csv', dtype=str, keep_default_na=False)
        public = pd.read_csv(ROOT / 'submission_public0466.csv', dtype=str, keep_default_na=False)
        assert mine.equals(public), 'V1 differs from the public 0.466 file'
        presets = dict(consensus.PRESETS)
        if VARIANT == 'CUSTOM':
            presets['CUSTOM'] = dict(w_top=float(CUSTOM['w_top']), w_mid=float(CUSTOM['w_mid']),
                                     rescue=int(CUSTOM.get('rescue', 0)), label='Custom weights')
        report, moves = consensus.build_variants(
            evidence, BASELINE, VARIANTS_DIR, presets, RRF_OFFSET,
            ROOT / 'outsider_neural_features.parquet' if OUTSIDERS_OK else None)
        print('V1 rebuilt exactly: the new code reproduces the public 0.466 file row for row.')
        return True, report, moves
    except Exception as exc:
        note('Consensus layer disabled.', f'{type(exc).__name__}: {str(exc)[:400]}', RED)
        return False, {}, pd.DataFrame()


def identity_map():
    keys = {}
    for path in (ROOT / 'neural_candidate_features.parquet', ROOT / 'outsider_neural_features.parquet'):
        if path.is_file():
            f = pd.read_parquet(path, columns=['smiles', 'metric_key'])
            keys.update({s: k for s, k in zip(f.smiles, f.metric_key) if isinstance(k, str)})
    return keys


def well_formed(path, same_candidates=True):
    """Minimal checks that do not depend on the new modules."""
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, 'rows'
    lists = f.smiles.str.split(';')
    assert lists.map(len).between(1, 25).all(), 'list length'
    assert lists.map(lambda v: all(x.strip() for x in v) and len(set(v)) == len(v)).all(), 'SMILES'
    if same_candidates and BASELINE.is_file():
        b = pd.read_csv(BASELINE, dtype=str, keep_default_na=False)
        assert all(sorted(x.split(';')) == sorted(y.split(';')) for x, y in zip(b.smiles, f.smiles)), 'candidates'


def choose_and_submit():
    """Stage 3b: chosen variant -> V1 -> V0 -> public 0.466 file -> baseline file (-> sample in a fast commit)."""
    t0 = time.time()
    chain = []
    if CONSENSUS_OK:
        chain += [(n, VARIANTS_DIR / f'submission_{n}.csv') for n in dict.fromkeys([VARIANT, 'V1', 'V0'])]
    chain += [('public 0.466 file', ROOT / 'submission_public0466.csv'), ('baseline file', BASELINE)]
    chosen = None
    for name, path in chain:
        if not path.is_file():
            continue
        try:
            rescue = name in REPORT and REPORT[name].get('rescued', 0) > 0
            if rescue:
                consensus.validate(path, SAMPLE_IDS, BASELINE, allow_new=True, identity_of=identity_map())
            well_formed(path, same_candidates=not rescue)
            chosen = (name, path)
            break
        except Exception as exc:
            note(f'{name} rejected.', f'{type(exc).__name__}: {str(exc)[:300]}', YELLOW)
    if chosen is None:
        assert not IS_RERUN, 'No valid answer file in the scoring rerun'
        chosen = ('sample (fast commit)', COMP / 'sample_submission.csv')
    shutil.copyfile(chosen[1], ROOT / 'submission.csv')
    STAGES.append(('Consensus and checks', time.time() - t0, 'ok'))
    return chosen[0]


def show_variant_board():
    if CONSENSUS_OK:
        board = pd.DataFrame([{
            'Variant': k, 'Rule': v['label'], 'Leader weight': f"{v['w_top']:.2f}",
            'Ranks 2вЂ“25 weight': f"{v['w_mid']:.2f}", 'Tail rescue': v['rescue'] or 'вЂ“',
            'Rows re-ordered': f"{v['changed_rows']:,}", 'Rank 1 changed': f"{v['changed_top1']:,}",
            'Rescued': f"{v['rescued']:,}"} for k, v in REPORT.items()])
        board.index = board.Variant
        table_html(board, highlight=SUBMITTED, numeric=board.columns[2:])
    sub = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    tiles([('Submitted', SUBMITTED, 'copied to submission.csv'),
           ('Rows', f'{len(sub):,}', 'one per molecule, sample order'),
           ('Candidates per row', f'{sub.smiles.str.count(";").add(1).median():.0f}', 'median, at most 25'),
           ('SHA-256', hashlib.sha256((ROOT / 'submission.csv').read_bytes()).hexdigest()[:12] + 'вЂ¦', 'of submission.csv')])

In [ ]:
from pathlib import Path
_physical = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)))\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces."""\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse.\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    key = np.round(fm, 5)\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    return iw / tot, q, len(shifts), len(mz)\n\n\ndef _score_spectrum(levels, weights, prep, tol):\n    iw, q, ns, npk = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\']))\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'])\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    tol = float(c[\'tol\'])\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
_post = "from __future__ import annotations\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\n\n@lru_cache(maxsize=30000)\ndef mol_formula(s):\n    try:\n        m = Chem.MolFromSmiles(str(s))\n        return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n    except Exception:\n        return None\n\ndef finite(x,default=np.nan):\n    try:\n        t=float(x)\n        return t if np.isfinite(t) else default\n    except (ValueError,TypeError):\n        return default\n\ndef select_spectra(frame,max_n=12):\n    out=[]\n    by_bucket={}\n    for r in frame.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs,dtype=float).ravel()\n        it=np.asarray(r.ms2_normalized_intensities,dtype=float).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        good=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if good.sum()<5:continue\n        order=np.argsort(it[good])[-120:]\n        mz,it=mz[good][order],it[good][order]\n        e=finite(getattr(r,'collision_energy_ev',30),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        item={'mz':mz,'it':it,'adduct':adduct,'prec':finite(getattr(r,'precursor_mz',None),float(mz.max())),'mode':mode}\n        bucket=(adduct,round(e/10))\n        by_bucket.setdefault(bucket,[]).append((int(good.sum()),item))\n    for key in sorted(by_bucket):\n        out += [v for _,v in sorted(by_bucket[key],key=lambda x:-x[0])[:2]]\n    return out[:max_n]\n\ndef support(payload,mid,s):\n    eng=(payload.get('engine2',{}).get(mid) or {}).get('smiles') or []\n    pc=(payload.get('pubchem',{}).get(mid) or {}).get('pc') or []\n    main=payload.get('base_tail',{}).get(mid) or []\n    fusion=payload.get('fusion_overflow',{}).get(mid) or []\n    def rank(v):return v.index(s)+1 if s in v else 0\n    return dict(engine2=rank(eng),pubchem=rank(pc),main=rank(main),fusion=rank(fusion))\n\ndef score_candidates(frag,spec,candidates,cache):\n    if not spec:return None\n    opt=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cache=cache)],float)\n    old=np.asarray([np.nan if a is None else a for a in frag.score_molecule(spec,candidates,cfg=frag.BASELINE_CFG,cache=cache)],float)\n    if not np.isfinite(opt).all() or not np.isfinite(old).all():return None\n    chunks=[spec[i:i+2] for i in range(0,len(spec),2)]\n    group=np.array([[np.nan if v is None else v for v in frag.score_molecule(ch,candidates,cache=cache)] for ch in chunks],float)\n    return opt,old,group\n\ndef decision(mid,control,listed,outs,query,payload,frag,cache,allow_tail=True,allow_top=True):\n    diag={'molecule_id':mid,'top1_changed':False,'tail_changed':False,'local_swaps':0,'reason':'unchanged','n_spectra':0,\n          'candidate_count':len(control),'top1_frag_margin':0.,'top1_neural_margin':0.,'new_candidate_source':''}\n    bysmiles=listed.set_index('smiles',drop=False)\n    if len(bysmiles)!=len(listed) or any(s not in bysmiles.index for s in control):\n        diag['reason']='evidence_mismatch';return control,diag\n    if not bool(listed.neural_ok.all()) or not bool(listed.processed.all()):\n        diag['reason']='unscored';return control,diag\n    n=len(control)\n    if n<3:return control,diag\n    anchor=mol_formula(control[0])\n    if anchor is None:\n        diag['reason']='formula_missing';return control,diag\n    cand=list(control[:min(12,n)])\n    outrows=[]\n    if allow_tail and outs is not None and len(outs) and n>=20:\n        for row in outs.sort_values('baseline_rank').itertuples(index=False):\n            if not bool(row.neural_ok) or not bool(row.processed):continue\n            s=str(row.smiles)\n            if s in control or mol_formula(s)!=anchor:continue\n            ev=support(payload,mid,s)\n            sources=int(bool(ev['engine2']))+int(bool(ev['pubchem']))+int(bool(ev['main']))\n            if sources<2 and not (0<ev['fusion']<=3):continue\n            outrows.append(row)\n            cand.append(s)\n            if len(outrows)>=8:break\n    spec=select_spectra(query)\n    diag['n_spectra']=len(spec)\n    if len(spec)<2:\n        diag['reason']='few_spectra';return control,diag\n    data=score_candidates(frag,spec,cand,cache)\n    if data is None:\n        diag['reason']='frag_unavailable';return control,diag\n    frag_new,frag_old,energy=data\n    pos={s:i for i,s in enumerate(cand)}\n    leader=control[0];lp=pos[leader]\n    neu_vals=np.asarray([finite(x) for x in listed.neural_score],float)\n    neural_sd=max(float(np.nanstd(neu_vals)),1e-6)\n    result=list(control)\n    lib=finite(bysmiles.loc[leader,'library_max'],0.)\n    if allow_top and lib<.87 and len(spec)>=3:\n        contenders=[]\n        for ix,s in enumerate(control[1:min(6,n)],1):\n            if mol_formula(s)!=anchor:continue\n            j=pos[s]; ev=bysmiles.loc[s]; lead_ev=bysmiles.loc[leader]\n            if not np.isfinite(frag_new[j]) or frag_new[j]<.27:continue\n            margin=float(frag_new[j]-frag_new[lp]); oldmargin=float(frag_old[j]-frag_old[lp])\n            if margin<.095 or oldmargin<.028:continue\n            votes=float((energy[:,j]>energy[:,lp]+.025).mean())\n            if votes<.70:continue\n            nmargin=(finite(ev.neural_score)-finite(lead_ev.neural_score))/neural_sd\n            if not np.isfinite(nmargin) or nmargin<.23:continue\n            if finite(ev.neural_cos)<finite(lead_ev.neural_cos)+.003 or finite(ev.neural_fp)<finite(lead_ev.neural_fp)-.012:continue\n            supports=0\n            for col in ('engine2_rank','pubchem_rank','main_rank'):\n                rank=finite(ev.get(col),0)\n                if 0<rank<=5:supports+=1\n            simulator_vote=0\n            for col in ('ice','gl'):\n                a,b=finite(ev.get(col)),finite(lead_ev.get(col))\n                if np.isfinite(a) and np.isfinite(b):\n                    if a>=b:simulator_vote+=1\n                    elif a<b-.35:simulator_vote-=1\n            if supports<1 and simulator_vote<1:continue\n            if simulator_vote<=-1:continue\n            contenders.append((margin+.05*nmargin+.025*simulator_vote+.01*supports,s,margin,nmargin))\n        if contenders:\n            _,winner,margin,nmargin=max(contenders,key=lambda v:v[0])\n            result.remove(winner);result.insert(0,winner)\n            diag.update(top1_changed=True,top1_frag_margin=round(margin,5),top1_neural_margin=round(nmargin,5),reason='fragment_neural_top1')\n    swaps=0\n    for i in range(2,min(11,n)):\n        prev,s=result[i-1],result[i]\n        if mol_formula(prev)!=mol_formula(s):continue\n        j,k=pos.get(s),pos.get(prev)\n        if j is None or k is None:continue\n        if frag_new[j]-frag_new[k]<.085 or frag_old[j]-frag_old[k]<.015:continue\n        if (energy[:,j]>energy[:,k]+.010).mean()<.65:continue\n        neui,neup=finite(bysmiles.loc[s,'neural_score']),finite(bysmiles.loc[prev,'neural_score'])\n        if not np.isfinite(neui) or neui<neup-.08*neural_sd:continue\n        result[i-1],result[i]=result[i],result[i-1]\n        swaps+=1\n        if swaps>=2:break\n    diag['local_swaps']=swaps\n    if not diag['top1_changed'] and swaps:diag['reason']='adjacent_fragment_consensus'\n    if allow_tail and outrows and len(result)==25:\n        last=result[-1]\n        if last in pos:\n            lastp=pos[last]\n            eligible=[]\n            listed_keys=set(listed.metric_key.dropna().astype(str))\n            for row in outrows:\n                s=str(row.smiles);j=pos[s]\n                if not isinstance(row.metric_key,str) or row.metric_key in listed_keys:continue\n                if not np.isfinite(frag_new[j]) or frag_new[j]<.28:continue\n                fm=float(frag_new[j]-frag_new[lastp])\n                if fm<.065 or frag_old[j]-frag_old[lastp]<.02:continue\n                if (energy[:,j]>energy[:,lastp]+.01).mean()<.65:continue\n                if finite(row.neural_score)<finite(bysmiles.loc[last,'neural_score'])+.22*neural_sd:continue\n                if finite(row.neural_cos)<=finite(bysmiles.loc[last,'neural_cos']) or finite(row.neural_fp)<=finite(bysmiles.loc[last,'neural_fp']):continue\n                ev=support(payload,mid,s)\n                supports=int(bool(ev['main']))+int(bool(ev['engine2']))+int(bool(ev['pubchem']))\n                eligible.append((fm,s,ev,supports))\n            if eligible:\n                margin,s,ev,supports=max(eligible,key=lambda q:q[0]+.015*q[3])\n                result[-1]=s\n                diag.update(tail_changed=True,new_candidate_source='+' .join(k for k in ('main','engine2','pubchem','fusion') if ev[k]),reason='tail_supported' if not diag['top1_changed'] and not swaps else diag['reason'])\n    assert len(result)==len(control) and len(set(result))==len(result)\n    assert set(result[:-1]).issubset(set(control)) and not (diag['top1_changed'] and result[0]==control[0])\n    return result,diag\n\ndef apply(root,comp,max_seconds=1500,allow_tail=True,allow_top=True):\n    import frag_rescore as frag\n    root=Path(root);comp=Path(comp)\n    control=root/'variants/submission_V2.csv'\n    if not control.is_file():raise FileNotFoundError(control)\n    src=pd.read_csv(control,dtype=str,keep_default_na=False)\n    need=['molecule_id','smiles','metric_key','processed','neural_ok','neural_score','neural_cos','neural_fp','baseline_rank']\n    listed=pd.read_parquet(root/'neural_candidate_features.parquet')\n    if not set(need).issubset(listed.columns):raise ValueError('Neural evidence columns missing')\n    outsiders_path=root/'outsider_neural_features.parquet'\n    outsider=pd.read_parquet(outsiders_path) if outsiders_path.is_file() else pd.DataFrame()\n    q=pd.read_parquet(root/'effective_query_spectra.parquet')\n    payload=json.loads((root/'e004_candidate_inputs.json').read_text())\n    lmap={str(mid):g.sort_values('baseline_rank') for mid,g in listed.groupby('molecule_id',sort=False)}\n    omap={str(mid):g for mid,g in outsider.groupby('molecule_id',sort=False)} if len(outsider) else {}\n    qmap={str(mid):g for mid,g in q.groupby('molecule_id',sort=False)}\n    start=time.monotonic()\n    cache={}; rows=[];audits=[];expired=False\n    frag.warmup()\n    for i,(mid,smiles) in enumerate(src.itertuples(index=False)):\n        mid=str(mid);old=smiles.split(';')\n        if time.monotonic()-start>max_seconds:\n            expired=True\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='time_budget',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        if mid not in lmap or mid not in qmap:\n            rows.append((mid,smiles));audits.append(dict(molecule_id=mid,reason='not_processed',top1_changed=False,tail_changed=False,local_swaps=0))\n            continue\n        try:\n            new,meta=decision(mid,old,lmap[mid],omap.get(mid),qmap[mid],payload,frag,cache,allow_tail=allow_tail,allow_top=allow_top)\n        except Exception as exc:\n            new,meta=old,dict(molecule_id=mid,reason='error:'+type(exc).__name__,top1_changed=False,tail_changed=False,local_swaps=0)\n        rows.append((mid,';'.join(new)));audits.append(meta)\n        if (i+1)%40==0:print('PARALLAX',i+1,'/',len(src),'elapsed',round(time.monotonic()-start,1),'s',flush=True)\n        if len(cache)>35000:cache.clear()\n    dest=pd.DataFrame(rows,columns=['molecule_id','smiles'])\n    ids=pd.read_parquet(comp/'test.parquet',columns=['molecule_id']).molecule_id.astype(str).unique().tolist()\n    if dest.molecule_id.tolist()!=src.molecule_id.tolist() or set(ids)!=set(dest.molecule_id):raise AssertionError('Molecule coverage changed')\n    for initial,new in zip(src.smiles,dest.smiles):\n        before=initial.split(';');after=new.split(';')\n        if not 1<=len(after)<=25 or len(after)!=len(set(after)) or any(not k for k in after):raise AssertionError('Submission format')\n        if after[0] not in before or not set(after[:-1]).issubset(set(before)):raise AssertionError('Unsafe candidate replacement')\n        if len(set(after)-set(before))>1:raise AssertionError('Too many outsiders')\n    out=root/'submission.csv';temp=root/'submission.parallax.tmp'\n    dest.to_csv(temp,index=False)\n    reread=pd.read_csv(temp,dtype=str,keep_default_na=False)\n    if not reread.equals(dest):raise AssertionError('CSV roundtrip failed')\n    temp.replace(out)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'spectral_parallax_audit.csv',index=False)\n    diagnostics={'control_sha256':hashlib.sha256(control.read_bytes()).hexdigest(),\n        'submission_sha256':hashlib.sha256(out.read_bytes()).hexdigest(),\n        'rows':len(dest),'changed_rows':int((src.smiles!=dest.smiles).sum()),\n        'top1_changed':int(sum(bool(r.get('top1_changed')) for r in audits)),\n        'tail_changed':int(sum(bool(r.get('tail_changed')) for r in audits)),\n        'adjacent_swaps':int(sum(int(r.get('local_swaps',0)) for r in audits)),\n        'time_budget_exceeded':expired,'elapsed_seconds':round(time.monotonic()-start,2),\n        'outcome':'experimental_unvalidated','baseline':'V2_0.468_user_reported'}\n    (root/'spectral_parallax_report.json').write_text(json.dumps(diagnostics,indent=2))\n    print('PARALLAX REPORT',json.dumps(diagnostics),flush=True)\n    return diagnostics\n"
assert __import__('hashlib').sha256(_physical.encode()).hexdigest() == 'eeacdcf0afb173a8b438054d93c30f061df7c9c69d1ae32db3530b2916f2e6a7'
assert __import__('hashlib').sha256(_post.encode()).hexdigest() == 'cd1594b3bbfa75ce30997559c5403438b7cec011879dcb9c0d48e111715dd85d'
(ROOT/'frag_rescore.py').write_text(_physical)
(CODE/'parallax_physical.py').write_text(_post)
print('Fragment source:', __import__('hashlib').sha256(_physical.encode()).hexdigest()[:12])


In [ ]:
BUNDLE = materialise_sources()

# ---- private-model swap: use our own longer-trained spectrum model ----
def _swap_to_ours(base_bundle):
    """Find our private scorer bundle and use it as the neural scorer.

    Accepts any attached data source that contains final_model.pt + config_used.json
    (works for a Kaggle dataset or for a kernel output), while ignoring the public
    kit bundle and the staged copy of it.
    """
    import shutil as _s
    from pathlib import Path as _P
    _hits = []
    for _p in _P('/kaggle/input').rglob('final_model.pt'):
        _par = _p.parent
        if 'casmi-spectrum-model-v1' in str(_p) or 'kit_staged' in str(_p):
            continue
        if (_par / 'config_used.json').is_file() or 'casmi_spectrum_model_ours' in str(_p) or 'scorer-v2-model' in str(_p):
            _hits.append(_p)
    if not _hits:
        print('OURS: private model not found under /kaggle/input, keeping the public bundle')
        return base_bundle
    _src = _hits[0].parent
    _dst = _P('/kaggle/working/ours_bundle'); _dst.mkdir(parents=True, exist_ok=True)
    _s.copy2(_P(base_bundle) / 'model.py', _dst / 'model.py')
    _s.copy2(_P(base_bundle) / 'features.py', _dst / 'features.py')
    _cfg = _src / 'config_used.json'
    if _cfg.is_file():
        _s.copy2(_cfg, _dst / 'config_used.json')
    _s.copy2(_P(base_bundle) / 'config.json', _dst / 'config.json')
    _s.copy2(_src / 'final_model.pt', _dst / 'final_model.pt')
    print('OURS BUNDLE READY:', sorted(p.name for p in _dst.iterdir()), 'from', _src)
    return str(_dst)


PUB_BUNDLE = BUNDLE
OURS_BUNDLE = _swap_to_ours(PUB_BUNDLE)
BUNDLE = OURS_BUNDLE
PIPELINE_DONE = run_public_pipeline()
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    sub_df = pd.read_csv(COMP / 'sample_submission.csv')
    sub_df['smiles'] = ';'.join(['C' + 'C'*i + 'O' for i in range(25)])
    sub_df.to_csv(ROOT / 'submission.csv', index=False)
    sub_df.to_csv(ROOT / 'submission_control_V2.csv', index=False)
    (ROOT / 'variants').mkdir(exist_ok=True)
    sub_df.to_csv(ROOT / 'variants/submission_V2.csv', index=False)
    sub_df.to_csv(ROOT / 'variants/submission_CUSTOM.csv', index=False)
    NEURAL_OK = False
    OUTSIDERS_OK = False
    CONSENSUS_OK = False
    SUBMITTED = 'sample (fast commit)'
    print('Fast commit: unique dummy SMILES written cleanly; scoring rerun runs full pipeline.')
else:
    if not PIPELINE_DONE:
        raise RuntimeError('A real baseline run is required; no placeholder-only submission')
    BUNDLE = PUB_BUNDLE
    NEURAL_OK = score_final_lists()
    shutil.copyfile(ROOT / 'neural_candidate_features.parquet', ROOT / 'neural_public.parquet')
    BUNDLE = OURS_BUNDLE
    if score_final_lists():
        shutil.copyfile(ROOT / 'neural_candidate_features.parquet', ROOT / 'neural_ours.parquet')
        import pandas as _pd
        _a = _pd.read_parquet(ROOT / 'neural_public.parquet').set_index(['molecule_id', 'smiles'])
        _b = _pd.read_parquet(ROOT / 'neural_ours.parquet').set_index(['molecule_id', 'smiles']).reindex(_a.index)
        def _z(_s):
            _g = _s.groupby(level=0)
            return (_s - _g.transform('mean')) / _g.transform(lambda x: x.std(ddof=0)).clip(lower=1e-6)
        _a['neural_score'] = 0.5 * _z(_a['neural_score'].fillna(0.0)) + 0.5 * _z(_b['neural_score'].fillna(0.0))
        _a['neural_cos'] = 0.5 * _a['neural_cos'] + 0.5 * _b['neural_cos'].fillna(0.0)
        _a['neural_fp'] = 0.5 * _a['neural_fp'] + 0.5 * _b['neural_fp'].fillna(0.0)
        _a['neural_std'] = 0.5 * _a['neural_std'] + 0.5 * _b['neural_std'].fillna(0.0)
        _a['neural_ok'] = _a['neural_ok'].astype(bool) & _b['neural_ok'].fillna(False).astype(bool)
        _a.reset_index().to_parquet(ROOT / 'neural_candidate_features.parquet', index=False)
        print('ENSEMBLE neural = 0.5*z(public) + 0.5*z(ours)', flush=True)
    BUNDLE = OURS_BUNDLE
    if not NEURAL_OK:
        raise RuntimeError('The 0.468 V2 neural checkpoint is required; refusing silent 0.465 fallback')
    OUTSIDERS_OK = score_outside_candidates(needed=True)
    CONSENSUS_OK, REPORT, MOVES = build_consensus()
    if not CONSENSUS_OK:
        raise RuntimeError('Unable to reproduce the V2 control')
    SUBMITTED = choose_and_submit()
    if SUBMITTED != VARIANT:
        raise RuntimeError(f'Expected {VARIANT}, got {SUBMITTED}')
    shutil.copy2(ROOT/'submission.csv', ROOT/'submission_control_V2.csv')
    print('Exact V2 control:', hashlib.sha256((ROOT/'submission_control_V2.csv').read_bytes()).hexdigest()[:16])


In [ ]:
import sys, traceback
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(CODE))
import sys, traceback, json, shutil
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    PARALLAX_DIAGNOSTICS = {'status': 'skipped_fast_commit'}
else:
    from parallax_physical import apply
    try:
        _remaining = max(0, int(8.55 * 3600 - (time.time() - NB_T0)))
        if not PHYSICAL:
            PARALLAX_DIAGNOSTICS = {'status': 'disabled in this experiment'}
        elif _remaining < 180:
            PARALLAX_DIAGNOSTICS = {'status':'skipped_due_to_runtime_budget','remaining_seconds':_remaining,'control_retained':True}
        else:
            PARALLAX_DIAGNOSTICS = apply(ROOT, COMP, max_seconds=min(PHYSICAL_POST_SECONDS, _remaining - 120), allow_tail=True, allow_top=True)
    except Exception:
        traceback.print_exc()
        shutil.copy2(ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
        PARALLAX_DIAGNOSTICS = {'error':'physical_overlay_failed', 'control_restored':True}
        (ROOT/'spectral_parallax_report.json').write_text(json.dumps(PARALLAX_DIAGNOSTICS,indent=2))
assert (ROOT/'submission.csv').is_file()
final = pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.notna().all() and final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(x.strip() for x in a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','rows',len(final),'control',SUBMITTED)
print('EXPERIMENT:', json.dumps(PARALLAX_DIAGNOSTICS))


In [ ]:
from pathlib import Path
import hashlib
_PUBLIC_RERANKER = "from __future__ import annotations\nimport json\nimport math\nimport time\nimport hashlib\nimport shutil\nfrom pathlib import Path\nfrom functools import lru_cache\n\nimport numpy as np\nimport pandas as pd\nfrom rdkit import Chem\nfrom rdkit.Chem import rdMolDescriptors\nimport frag_rescore as frag\n\n@lru_cache(maxsize=40000)\ndef formula(s):\n    m = Chem.MolFromSmiles(str(s))\n    return rdMolDescriptors.CalcMolFormula(m) if m is not None else None\n\ndef number(x, default=float('nan')):\n    try:\n        v=float(x)\n        return v if math.isfinite(v) else default\n    except Exception:\n        return default\n\ndef member_keys(comp):\n    frame=pd.read_parquet(Path(comp)/'train.parquet',columns=['inchikey14'])\n    return set(frame.inchikey14.dropna().astype(str))\n\ndef balanced_spectra(query, max_groups=12):\n    groups={}\n    for r in query.itertuples(index=False):\n        mz=np.asarray(r.ms2_mzs, dtype=np.float64).ravel()\n        it=np.asarray(r.ms2_normalized_intensities, dtype=np.float64).ravel()\n        n=min(len(mz),len(it));mz,it=mz[:n],it[:n]\n        ok=np.isfinite(mz)&np.isfinite(it)&(it>0)\n        if ok.sum()<6:continue\n        order=np.argsort(it[ok])[-120:]\n        mz,it=mz[ok][order],it[ok][order]\n        energy=number(getattr(r,'collision_energy_ev',30.),30.)\n        adduct=str(getattr(r,'adduct','[M+H]+'))\n        mode=1 if str(getattr(r,'ionization_mode','positive')).lower()=='positive' else -1\n        spec={'mz':mz,'it':it,'prec':number(getattr(r,'precursor_mz',None),float(mz.max())),\n              'adduct':adduct,'mode':mode}\n        bucket=(adduct,int(round(energy/10.)))\n        groups.setdefault(bucket,[]).append((int(ok.sum()), spec))\n    chosen=[]\n    for group in sorted(groups):\n        chosen.extend(s for _,s in sorted(groups[group],key=lambda x:-x[0])[:2])\n    return chosen[:max_groups]\n\ndef evidence_rows(df, order):\n    data=df.drop_duplicates('smiles').set_index('smiles',drop=False)\n    if any(s not in data.index for s in order):\n        return None\n    if ('neural_ok' in data and not data.loc[order,'neural_ok'].astype(bool).all()):\n        return None\n    if ('processed' in data and not data.loc[order,'processed'].astype(bool).all()):\n        return None\n    return data\n\ndef source_votes(candidate, leader):\n    good=0;bad=0\n    for label in ('engine2_rank','pubchem_rank','main_rank'):\n        c=number(candidate.get(label),0.);l=number(leader.get(label),0.)\n        if 0<c<=6:good+=1\n        if 0<l<=4 and (c<=0 or c>12):bad+=1\n    sim=0\n    for label in ('ice','gl'):\n        c=number(candidate.get(label));l=number(leader.get(label))\n        if math.isfinite(c) and math.isfinite(l):\n            if c>=l-.025:sim+=1\n            elif c<l-.30:sim-=1\n    return good,bad,sim\n\ndef paired_scores(spectra, smiles, cache):\n    matrix=[];matrix_old=[]\n    for spectrum in spectra:\n        vals=frag.score_molecule([spectrum],smiles,cache=cache)\n        old=frag.score_molecule([spectrum],smiles,cfg=frag.BASELINE_CFG,cache=cache)\n        if any(x is None or not math.isfinite(float(x)) for x in vals+old):return None\n        matrix.append(vals)\n        matrix_old.append(old)\n    return np.asarray(matrix,dtype=float),np.asarray(matrix_old,dtype=float)\n\ndef propose(order, original, data, query, train, cache):\n    record={'action':'none','top1_changed':False,'swaps':0,'n_spectra':0,\n            'protected_train':False,'physical_gain':None,'neural_gain_sd':None}\n    if len(order)<4 or not order or original[0]!=order[0]:\n        record['action']='protected_previous_top1';return order,record\n    evid=evidence_rows(data,order)\n    if evid is None:\n        record['action']='incomplete_evidence';return order,record\n    keys=[str(evid.loc[s,'metric_key']) if 'metric_key' in evid.columns else '' for s in order]\n    trained=np.asarray([k in train for k in keys],dtype=bool)\n    record['protected_train']=bool(trained.any())\n    leader=order[0]\n    if formula(leader) is None:\n        return order,record\n    lib=number(evid.loc[leader].get('library_max'),0.)\n    if lib>=.90:\n        record['action']='library_protected';return order,record\n    selected=balanced_spectra(query)\n    record['n_spectra']=len(selected)\n    if len(selected)<3:\n        record['action']='insufficient_spectra';return order,record\n    head=order[:min(10,len(order))]\n    p=paired_scores(selected,head,cache)\n    if p is None:\n        record['action']='fragment_unavailable';return order,record\n    a,b=p\n    neu=np.asarray([number(evid.loc[s].get('neural_score')) for s in order],float)\n    if not np.isfinite(neu).all():\n        record['action']='neural_unavailable';return order,record\n    ns=max(float(neu.std()),.00001)\n    updated=list(order)\n    candidates=[]\n    if not trained[0]:\n        for i in range(1,min(6,len(head))):\n            if trained[i] or formula(head[i])!=formula(leader):continue\n            new_margin=a[:,i]-a[:,0]\n            old_margin=b[:,i]-b[:,0]\n            mean=float(np.mean(new_margin));med=float(np.median(new_margin))\n            low=float(np.quantile(new_margin,.25));votes=float((new_margin>.020).mean())\n            old=float(np.mean(old_margin))\n            nz=float((neu[i]-neu[0])/ns)\n            if mean<.095 or med<.070 or low<-.010 or votes<.75 or old<.022 or nz<.18:continue\n            ev=evid.loc[head[i]];lv=evid.loc[leader]\n            if number(ev.get('neural_cos'),0.)<number(lv.get('neural_cos'),0.)+.002:continue\n            if number(ev.get('neural_fp'),0.)<number(lv.get('neural_fp'),0.)-.010:continue\n            votes_source,bad,sim=source_votes(ev,lv)\n            if votes_source<1 and sim<1:continue\n            if bad>=2 or sim<0:continue\n            confidence=mean+.045*nz+.020*sim+.010*votes_source+.018*max(0,low)\n            candidates.append((confidence,i,mean,nz))\n        if candidates:\n            _,idx,gain,nz=max(candidates)\n            updated.insert(0,updated.pop(idx))\n            record.update(action='provenance_spectral_top1',top1_changed=True,\n                          physical_gain=round(gain,5),neural_gain_sd=round(nz,4))\n    swaps=0\n    for pos in range(1,min(8,len(updated)-1)):\n        before=updated[pos];after=updated[pos+1]\n        if formula(before)!=formula(after):continue\n        i,j=head.index(before) if before in head else -1, head.index(after) if after in head else -1\n        if i<0 or j<0:continue\n        if trained[i] or trained[j]:continue\n        d=a[:,j]-a[:,i];do=b[:,j]-b[:,i]\n        if float(np.mean(d))<.068 or float(np.median(d))<.045 or float((d>.010).mean())<.67:continue\n        if float(np.mean(do))<.015:continue\n        if neu[j]-neu[i]<-.04*ns:continue\n        c=source_votes(evid.loc[after],evid.loc[before])[0]\n        if c<1 and neu[j]<=neu[i]:continue\n        updated[pos],updated[pos+1]=updated[pos+1],updated[pos]\n        swaps+=1\n        if swaps>=2:break\n    record['swaps']=swaps\n    if swaps and not record['top1_changed']:record['action']='paired_energy_swap'\n    # The private 0.475 model freezes known training candidates. This public analogue\n    # only allows permutations among untrained structures, preserving every training slot.\n    for i,s in enumerate(order):\n        if trained[i] and updated[i]!=s:\n            return order,dict(record,action='training_slot_guard',top1_changed=False,swaps=0)\n    return updated,record\n\ndef run(root,comp,time_budget=900):\n    root=Path(root);comp=Path(comp)\n    path=root/'submission.csv'\n    control=root/'submission_control_V2.csv'\n    for req in [path,control,root/'neural_candidate_features.parquet',root/'effective_query_spectra.parquet',comp/'train.parquet']:\n        if not req.exists():raise FileNotFoundError(req)\n    existing=pd.read_csv(path,dtype=str,keep_default_na=False)\n    first=pd.read_csv(control,dtype=str,keep_default_na=False)\n    assert existing.molecule_id.tolist()==first.molecule_id.tolist()\n    assert existing.molecule_id.is_unique\n    shutil.copy2(path,root/'submission_physical0472_control.csv')\n    nfeatures=pd.read_parquet(root/'neural_candidate_features.parquet')\n    spectra=pd.read_parquet(root/'effective_query_spectra.parquet')\n    groups={str(m):df for m,df in nfeatures.groupby('molecule_id',sort=False)}\n    specgroups={str(m):df for m,df in spectra.groupby('molecule_id',sort=False)}\n    known=member_keys(comp)\n    frag.warmup()\n    t0=time.monotonic();cache={};newrows=[];audits=[]\n    for idx,(record,ctrl) in enumerate(zip(existing.itertuples(index=False),first.itertuples(index=False))):\n        mid=str(record.molecule_id)\n        arr=str(record.smiles).split(';');original=str(ctrl.smiles).split(';')\n        if time.monotonic()-t0>time_budget:\n            new=arr;meta={'action':'runtime_budget','top1_changed':False,'swaps':0}\n        elif mid not in groups or mid not in specgroups:\n            new=arr;meta={'action':'no_source_spectra','top1_changed':False,'swaps':0}\n        else:\n            try:\n                new,meta=propose(arr,original,groups[mid],specgroups[mid],known,cache)\n            except Exception as e:\n                new=arr;meta={'action':'failed_'+type(e).__name__,'top1_changed':False,'swaps':0}\n        if set(new)!=set(arr) or len(new)!=len(arr):\n            raise AssertionError('Candidate identities may not change')\n        newrows.append((mid,';'.join(new)))\n        audits.append({'molecule_id':mid,**meta})\n        if len(cache)>25000:cache.clear()\n        if (idx+1)%50==0:\n            print('PROVENANCE',idx+1,'/',len(existing),'elapsed',round(time.monotonic()-t0,1),'s',flush=True)\n    result=pd.DataFrame(newrows,columns=['molecule_id','smiles'])\n    assert result.molecule_id.tolist()==existing.molecule_id.tolist()\n    assert result.molecule_id.is_unique\n    assert result.smiles.str.split(';').map(len).between(1,25).all()\n    assert result.smiles.str.split(';').map(lambda a:len(set(a))==len(a) and all(a)).all()\n    temp=root/'submission.provenance.tmp.csv'\n    result.to_csv(temp,index=False)\n    assert pd.read_csv(temp,dtype=str,keep_default_na=False).equals(result)\n    temp.replace(path)\n    report=pd.DataFrame(audits)\n    report.to_csv(root/'provenance_tournament_audit.csv',index=False)\n    summary={'source':'0.472 physical consensus public pipeline',\n             'private_rr3_weights_used':False,'train_structure_keys':len(known),\n             'rows':len(result),'changed':int((existing.smiles!=result.smiles).sum()),\n             'top1_changed':int(report.top1_changed.fillna(False).astype(bool).sum()),\n             'adjacent_swaps':int(report.swaps.fillna(0).sum()),\n             'elapsed_seconds':round(time.monotonic()-t0,2),\n             'control_hash':hashlib.sha256((root/'submission_physical0472_control.csv').read_bytes()).hexdigest(),\n             'final_hash':hashlib.sha256(path.read_bytes()).hexdigest(),\n             'note':'Leaderboard improvement not validated'}\n    (root/'provenance_tournament_report.json').write_text(json.dumps(summary,indent=2))\n    print('PROVENANCE_REPORT',json.dumps(summary),flush=True)\n    return summary\n"
PROVENANCE_CODE = ROOT/'parallax_public_provenance.py'
PROVENANCE_CODE.write_text(_PUBLIC_RERANKER)
assert hashlib.sha256(PROVENANCE_CODE.read_bytes()).hexdigest() == hashlib.sha256(_PUBLIC_RERANKER.encode()).hexdigest()
print('Public provenance-aware reranker:',PROVENANCE_CODE)

In [ ]:
import traceback
try:
    import importlib.util
    spec=importlib.util.spec_from_file_location('parallax_public_provenance',PROVENANCE_CODE)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    _left=max(0,int(8.65*3600-(time.time()-NB_T0)))
    if not PROVENANCE:
        PROVENANCE_REPORT={'status':'disabled in this experiment'}
    elif _left<240:
        PROVENANCE_REPORT={'status':'skipped_because_of_kaggle_runtime','seconds_remaining':_left}
    else:
        PROVENANCE_REPORT=module.run(ROOT,COMP,time_budget=min(960,_left-180))
except Exception as exc:
    traceback.print_exc()
    shutil.copy2(ROOT/'submission_physical0472_control.csv' if (ROOT/'submission_physical0472_control.csv').is_file() else ROOT/'submission_control_V2.csv',ROOT/'submission.csv')
    PROVENANCE_REPORT={'status':'restored_control_after_error','exception':repr(exc)}

final=pd.read_csv(ROOT/'submission.csv',dtype=str,keep_default_na=False)
assert list(final.columns)==['molecule_id','smiles']
assert final.molecule_id.tolist()==SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1,25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a))==len(a)).all()
print('SUBMISSION READY:',ROOT/'submission.csv','molecules',len(final),'provenance',PROVENANCE_REPORT)


In [ ]:
# ── v8 · mass prior table (MASS_PEN / MASS_PPM; empty when off) ──────────────────────────────────────────────────
import json, traceback
MASS_PEN, MASS_PPM, MASS_REPORT = {}, {}, dict(status='off in this preset')
try:
    if MASS_ON and globals().get('PIPELINE_DONE'):
        from rdkit import Chem, RDLogger
        from rdkit.Chem import Descriptors
        RDLogger.DisableLog('rdApp.*')
        # neutral mass from the competition's own precursor m/z and adduct, with the constants the band was measured with
        _ADD = {'[M+H]+': 1.007276, '[M-H]-': -1.007276, '[M+Na]+': 22.989218, '[M+NH4]+': 18.033823,
                '[M+K]+': 38.963158, '[M+CH2O2-H]-': 44.998201, '[M+Cl]-': 34.969402}
        _eq = pd.read_parquet(COMP / 'test.parquet', columns=['molecule_id', 'precursor_mz', 'adduct'])
        _eq['nm'] = pd.to_numeric(_eq.precursor_mz, errors='coerce') - _eq.adduct.map(_ADD)
        _eq = _eq[np.isfinite(_eq.nm) & (_eq.nm > 50)]
        _nm = {str(k): float(v) for k, v in _eq.groupby('molecule_id').nm.median().items()}
        _nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet', columns=['molecule_id', 'smiles'])
        _mass, _n_pen = {}, 0
        for _mid, _g in _nf.groupby('molecule_id', sort=False):
            _mid = str(_mid)
            nm = _nm.get(_mid)
            if not nm:
                continue
            pens, ppms = {}, {}
            for s in _g.smiles.astype(str):
                if s not in _mass:
                    m = Chem.MolFromSmiles(s)
                    _mass[s] = float(Descriptors.ExactMolWt(m)) if m is not None else None
                if _mass[s] is None:
                    continue
                ppm = (_mass[s] - nm) / nm * 1e6
                ppms[s] = ppm
                d = abs(ppm - MASS_CENTER)
                if d > MASS_FREE:
                    pens[s] = -MASS_W * min(1.0, (d - MASS_FREE) / (MASS_FULL - MASS_FREE))
                    _n_pen += 1
            MASS_PPM[_mid] = ppms
            if pens:
                MASS_PEN[_mid] = pens
        _all = [abs(v - MASS_CENTER) for d in MASS_PPM.values() for v in d.values()]
        MASS_REPORT = dict(status='on', molecules=len(MASS_PPM), candidates=len(_all), penalised=_n_pen,
                           beyond_5ppm=int(sum(x > 5 for x in _all)), lists_touched=len(MASS_PEN))
    elif MASS_ON:
        MASS_REPORT = dict(status='off: no pipeline answer in this run')
except Exception as exc:
    traceback.print_exc()
    MASS_PEN, MASS_PPM, MASS_REPORT = {}, {}, dict(status='off after an error', exception=repr(exc)[:300])
print('V8', V8, '| mass prior', json.dumps(MASS_REPORT))

In [ ]:
LIB_MAX = {}
# в”Ђв”Ђ Variant M: rank 1 from fragment check on V2, ranks 2-25 from CUSTOM + DocPrior + In-Pool Dedup в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
import os, json, shutil, hashlib, traceback, glob
from pathlib import Path
from functools import lru_cache
import pandas as pd
import numpy as np


def _load_pop_prior(root, input_root):
    """Load PubChem documentation prior and spectral annotations for InChIKey14 lookups."""
    pop_map, spec_map, np_map = {}, {}, {}
    try:
        hits = sorted(glob.glob(str(input_root / '**/pool_popularity.csv'), recursive=True))
        if hits:
            cols = ['inchikey14', 'pop', 'src_spectral_library', 'np_coconut', 'np_lotus', 'src_natural_product']
            p_df = pd.read_csv(hits[0], usecols=cols)
            pop_map = dict(zip(p_df['inchikey14'], p_df['pop']))
            spec_map = dict(zip(p_df['inchikey14'], p_df['src_spectral_library']))
            np_map = dict(zip(p_df['inchikey14'], (p_df['np_coconut'] | p_df['np_lotus'] | p_df['src_natural_product']).astype(int)))
            print(f'Documentation Prior: loaded {len(p_df)} pool records from {Path(hits[0]).name}')
    except Exception as e:
        print(f'Note: pool_popularity lookup exception ({e}); proceeding with neutral prior.')
    return pop_map, spec_map, np_map


def pxm_merge(physical, v2, custom, pop_map=None, spec_map=None, np_map=None, nf_evidence=None):
    """physical, v2, custom: molecule_id, smiles frames in the same row order. Returns (frame, counts)."""
    assert list(physical.columns) == list(v2.columns) == list(custom.columns) == ['molecule_id', 'smiles']
    assert physical.molecule_id.tolist() == v2.molecule_id.tolist() == custom.molecule_id.tolist(), 'row order differs'
    rows = []
    counts = dict(rows=0, merged=0, leader_moves=0, ranks_reordered=0, skipped={}, skipped_leader_moves=0, moved_ids=[])
    
    pop_map = pop_map or {}
    spec_map = spec_map or {}
    np_map = np_map or {}
    evidence_by_mid = nf_evidence or {}

    for mid, ps, vs, cs in zip(custom.molecule_id, physical.smiles, v2.smiles, custom.smiles):
        p, v, c = ps.split(';'), vs.split(';'), cs.split(';')
        counts['rows'] += 1
        
        # Variant M Leader Rule (Strictly Locked Rank 1):
        # If the fragment check on V2 identified a superior physical bond cleavage leader (p[0] != v[0]):
        # Place p[0] at Rank 1, followed by the high-resolution CUSTOM order.
        if p[0] != v[0]:
            leader = p[0]
            rest = [s for s in c if s != p[0]]
            counts['merged'] += 1
            counts['leader_moves'] += 1
            counts['moved_ids'].append(mid)
        else:
            leader = c[0]
            rest = c[1:]
            counts['merged'] += 1

        # Re-rank Ranks 2вЂ“25 by Documentation Prior + Spectral Library + Natural Product Bonus
        cinfo = evidence_by_mid.get(str(mid), {})
        lib_hit = float(LIB_MAX.get(str(mid), 0.0)) >= LIB_HIT_GATE
        if rest and cinfo:
            def _score_cand(s):
                if s in cinfo:
                    br, nr, k14 = cinfo[s]
                    p_val = float(pop_map.get(k14, 0.0))
                    s_val = float(spec_map.get(k14, 0.0))
                    np_val = float(np_map.get(k14, 0.0))
                    return 1.0 / (3.0 + br) + 0.75 / (3.0 + nr) + POP_W * (p_val / 20.0) + (SPEC_W if lib_hit else 0.0) * s_val + NP_W * np_val
                idx = c.index(s) if s in c else 25
                return 1.0 / (3.0 + idx + 1)
            
            old_rest = list(rest)
            _v8_pen = MASS_PEN.get(str(mid), {})   # v8: 0.0 for every candidate when the mass prior is off
            rest.sort(key=lambda _s: _score_cand(_s) + _v8_pen.get(_s, 0.0), reverse=True)
            if rest != old_rest:
                counts['ranks_reordered'] += 1
            
        if PIN_OLD_LEADER and p[0] != v[0] and v[0] in rest and rest[0] != v[0]:
            rest = [v[0]] + [x for x in rest if x != v[0]]
            counts['old_leader_pinned'] = counts.get('old_leader_pinned', 0) + 1
        rows.append((mid, [leader] + rest))
    return rows, counts


# ---- run (everything above is definitions) ----
def _pxm_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()


def _pxm_main():
    rep = dict(status='disabled')
    if not globals().get('FRAG_ON_V2', False):
        return rep
    assert not globals().get('PROVENANCE', False), 'variant M is defined without the provenance reranker'
    sub_path = ROOT / 'submission.csv'
    v_path, c_path = ROOT / 'variants/submission_V2.csv', ROOT / 'variants/submission_CUSTOM.csv'
    rep['input_sha256'] = dict(physical_v2=_pxm_sha(sub_path), v2=_pxm_sha(v_path), custom=_pxm_sha(c_path))
    diag = globals().get('PARALLAX_DIAGNOSTICS') or {}
    rep['overlay'] = {k: diag[k] for k in ('status', 'error', 'control_sha256', 'submission_sha256', 'changed_rows',
                                           'top1_changed', 'tail_changed', 'adjacent_swaps') if k in diag}
    if diag.get('control_sha256'):
        if diag['control_sha256'] != rep['input_sha256']['v2']:
            print('WARNING: control_sha256 mismatch, proceeding safely with current v2')
    if diag.get('submission_sha256'):
        if diag['submission_sha256'] != rep['input_sha256']['physical_v2']:
            print('WARNING: submission_sha256 mismatch, proceeding safely with current physical_v2')
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in (sub_path, v_path, c_path)]
    for f in frames:
        assert list(f.columns) == ['molecule_id', 'smiles'] and f.molecule_id.tolist() == SAMPLE_IDS, \
            'an input file differs from sample_submission'

    from rdkit import Chem

    from rdkit.Chem.MolStandardize import rdMolStandardize
    _TAUT = rdMolStandardize.TautomerEnumerator()
    _IDMAP = identity_map()

    @lru_cache(maxsize=100000)
    def _to_k14(s):
        k = _IDMAP.get(s)
        if isinstance(k, str) and k:
            return k
        try:
            m = Chem.MolFromSmiles(str(s))
            if m is None:
                return ''
            Chem.RemoveStereochemistry(m)
            return Chem.MolToInchiKey(_TAUT.Canonicalize(m))[:14]
        except Exception:
            return ''

    global LIB_MAX
    LIB_MAX = {}
    out_by_mid = {}
    try:
        _nf = pd.read_parquet(ROOT / 'neural_candidate_features.parquet', columns=['molecule_id', 'library_max'])
        LIB_MAX = {str(m): float(v) for m, v in _nf.groupby('molecule_id', sort=False).library_max.max().items()}
        _op = ROOT / 'outsider_neural_features.parquet'
        if OUTSIDER_FILL and _op.is_file():
            _of = pd.read_parquet(_op)
            _sc = pd.to_numeric(_of.neural_score, errors='coerce')
            _of = _of[_of.neural_ok.astype(bool) & np.isfinite(_sc)].assign(_s=_sc)
            for _m, _g in _of.groupby('molecule_id', sort=False):
                out_by_mid[str(_m)] = _g.sort_values('_s', ascending=False, kind='stable').smiles.astype(str).tolist()
    except Exception as e:
        print(f'Note: final-layer inputs exception ({e}); gate off, no outsider fill')
    rep['lib_hit_molecules'] = int(sum(v >= LIB_HIT_GATE for v in LIB_MAX.values()))

    # Load documentation prior and candidate evidence
    # Pre-index LOTUS natural product candidates for vacant-slot backfill
    lotus_by_mid = {}
    lotus_fills = 0
    try:
        _lotus_paths = list(INPUT.rglob('candidates_lotus.parquet'))
        _spec_p = ROOT / 'effective_query_spectra.parquet'
        if _lotus_paths and _spec_p.is_file():
            import numpy as _np
            _lotus_df = pd.read_parquet(_lotus_paths[0]).sort_values('monoisotopic_mass').reset_index(drop=True)
            _l_masses = _lotus_df['monoisotopic_mass'].to_numpy(dtype=float)
            _l_smiles = _lotus_df['smiles'].to_numpy()
            _spec_df = pd.read_parquet(_spec_p, columns=['molecule_id', 'nm']).drop_duplicates('molecule_id')
            _nm_map = dict(zip(_spec_df['molecule_id'].astype(str), _spec_df['nm']))
            for _mid, _nm in _nm_map.items():
                if pd.notna(_nm) and _nm > 50:
                    _tol = float(_nm) * 15e-6
                    _lo = _np.searchsorted(_l_masses, _nm - _tol, 'left')
                    _hi = _np.searchsorted(_l_masses, _nm + _tol, 'right')
                    if _hi > _lo:
                        lotus_by_mid[_mid] = _l_smiles[_lo:_hi].tolist()
            print(f'LOTUS Candidate index ready for {len(lotus_by_mid)} molecules from {_lotus_paths[0]}')
    except Exception as _e:
        print(f'Note: LOTUS candidate loading skipped ({_e})')

    pop_map, spec_map, np_map = _load_pop_prior(ROOT, INPUT)
    evidence_by_mid = {}
    nf_path = ROOT / 'neural_candidate_features.parquet'
    if nf_path.is_file():
        try:
            nf = pd.read_parquet(nf_path)
            for mid, g in nf.groupby('molecule_id', sort=False):
                order = np.argsort(-g['neural_score'].to_numpy(), kind='stable')
                nranks = np.empty(len(g), dtype=int)
                nranks[order] = np.arange(1, len(g) + 1)
                cand_info = {}
                for s, br, nr, mk in zip(g['smiles'], g['baseline_rank'], nranks, g['metric_key']):
                    k14 = mk if (isinstance(mk, str) and len(mk) == 14) else _to_k14(s)
                    cand_info[s] = (int(br), int(nr), k14)
                evidence_by_mid[str(mid)] = cand_info
        except Exception as e:
            print(f'Note: neural evidence indexing exception ({e})')

    merged_tuples, counts = pxm_merge(*frames, pop_map=pop_map, spec_map=spec_map, np_map=np_map, nf_evidence=evidence_by_mid)

    # In-Pool InChIKey14 Skeletal Deduplication
    reclaimed_smiles = []
    dedup_swaps = 0
    outsider_fills = 0
    for (mid, prim_list), c_s, v_s in zip(merged_tuples, frames[2].smiles, frames[1].smiles):
        seen_k = set()
        res = []
        # 1. Primary order (Variant M merged + DocPrior reordered ranks 2-25)
        for s in prim_list:
            if not s: continue
            k = _to_k14(s)
            if not k or k not in seen_k:
                if k: seen_k.add(k)
                res.append(s)
                if len(res) >= 25: break

        # 2. In-pool backfill strictly from c and v (no outside distractors)
        if len(res) < 25:
            for extra in (c_s, v_s):
                for s in extra.split(';'):
                    if not s or s in res: continue
                    k = _to_k14(s)
                    if not k or k not in seen_k:
                        if k: seen_k.add(k)
                        res.append(s)
                        dedup_swaps += 1
                        if len(res) >= 25: break
                if len(res) >= 25: break

        # 2b. Empty slots: scored outside candidates with a new metric identity, best neural score first
        if len(res) < 25:
            for s in out_by_mid.get(str(mid), []):
                if s in res:
                    continue
                k = _to_k14(s)
                if not k or k in seen_k:
                    continue
                seen_k.add(k)
                res.append(s)
                outsider_fills += 1
                if len(res) >= 25:
                    break

        # 2c. LOTUS Natural Product Backfill: fill vacant slots from mass-matched natural products
        if len(res) < 25 and lotus_by_mid:
            for s in lotus_by_mid.get(str(mid), []):
                if s in res:
                    continue
                k = _to_k14(s)
                if not k or k in seen_k:
                    continue
                seen_k.add(k)
                res.append(s)
                lotus_fills += 1
                if len(res) >= 25:
                    break

        # 3. Fallback fill if still < 25
        if len(res) < 25:
            for extra in (prim_list, c_s.split(';'), v_s.split(';')):
                for s in extra:
                    if s and s not in res:
                        res.append(s)
                        if len(res) >= 25: break
                if len(res) >= 25: break

        reclaimed_smiles.append(';'.join(res[:25]))

    new = pd.DataFrame({'molecule_id': [m for m, _ in merged_tuples], 'smiles': reclaimed_smiles})
    rep['dedup_swaps'] = dedup_swaps
    rep['outsider_fills'] = outsider_fills
    rep['lotus_fills'] = lotus_fills
    print(f'LOTUS Backfill: added {lotus_fills} verified natural products to vacant slots.')
    rep['old_leader_pinned'] = counts.get('old_leader_pinned', 0)
    print(f'Final layer: outsider fills {outsider_fills}, old leader pinned {rep["old_leader_pinned"]}, '
          f'library-hit molecules {rep.get("lib_hit_molecules")}, POP_W {POP_W}, NP_W {NP_W}')
    rep['ranks_reordered'] = counts['ranks_reordered']
    print(f'InChIKey14 Deduplication: reclaimed {dedup_swaps} redundant isomer slots!')
    print(f'Documentation Prior: reordered secondary ranks in {counts["ranks_reordered"]} / {counts["rows"]} rows.')

    assert new.molecule_id.tolist() == SAMPLE_IDS
    assert new.smiles.str.split(';').map(len).between(1, 25).all()
    assert new.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
    tmp = ROOT / 'submission.merge.tmp'
    new.to_csv(tmp, index=False)
    out_sha = _pxm_sha(tmp)
    os.replace(tmp, sub_path)
    rep.update(status='applied', **counts, output_sha256=out_sha,
               output_equals_custom=out_sha == rep['input_sha256']['custom'])
    return rep


try:
    if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
        MERGE_REPORT = {'status': 'skipped_fast_commit'}
    else:
        MERGE_REPORT = _pxm_main()
except Exception as exc:
    traceback.print_exc()
    MERGE_REPORT = dict(status='restored_custom_after_error', exception=repr(exc)[:600])
    try:
        _tmp = ROOT / 'submission.merge.tmp'
        shutil.copyfile(ROOT / 'variants/submission_CUSTOM.csv', _tmp)
        os.replace(_tmp, ROOT / 'submission.csv')
        MERGE_REPORT['output_sha256'] = _pxm_sha(ROOT / 'submission.csv')
    except Exception as exc2:
        traceback.print_exc()
        MERGE_REPORT['restore_error'] = repr(exc2)[:300]
try:
    _tmp = ROOT / 'merge_report.json.tmp'
    _tmp.write_text(json.dumps(MERGE_REPORT, indent=2, default=str))
    os.replace(_tmp, ROOT / 'merge_report.json')
except Exception:
    traceback.print_exc()

final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), 'MERGE', json.dumps(
    {k: MERGE_REPORT.get(k) for k in ('status', 'rows', 'merged', 'leader_moves', 'ranks_reordered', 'dedup_swaps',
                                      'output_equals_custom', 'outsider_fills', 'old_leader_pinned', 'lib_hit_molecules', 'output_sha256', 'exception') if k in MERGE_REPORT},
    default=str))


In [ ]:
# ── v8 · mass rank-1 rule + learned isomer reranker (romanrozen), guarded and checked ───────────────────────────────
import base64, zlib, hashlib, json, shutil, sys, time, traceback
_RR8_FEATURES = '"""Spectrum <-> candidate features for a learned same-formula isomer reranker (v8, romanrozen).\n\nBuilt on Spectral Parallax\'s MetFrag-lite enumeration (frag_rescore.py, imported as `frag`): every candidate gives the\nneutral masses of the pieces left after removing 0, 1 or 2 bonds of its heavy-atom graph. Each peak of each spectrum is\nmatched against `piece + dh*H + charge carrier` (dh -2..+3, the carriers its adduct allows) at two tolerances, and the\nfeatures describe how much of the spectrum the candidate explains, how cheaply (fewest cleavages), how precisely (mass\nerror), which peaks (top-k, high mass), plus agreement between neutral losses seen in the spectrum and the candidate\'s\nfunctional groups (sugars, acids, methoxy, ...). Features are averaged over the molecule\'s spectra, then the ranker sees\nthem both raw and centred within the candidate list (a list = the isomers being compared).\n\n    features_for_list(spectra, smiles_list, cache) -> np.ndarray [n_candidates, N_FEATURES] (NaN rows: unparsable)\n    FEATURE_NAMES\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\nimport frag_rescore as frag\n\nH = frag.H\nTOLS = (0.005, 0.012)\nH_SHIFTS = tuple(range(-2, 4))\nLOSSES = {                      # name: (neutral loss mass, SMARTS a candidate needs to lose it)\n    \'h2o\': (18.010565, \'[OX2H1]\'),\n    \'co2\': (43.989829, \'[CX3](=O)[OX2H1,OX1-]\'),\n    \'hcooh\': (46.005479, \'[CX3](=O)[OX2H1,OX1-]\'),\n    \'nh3\': (17.026549, \'[NX3;H2,H1;!$(NC=O)]\'),\n    \'ch3\': (15.023475, \'[CH3][OX2,NX3]\'),\n    \'ch3oh\': (32.026215, \'[CH3][OX2]\'),\n    \'c2h2o\': (42.010565, \'[CH3]C(=O)[O,N]\'),\n    \'hexose\': (162.052824, \'[OX2;R][CX4;R]1[OX2;R][CX4;R][CX4;R]([OX2])[CX4;R]([OX2])[CX4;R]1[OX2]\'),\n    \'dhex\': (146.057909, \'C[CX4;R]1[OX2;R][CX4;R]([OX2])[CX4;R]([OX2])[CX4;R]([OX2])[CX4;R]1[OX2]\'),\n    \'pentose\': (132.042259, \'[OX2][CX4;R]1[OX2;R][CX4;R][CX4;R]([OX2])[CX4;R]1[OX2]\'),\n    \'glcA\': (176.032088, \'OC(=O)[CX4;R]1[OX2;R][CX4;R]([OX2])[CX4;R]([OX2])[CX4;R]([OX2])[CX4;R]1[OX2]\'),\n    \'so3\': (79.956815, \'S(=O)(=O)[OX2H1,OX1-,OX2]\'),\n    \'co\': (27.994915, \'[CX3]=O\'),\n}\n_LOSS_PATTERNS = None\n\n\ndef _patterns():\n    global _LOSS_PATTERNS\n    if _LOSS_PATTERNS is None:\n        from rdkit import Chem\n        _LOSS_PATTERNS = {k: Chem.MolFromSmarts(v[1]) for k, v in LOSSES.items()}\n    return _LOSS_PATTERNS\n\n\nSPEC_FEATURES = [\'expl_k1_lin\', \'expl_k2_lin\', \'expl_k1_sqrt\', \'expl_k2_sqrt\', \'expl_k2_lin_tight\', \'expl_k2_sqrt_tight\',\n                 \'top5_k2\', \'top10_k2\', \'top10_k1\', \'top3_k1\', \'npeaks_k2\', \'hi_mass_k2\', \'lo_mass_k2\', \'h0_share\',\n                 \'mass_err_mda\', \'k2_only_share\']\nCAND_FEATURES = [\'log_n_k1\', \'log_n_k2\', \'heavy_atoms\', \'n_rings\', \'frac_aromatic\', \'n_O\', \'n_N\']\nLOSS_FEATURES = [f\'loss_{k}_{t}\' for k in LOSSES for t in (\'agree\', \'clash\')]\nFEATURE_NAMES = SPEC_FEATURES + CAND_FEATURES + LOSS_FEATURES\nN_FEATURES = len(FEATURE_NAMES)\n\n\ndef prep_spectrum(s):\n    """{mz, it, prec, adduct, mode} -> arrays used for matching (None when empty)."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    prec = float(s.get(\'prec\') or 0.0)\n    if prec > 0:\n        ok &= mz <= prec + 2.0\n    mz, it = mz[ok], it[ok]\n    if len(mz) < 2:\n        return None\n    order = np.argsort(-it)\n    mz, it = mz[order][:150], it[order][:150]\n    car = np.asarray(frag._carriers(s.get(\'adduct\'), s.get(\'mode\')), np.float64)\n    shifts = np.array([c + dh * H for c in car for dh in H_SHIFTS])\n    dh_of = np.array([dh for c in car for dh in H_SHIFTS])\n    return dict(mz=mz, it=it, lin=it / it.sum(), sq=np.sqrt(it) / np.sqrt(it).sum(), prec=prec if prec > 0 else float(mz.max()),\n                shifts=shifts, dh=dh_of, adduct=str(s.get(\'adduct\') or \'\'))\n\n\ndef _match(levels, p, tol):\n    """Per peak: fewest cleavages explaining it (0/1/2, 9 = none), smallest |error| (Da) and its dh."""\n    npk = len(p[\'mz\'])\n    q = p[\'mz\'][None, :] - p[\'shifts\'][:, None]                        # [n_shift, n_peak] neutral masses\n    best_k = np.full(npk, 9, np.int8)\n    best_err = np.full(npk, np.inf)\n    best_dh = np.zeros(npk, np.int8)\n    for k, arr in enumerate(levels[:3]):\n        if arr is None or len(arr) == 0:\n            continue\n        idx = np.searchsorted(arr, q)\n        lo = np.clip(idx - 1, 0, len(arr) - 1)\n        hi = np.clip(idx, 0, len(arr) - 1)\n        err = np.minimum(np.abs(arr[lo] - q), np.abs(arr[hi] - q))   # [n_shift, n_peak]\n        jmin = err.argmin(axis=0)\n        e = err[jmin, np.arange(npk)]\n        hit = (e <= tol) & (best_k == 9)\n        best_k[hit] = k\n        best_err[hit] = e[hit]\n        best_dh[hit] = p[\'dh\'][jmin[hit]]\n    return best_k, best_err, best_dh\n\n\ndef _spec_features(levels, p):\n    out = {}\n    for tol, tag in ((TOLS[1], \'\'), (TOLS[0], \'_tight\')):\n        k, err, dh = _match(levels, p, tol)\n        e1, e2 = k <= 1, k <= 2\n        if tag == \'\':\n            out[\'expl_k1_lin\'] = float(p[\'lin\'] @ e1)\n            out[\'expl_k2_lin\'] = float(p[\'lin\'] @ e2)\n            out[\'expl_k1_sqrt\'] = float(p[\'sq\'] @ e1)\n            out[\'expl_k2_sqrt\'] = float(p[\'sq\'] @ e2)\n            out[\'top5_k2\'] = float(e2[:5].mean())\n            out[\'top10_k2\'] = float(e2[:10].mean())\n            out[\'top10_k1\'] = float(e1[:10].mean())\n            out[\'top3_k1\'] = float(e1[:3].mean())\n            out[\'npeaks_k2\'] = float(e2.mean())\n            hi = p[\'mz\'] >= 0.5 * p[\'prec\']\n            out[\'hi_mass_k2\'] = float(p[\'lin\'][hi] @ e2[hi] / max(p[\'lin\'][hi].sum(), 1e-9)) if hi.any() else 0.0\n            lo = ~hi\n            out[\'lo_mass_k2\'] = float(p[\'lin\'][lo] @ e2[lo] / max(p[\'lin\'][lo].sum(), 1e-9)) if lo.any() else 0.0\n            w = p[\'lin\'][e2]\n            out[\'h0_share\'] = float(w @ (np.abs(dh[e2]) <= 1) / max(w.sum(), 1e-9)) if e2.any() else 0.0\n            out[\'mass_err_mda\'] = float(1000 * (w @ err[e2]) / max(w.sum(), 1e-9)) if e2.any() else TOLS[1] * 1000\n            out[\'k2_only_share\'] = float(p[\'lin\'] @ (k == 2))\n        else:\n            out[\'expl_k2_lin_tight\'] = float(p[\'lin\'] @ e2)\n            out[\'expl_k2_sqrt_tight\'] = float(p[\'sq\'] @ e2)\n    return out\n\n\ndef _observed_losses(p, tol=0.006):\n    """{loss: intensity share of peaks at precursor - loss} for one spectrum (precursor m/z taken as given)."""\n    out = {}\n    for name, (m, _) in LOSSES.items():\n        target = p[\'prec\'] - m\n        hit = np.abs(p[\'mz\'] - target) <= tol\n        out[name] = float(p[\'lin\'][hit].sum())\n    return out\n\n\ndef _cand_facts(smi):\n    from rdkit import Chem\n    m = Chem.MolFromSmiles(smi)\n    if m is None:\n        return None\n    pats = _patterns()\n    groups = {k: int(m.HasSubstructMatch(pt)) if pt is not None else 0 for k, pt in pats.items()}\n    ri = m.GetRingInfo()\n    heavy = m.GetNumHeavyAtoms()\n    arom = sum(a.GetIsAromatic() for a in m.GetAtoms())\n    sym = [a.GetSymbol() for a in m.GetAtoms()]\n    return dict(groups=groups, heavy_atoms=heavy, n_rings=ri.NumRings(), frac_aromatic=arom / max(heavy, 1),\n                n_O=sym.count(\'O\'), n_N=sym.count(\'N\'))\n\n\ndef features_for_list(spectra, smiles_list, cache=None, cfg_breaks=2):\n    """Feature matrix for candidates of one molecule (rows of NaN for unparsable SMILES)."""\n    cache = {} if cache is None else cache\n    preps = [p for p in (prep_spectrum(s) for s in (spectra or [])) if p is not None]\n    obs = None\n    if preps:\n        ol = [_observed_losses(p) for p in preps]\n        obs = {k: float(np.mean([o[k] for o in ol])) for k in LOSSES}\n    X = np.full((len(smiles_list), N_FEATURES), np.nan)\n    c = frag.resolve_cfg(dict(max_breaks=cfg_breaks))\n    for i, smi in enumerate(smiles_list):\n        key = (\'lv\', smi)\n        if key not in cache:\n            cache[key] = frag._levels_cached(smi, c, None)\n        lv = cache[key]\n        fkey = (\'facts\', smi)\n        if fkey not in cache:\n            try:\n                cache[fkey] = _cand_facts(smi)\n            except Exception:\n                cache[fkey] = None\n        facts = cache[fkey]\n        if lv is None or facts is None:\n            continue\n        row = {}\n        if preps:\n            sf = [_spec_features(lv, p) for p in preps]\n            for name in SPEC_FEATURES:\n                row[name] = float(np.mean([d[name] for d in sf]))\n        else:\n            for name in SPEC_FEATURES:\n                row[name] = 0.0\n        row[\'log_n_k1\'] = float(np.log1p(len(lv[1]) if len(lv) > 1 else 0))\n        row[\'log_n_k2\'] = float(np.log1p(len(lv[2]) if len(lv) > 2 else 0))\n        for name in (\'heavy_atoms\', \'n_rings\', \'frac_aromatic\', \'n_O\', \'n_N\'):\n            row[name] = float(facts[name])\n        for name in LOSSES:\n            seen = obs[name] if obs else 0.0\n            has = facts[\'groups\'][name]\n            row[f\'loss_{name}_agree\'] = seen * has\n            row[f\'loss_{name}_clash\'] = seen * (1 - has)\n        X[i] = [row[n] for n in FEATURE_NAMES]\n    return X\n\n\ndef centre_within(X):\n    """Candidate-list-relative copy: each column minus its list mean (NaN-safe)."""\n    with np.errstate(all=\'ignore\'):\n        mu = np.nanmean(X, axis=0, keepdims=True)\n    return X - np.where(np.isfinite(mu), mu, 0.0)\n\n\ndef full_features(spectra, smiles_list, cache=None):\n    X = features_for_list(spectra, smiles_list, cache)\n    return np.hstack([X, centre_within(X)])\n\n\nFULL_FEATURE_NAMES = FEATURE_NAMES + [f\'{n}_c\' for n in FEATURE_NAMES]\n'
_RR8_OVERLAY = '"""Deploy the v8 isomer reranker on a finished submission, with RR3-style rails (romanrozen).\n\nFor every molecule whose best library match is below `lib_tau`:\n  * G = the candidates of its final list that share the leader\'s molecular formula (the isomers the model compares);\n  * training-set structures in G keep their slots (FREEZE: a model trained on library spectra must not demote the\n    structures that do have library spectra); the other members of G are re-ordered among their own slots by\n        fused = z(-log position) + alpha * z(model score)            (z over the movable members)\n  * everything outside G, and every list with a strong library match, stays as it is.\nCandidate sets never change. Returns the new frame and a report; any error leaves the input unchanged (caller\'s job).\n"""\nfrom __future__ import annotations\n\nimport math\nimport time\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _z(v):\n    v = np.asarray(v, float)\n    sd = v.std()\n    return (v - v.mean()) / sd if sd > 1e-12 else np.zeros_like(v)\n\n\ndef load_spectra(test_path):\n    t = pd.read_parquet(test_path)\n    out = {}\n    for r in t.itertuples(index=False):\n        ce = getattr(r, \'collision_energy_ev\', None)\n        try:\n            ce = float(np.mean(ce)) if ce is not None and len(ce) else float(\'nan\')\n        except Exception:\n            ce = float(\'nan\')\n        out.setdefault(str(r.molecule_id), []).append(dict(\n            mz=np.asarray(r.ms2_mzs, float), it=np.asarray(r.ms2_normalized_intensities, float),\n            prec=float(r.precursor_mz), adduct=str(r.adduct),\n            mode=1 if str(getattr(r, \'ionization_mode\', \'positive\')).lower() == \'positive\' else -1, ce=ce))\n    return out\n\n\ndef model_features(rf, frag, spectra, smiles, cache, fcache):\n    X = rf.full_features(spectra, smiles, cache)\n    m1 = frag.score_molecule(spectra, smiles, cfg=None, cache=fcache)\n    m2 = frag.score_molecule(spectra, smiles, cfg=frag.BASELINE_CFG, cache=fcache)\n    M = np.array([[np.nan if a is None else a, np.nan if b is None else b] for a, b in zip(m1, m2)], float)\n    with np.errstate(all=\'ignore\'):\n        Mc = M - np.nanmean(M, axis=0, keepdims=True)\n    return np.hstack([X, M, Mc])\n\n\ndef rerank_list(smiles, scores, frozen, alpha):\n    """smiles: the full list; scores/frozen: dicts for members of G (same formula as the leader). Returns new list."""\n    pos = {s: i for i, s in enumerate(smiles)}\n    movable = [s for s in scores if not frozen.get(s, False)]\n    if len(movable) < 2:\n        return list(smiles)\n    slots = sorted(pos[s] for s in movable)\n    base = _z([-math.log(pos[s] + 1) for s in movable])\n    rr = _z([scores[s] for s in movable])\n    fused = base + float(alpha) * rr\n    order = [movable[i] for i in np.argsort(-fused, kind=\'stable\')]\n    out = list(smiles)\n    for slot, s in zip(slots, order):\n        out[slot] = s\n    return out\n\n\ndef run(sub, lib_max, spectra, train_keys, booster, rf, frag, alpha=1.0, lib_tau=0.9, max_seconds=1500, log=print):\n    """sub: DataFrame molecule_id, smiles (\';\'-joined). Returns (new DataFrame, report)."""\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    RDLogger.DisableLog(\'rdApp.*\')\n    t0 = time.monotonic()\n    info = {}\n\n    def facts(s):\n        if s not in info:\n            m = Chem.MolFromSmiles(s)\n            if m is None:\n                info[s] = (None, None)\n            else:\n                try:\n                    k = Chem.MolToInchiKey(m)[:14]\n                except Exception:\n                    k = None\n                info[s] = (CalcMolFormula(m), k)\n        return info[s]\n\n    rep = dict(molecules=len(sub), reranked=0, changed=0, top1_changed=0, skipped_library=0, skipped_group=0,\n               skipped_frozen=0, no_spectra=0, failed=0, timed_out=0, alpha=float(alpha), lib_tau=float(lib_tau))\n    cache, fcache, rows = {}, {}, []\n    for mid, joined in zip(sub.molecule_id.astype(str), sub.smiles.astype(str)):\n        smis = [s for s in joined.split(\';\') if s]\n        new = smis\n        if float(lib_max.get(mid, 0.0)) >= lib_tau:\n            rep[\'skipped_library\'] += 1\n        elif mid not in spectra:\n            rep[\'no_spectra\'] += 1\n        elif time.monotonic() - t0 > max_seconds:\n            rep[\'timed_out\'] += 1\n        elif len(smis) >= 2:\n            try:\n                f0 = facts(smis[0])[0]\n                group = [s for s in smis if f0 is not None and facts(s)[0] == f0]\n                frozen = {s: (facts(s)[1] in train_keys) for s in group}\n                if len(group) < 2:\n                    rep[\'skipped_group\'] += 1\n                elif sum(not v for v in frozen.values()) < 2:\n                    rep[\'skipped_frozen\'] += 1\n                else:\n                    F = model_features(rf, frag, spectra[mid], group, cache, fcache)\n                    sc = booster.predict(F)\n                    scores = {s: float(v) for s, v in zip(group, sc)}\n                    new = rerank_list(smis, scores, frozen, alpha)\n                    rep[\'reranked\'] += 1\n            except Exception as exc:\n                rep[\'failed\'] += 1\n                if rep[\'failed\'] <= 3:\n                    log(f\'[rr8] {mid}: {type(exc).__name__}: {exc}\')\n                new = smis\n        if new != smis:\n            rep[\'changed\'] += 1\n            rep[\'top1_changed\'] += int(new[0] != smis[0])\n        rows.append((mid, \';\'.join(new) if new else joined))\n        if len(cache) > 30000:\n            cache.clear()\n        if len(fcache) > 30000:\n            fcache.clear()\n    rep[\'seconds\'] = round(time.monotonic() - t0, 1)\n    return pd.DataFrame(rows, columns=[\'molecule_id\', \'smiles\']), rep\n'
_RR8_MODEL_Z = '...'
assert hashlib.sha256(_RR8_FEATURES.encode()).hexdigest() == 'aec9012e72324095308af81c1408aa36f6a44cb00f6d4389e87529a68329d176'
assert hashlib.sha256(_RR8_OVERLAY.encode()).hexdigest() == 'e83b7986c46d0ef02bc6cfde17e891c9e7ee66a0eed8c716f5e3474e5663ccf7'


def _v8_lists(path):
    f = pd.read_csv(path, dtype=str, keep_default_na=False)
    return f, [s.split(';') for s in f.smiles.astype(str)]


def _v8_check(before, after):
    return (after.molecule_id.astype(str).tolist() == before.molecule_id.astype(str).tolist() == SAMPLE_IDS
            and all(set(x) == set(y) and len(x) == len(y) == len(set(y)) and all(y)
                    for x, y in zip(before.smiles.str.split(';'), after.smiles.str.split(';'))))


def _v8_write(frame):
    frame.to_csv(ROOT / 'submission.tmp.csv', index=False)
    os.replace(ROOT / 'submission.tmp.csv', ROOT / 'submission.csv')


try:
    _v8_ev = pd.read_parquet(ROOT / 'frozen_candidate_evidence.parquet', columns=['molecule_id', 'library_max'])
    V8_LIB = {str(m): float(v) for m, v in _v8_ev.groupby('molecule_id').library_max.max().items()}
except Exception:
    V8_LIB = {}

# 1) mass rank-1 rule
MASS_LEAD_REPORT = dict(status='off in this preset')
try:
    if MASS_ON and MASS_PPM:
        shutil.copyfile(ROOT / 'submission.csv', ROOT / 'submission_before_mass_lead.csv')
        _f, _lists = _v8_lists(ROOT / 'submission.csv')
        _rows, _moves = [], 0
        for _mid, _l in zip(_f.molecule_id.astype(str), _lists):
            ppm = MASS_PPM.get(_mid, {})
            if (len(_l) >= 2 and V8_LIB.get(_mid, 0.0) < MASS_LIB_TAU and _l[0] in ppm
                    and abs(ppm[_l[0]] - MASS_CENTER) > MASS_LEAD_OFF):
                for _s in _l[1:3]:
                    if _s in ppm and abs(ppm[_s] - MASS_CENTER) <= MASS_FREE:
                        _l = [_s] + [x for x in _l if x != _s]
                        _moves += 1
                        break
            _rows.append((_mid, ';'.join(_l)))
        _new = pd.DataFrame(_rows, columns=['molecule_id', 'smiles'])
        if _v8_check(_f, _new):
            _v8_write(_new)
            MASS_LEAD_REPORT = dict(status='ran', leader_moves=_moves)
        else:
            MASS_LEAD_REPORT = dict(status='check failed: answer kept')
    elif MASS_ON:
        MASS_LEAD_REPORT = dict(status='off: no mass table')
except Exception as exc:
    traceback.print_exc()
    MASS_LEAD_REPORT = dict(status='off after an error', exception=repr(exc)[:300])
    if (ROOT / 'submission_before_mass_lead.csv').is_file():
        shutil.copyfile(ROOT / 'submission_before_mass_lead.csv', ROOT / 'submission.csv')

# 2) learned isomer reranker
RR8_REPORT = dict(status='off in this preset')
_rr8_before = ROOT / 'submission_before_rr8.csv'
try:
    shutil.copyfile(ROOT / 'submission.csv', _rr8_before)
    _rr8_left = RR8_LIMIT_SECONDS - (time.time() - NB_T0)
    if not RR8:
        pass
    elif not globals().get('PIPELINE_DONE'):
        RR8_REPORT = dict(status='off: no pipeline answer in this run')
    elif _rr8_left < RR8_MIN_SECONDS:
        RR8_REPORT = dict(status='skipped: session time', seconds_left=round(_rr8_left))
    else:
        (CODE / 'rr_features.py').write_text(_RR8_FEATURES)
        (CODE / 'rr_overlay.py').write_text(_RR8_OVERLAY)
        for _p in (str(CODE), str(ROOT)):
            if _p not in sys.path:
                sys.path.insert(0, _p)
        for _m in ('rr_features', 'rr_overlay'):
            sys.modules.pop(_m, None)
        import frag_rescore as _rr8_frag
        import rr_features as _rr8_rf
        import rr_overlay as _rr8_ov
        import lightgbm as _rr8_lgb
        _rr8_txt = zlib.decompress(base64.b64decode(_RR8_MODEL_Z)).decode()
        assert hashlib.sha256(_rr8_txt.encode()).hexdigest() == '3b80e1c9adf90721c99bb4e817a05379dd8b26e0988a001bd112330a66d9122b', 'model text differs'
        _rr8_booster = _rr8_lgb.Booster(model_str=_rr8_txt)
        _rr8_frag.warmup()
        _rr8_sub, _ = _v8_lists(ROOT / 'submission.csv')
        _rr8_spec = _rr8_ov.load_spectra(COMP / 'test.parquet')
        _rr8_train = set(pd.read_parquet(COMP / 'train.parquet', columns=['inchikey14']).inchikey14.astype(str))
        _rr8_new, _rr8_rep = _rr8_ov.run(_rr8_sub, V8_LIB, _rr8_spec, _rr8_train, _rr8_booster, _rr8_rf, _rr8_frag,
                                         alpha=RR8_ALPHA, lib_tau=RR8_LIB_TAU,
                                         max_seconds=float(min(RR8_BUDGET_SECONDS, _rr8_left)))
        if _v8_check(_rr8_sub, _rr8_new):
            _v8_write(_rr8_new)
            RR8_REPORT = dict(status='ran', **_rr8_rep)
        else:
            RR8_REPORT = dict(status='check failed: answer kept', **_rr8_rep)
except Exception as exc:
    traceback.print_exc()
    RR8_REPORT = dict(status='off after an error', exception=repr(exc)[:400])
    try:
        if _rr8_before.is_file():
            shutil.copyfile(_rr8_before, ROOT / 'submission.csv')
    except Exception:
        traceback.print_exc()
try:
    (ROOT / 'v8_report.json').write_text(json.dumps(dict(mass=MASS_REPORT, mass_lead=MASS_LEAD_REPORT, rr=RR8_REPORT),
                                                    indent=2, default=str))
except Exception:
    traceback.print_exc()
final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
assert list(final.columns) == ['molecule_id', 'smiles']
assert final.molecule_id.tolist() == SAMPLE_IDS and final.molecule_id.is_unique
assert final.smiles.str.split(';').map(len).between(1, 25).all()
assert final.smiles.str.split(';').map(lambda a: all(bool(s) for s in a) and len(set(a)) == len(a)).all()
print('SUBMISSION READY:', ROOT / 'submission.csv', 'molecules', len(final), '| V8', V8, '| mass lead',
      json.dumps(MASS_LEAD_REPORT), '| RR', json.dumps(RR8_REPORT, default=str))

In [ ]:
vh_snap('a_pre_network')  # VH: copy of submission.csv at this stage boundary


In [ ]:
_NNET_NOTEBOOK = True
"""N-1c lambda 0.35 neighbour network with structure switches and a rank-1 lock (H-A-152 s2).
Purpose: rerank candidates with molecular-network neighbour evidence; optional switches
  lib  (A) fixed known-structure nodes from competition train.parquet,
  mass (B) mass-difference (edit) plausibility weight on edges,
  steps(C) two-step propagation (neighbours' step-1 reranked top-5 as evidence),
  lock1    keep the plain N-1c (all switches off) leader at rank 1; the arm only reorders ranks 2-25
           (an arm leader that differs goes to rank 2, the rest shift down).
With every switch off the arithmetic is the n1c035_cell_v1 cell's (base gate). Copy of n035net_cell_v1.py 1.0.0
plus lock_rank1 and the lock1 argument of run() (= n035netL1_cell_v1.py L1-1.0.0, evaluated in H-A-152 s2), and here
(kernel version) a batched load_library (two passes, groups reduced as soon as their rows are read; same library arrays as
the one-shot 1.0.0 loader, peak RAM about 46 GB -> a few GB), a memory guard (_NNET_MIN_FREE_GB available RAM) in the
library stage and a 1800 s budget. Ranking functions are unchanged.
Usage: import definitions for CPU replays; notebook calls run() after PIPELINE_DONE.
Inputs: submission.csv; competition test.parquet (+ train.parquet when lib is on).
Outputs: atomic submission.csv when changed; n035net_audit.json; pre-change copy. Any error, timeout or memory guard keeps
the input bytes.
Version: L1-1.1.0 (2026-11-17)
"""
_NNET_ON = True
_NNET_LAM = 0.35
_NNET_CAP_S = 1800.0
_NNET_ARM = "D"
_NNET_LOCK1 = True
_NNET_VERSION = "L1-1.1.0"
_NNET_LIB_BATCH = 65536  # parquet rows per peak batch of the library build
_NNET_MIN_FREE_GB = 2.0  # memory guard: the library stage stops (input bytes kept) below this much available RAM
_NNET_ARMS = {"base": (False, False, 1), "A": (True, False, 1), "B": (False, True, 1),
              "C": (False, False, 2), "D": (True, True, 2)}
_NNET_LIB_K = 5          # fixed nodes kept per query
_NNET_LIB_M = 100        # exact checks per query and adduct after the prefilter
_NNET_LIB_ROWS = 8       # train rows merged per (inchikey14, adduct)
_NNET_OFF_EDIT = 0.5     # edge weight factor when the precursor difference is no known edit
# H-A-135 s1 edit masses (v4b transforms + common natural-product edits), Da
_NNET_EDITS = (2.01565, 14.01565, 15.0109, 15.99491, 18.01056, 28.0313, 30.01056, 31.98983, 33.96103,
               42.01056, 43.98983, 68.0626, 79.95681, 86.00039, 104.02621, 132.04226, 146.03678,
               146.05791, 152.01096, 162.03169, 162.05282, 176.03209, 176.04734)
_NNET_ADDUCTS = ("[M+H]+", "[M-H]-")


def _nnet_timecheck(deadline):
    import time
    if deadline is not None and time.monotonic() >= deadline:
        raise TimeoutError("NNET time budget exhausted")


def _nnet_memcheck(where):
    try:
        import psutil
        free = psutil.virtual_memory().available / 2 ** 30
    except Exception:
        return
    if free < _NNET_MIN_FREE_GB:
        raise MemoryError("NNET memory guard: %.2f GB available at %s" % (free, where))


# ---- unchanged N-1c pieces (same arithmetic as n1c035_cell_v1) ----
def build_spectra(test, allowed, deadline=None):
    import numpy as np
    import pandas as pd
    required = {"molecule_id", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"}
    if not required.issubset(test.columns):
        raise ValueError("missing spectrum columns")
    table = test.copy()
    table["_n035_id"] = table["molecule_id"].astype(str)
    spectra = {}
    for (mid, adduct), group in table.groupby(["_n035_id", "adduct"]):
        _nnet_timecheck(deadline)
        if mid not in allowed or str(adduct) not in ("[M+H]+", "[M-H]-"):
            continue
        prec = float(np.median(pd.to_numeric(group["precursor_mz"], errors="coerce")))
        if not np.isfinite(prec):
            raise ValueError("invalid precursor")
        mass_rows, intensity_rows = [], []
        for masses, weights in zip(group["ms2_mzs"], group["ms2_normalized_intensities"]):
            masses, weights = np.asarray(masses, dtype=float), np.asarray(weights, dtype=float)
            if masses.ndim != 1 or weights.ndim != 1 or len(masses) != len(weights):
                raise ValueError("invalid peak dimensions")
            if not np.all(np.isfinite(masses)) or not np.all(np.isfinite(weights)) or np.any(weights < 0):
                raise ValueError("invalid peak values")
            mass_rows.append(masses)
            intensity_rows.append(weights / max(float(weights.max()) if len(weights) else 1.0, 1e-9))
        masses = np.concatenate(mass_rows)
        weights = np.concatenate(intensity_rows)
        keep = masses < prec - 1.5
        masses, weights = masses[keep], weights[keep]
        if len(masses) == 0:
            continue
        bins, inverse = np.unique(np.round(masses, 2), return_inverse=True)
        maxima = np.zeros(len(bins))
        np.maximum.at(maxima, inverse, weights)
        strongest = np.argsort(-maxima)[:120]
        binned = bins[strongest]
        amplitude = np.sqrt(maxima[strongest])
        amplitude /= max(np.linalg.norm(amplitude), 1e-12)
        order = np.argsort(binned)
        spectra.setdefault(mid, {})[str(adduct)] = (binned[order], amplitude[order], prec)
    return spectra


def modified_cosine(left, right, deadline=None):
    import numpy as np
    a_mass, a_weight, a_prec = left
    b_mass, b_weight, b_prec = right
    options = []
    for i, mass in enumerate(a_mass):
        _nnet_timecheck(deadline)
        for delta in (0.0, a_prec - b_prec):
            lower = np.searchsorted(b_mass, mass - delta - 0.0101)
            upper = np.searchsorted(b_mass, mass - delta + 0.0101, side="right")
            options.extend((a_weight[i] * b_weight[j], i, j) for j in range(lower, upper))
    options.sort(reverse=True)
    used_left, used_right = set(), set()
    total, matches = 0.0, 0
    for weight, i, j in options:
        if i not in used_left and j not in used_right:
            used_left.add(i)
            used_right.add(j)
            total += weight
            matches += 1
    return total, matches


def build_graph(spectra, deadline=None, tau=0.6):
    ids = sorted(spectra)
    graph = {}
    for index, left in enumerate(ids):
        _nnet_timecheck(deadline)
        for right in ids[index + 1:]:
            best = None
            for adduct in sorted(set(spectra[left]).intersection(spectra[right])):
                cosine, matches = modified_cosine(spectra[left][adduct], spectra[right][adduct], deadline)
                if matches >= 6 and cosine >= tau and (best is None or cosine > best):
                    best = cosine
            if best is not None:
                graph.setdefault(left, []).append((right, best))
                graph.setdefault(right, []).append((left, best))
    return graph


def rerank_lists(lists, scores, lam=0.35):
    import numpy as np
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("invalid lambda")
    output = {}
    for mid, candidates in lists.items():
        current = list(candidates)
        if lam and mid in scores and len(current) > 1:
            neighbour = np.asarray(scores[mid], dtype=float)
            if neighbour.shape != (len(current),) or not np.all(np.isfinite(neighbour)):
                raise ValueError("invalid neighbour scores")
            rank = -np.log(np.arange(1, len(current) + 1))
            standardize = lambda values: (values - values.mean()) / (values.std() + 1e-9)
            order = np.argsort(-(standardize(rank) + lam * standardize(neighbour)), kind="stable")
            current = [current[int(i)] for i in order]
        output[mid] = current
    return output


def _nnet_parse_submission(raw):
    import csv, io
    rows = list(csv.reader(io.StringIO(raw.decode("utf-8"), newline="")))
    if not rows or rows[0] != ["molecule_id", "smiles"] or len(rows) < 2:
        raise ValueError("invalid submission header or no rows")
    lists = {}
    for row in rows[1:]:
        if len(row) != 2 or not row[0] or row[0] in lists:
            raise ValueError("invalid or repeated molecule id")
        candidates = row[1].split(";")
        if not 1 <= len(candidates) <= 25 or any(not x.strip() for x in candidates):
            raise ValueError("invalid candidate list")
        lists[row[0]] = candidates
    return lists


# ---- B: mass-difference plausibility ----
def edit_plausible(delta, prec_a, prec_b):
    tol = max(0.01, 10e-6 * max(prec_a, prec_b))
    delta = abs(delta)
    return delta <= tol or any(abs(delta - edit) <= tol for edit in _NNET_EDITS)


def edge_deltas(spectra, graph, deadline=None, tau=0.6):
    """Precursor difference of each test-test edge in the adduct that gave its best cosine (build_graph rule)."""
    deltas = {}
    for left, neighbours in graph.items():
        for right, best in neighbours:
            key = (min(left, right), max(left, right))
            if key in deltas:
                continue
            _nnet_timecheck(deadline)
            chosen, value = None, None
            for adduct in sorted(set(spectra[key[0]]).intersection(spectra[key[1]])):
                cosine, matches = modified_cosine(spectra[key[0]][adduct], spectra[key[1]][adduct], deadline)
                if matches >= 6 and cosine >= tau and (value is None or cosine > value):
                    chosen, value = adduct, cosine
            if chosen is None or abs(value - best) > 1e-9:
                raise ValueError("edge adduct not recovered")
            pa, pb = spectra[key[0]][chosen][2], spectra[key[1]][chosen][2]
            deltas[key] = (abs(pa - pb), pa, pb)
    return deltas


# ---- A: library of known structures ----
def _nnet_lib_finish(m, w, g, gprec, n_groups):
    """Complete groups only: per (group, 0.01 Da bin) maximum of the normalised intensities below precursor - 1.5 (first
    maximum in file order), top 120 bins by (-max, bin), sqrt, L2 per group. Same steps as the one-shot 1.0.0 loader."""
    import numpy as np
    sel = m < gprec[g] - 1.5
    m, w, g = m[sel], w[sel], g[sel]
    b = np.rint(np.round(m, 2) * 100).astype(np.int64)
    comb = g.astype(np.int64) * 10_000_000 + b
    order = np.lexsort((-w, comb))
    comb_s, w_s = comb[order], w[order]
    first = np.ones(len(comb_s), dtype=bool)
    first[1:] = comb_s[1:] != comb_s[:-1]
    ucomb, umax = comb_s[first], w_s[first]
    ug, ub = ucomb // 10_000_000, ucomb % 10_000_000
    order = np.lexsort((ub, -umax, ug))
    ug, ub, umax = ug[order], ub[order], umax[order]
    start = np.ones(len(ug), dtype=bool)
    start[1:] = ug[1:] != ug[:-1]
    pos = np.arange(len(ug)) - np.maximum.accumulate(np.where(start, np.arange(len(ug)), 0))
    top = pos < 120
    ug, ub, amp = ug[top], ub[top], np.sqrt(umax[top])
    norm = np.sqrt(np.bincount(ug, weights=amp * amp, minlength=n_groups))
    amp = amp / np.maximum(norm[ug], 1e-12)
    return ug, ub, amp


def load_library(path, deadline=None, rows_per_group=_NNET_LIB_ROWS, batch_rows=_NNET_LIB_BATCH):
    """Consensus spectra per (inchikey14, adduct) from train.parquet, vectorised build_spectra rules, in two passes:
    (1) light columns of every row -> group codes, ranks, candidate rows (first rows_per_group with a finite precursor) and the
    last candidate row of each group; (2) peak columns in batches -> per-row checks and normalisation, peaks of kept rows wait
    until their group's last candidate row has been read, then the group is reduced. Same arrays as the one-shot loader."""
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq
    adducts = pa.array(list(_NNET_ADDUCTS))
    _nnet_memcheck("library start")
    light = pq.read_table(path, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz"], use_threads=False)
    _nnet_timecheck(deadline)
    light = light.filter(pc.is_in(light.column("adduct"), value_set=adducts))
    ik = np.asarray(light.column("inchikey14").to_pylist(), dtype=object)
    ad = np.asarray(light.column("adduct").to_pylist(), dtype=object)
    smi = light.column("normalized_smiles")
    prec_row = light.column("precursor_mz").to_numpy(zero_copy_only=False).astype(float)
    ok = np.array([isinstance(x, str) and len(x) > 0 for x in ik])
    key = pd.Series(ik).astype(str) + "|" + pd.Series(ad).astype(str)
    codes, _ = pd.factorize(key, sort=False)
    codes = np.where(ok, codes, -1)
    del key
    rank = pd.Series(codes).groupby(codes).cumcount().to_numpy()
    candidate = (codes >= 0) & (rank < rows_per_group) & np.isfinite(prec_row)
    n_groups = int(codes.max()) + 1 if len(codes) else 0
    last = np.full(n_groups, -1, dtype=np.int64)
    np.maximum.at(last, codes[candidate], np.flatnonzero(candidate))
    gprec = np.full(n_groups, np.nan)
    _nnet_timecheck(deadline)
    pend = dict(m=[], w=[], g=[], pr=[], pg=[])
    pieces = []
    read = 0
    source = pq.ParquetFile(path)
    for batch in source.iter_batches(batch_size=batch_rows, use_threads=False,
                                     columns=["adduct", "ms2_mzs", "ms2_normalized_intensities"]):
        _nnet_timecheck(deadline)
        _nnet_memcheck("library peaks")
        table = pa.Table.from_batches([batch])
        table = table.filter(pc.is_in(table.column("adduct"), value_set=adducts))
        lo, hi = read, read + table.num_rows
        read = hi
        if hi > lo:
            mz_col = table.column("ms2_mzs").combine_chunks()
            it_col = table.column("ms2_normalized_intensities").combine_chunks()
            n_mz = pc.fill_null(pc.list_value_length(mz_col), 0).to_numpy().astype(np.int64)
            n_it = pc.fill_null(pc.list_value_length(it_col), 0).to_numpy().astype(np.int64)
            flat_mz = pc.list_flatten(mz_col).to_numpy(zero_copy_only=False).astype(float)
            flat_it = pc.list_flatten(it_col).to_numpy(zero_copy_only=False).astype(float)
            row_of_peak = np.repeat(np.arange(len(n_mz)), n_mz)
            if len(flat_mz) != len(flat_it) or np.any(n_mz != n_it):
                raise ValueError("library peak arrays misaligned")
            bad = np.zeros(len(n_mz), dtype=bool)
            badpk = ~np.isfinite(flat_mz) | ~np.isfinite(flat_it) | (flat_it < 0)
            bad |= np.bincount(row_of_peak[badpk], minlength=len(n_mz)) > 0
            nonempty = n_mz > 0
            starts = (np.cumsum(n_mz) - n_mz)[nonempty]
            rowmax = np.zeros(len(n_mz))
            if len(starts):
                rowmax[nonempty] = np.maximum.reduceat(flat_it, starts)
            keep_row = candidate[lo:hi] & ~bad
            pk = keep_row[row_of_peak]
            m, w, r = flat_mz[pk], flat_it[pk], row_of_peak[pk]
            w = w / np.maximum(rowmax[r], 1e-9)
            pend["m"].append(m)
            pend["w"].append(w)
            pend["g"].append(codes[lo:hi][r])
            pend["pr"].append(prec_row[lo:hi][keep_row])
            pend["pg"].append(codes[lo:hi][keep_row])
            del mz_col, it_col, flat_mz, flat_it, row_of_peak, badpk, m, w, r
        del table, batch
        if not pend["pg"]:
            continue
        arrays = {k: np.concatenate(v) for k, v in pend.items()}
        done_rows = last[arrays["pg"]] < hi
        if done_rows.any():
            prec_g = pd.Series(arrays["pr"][done_rows]).groupby(arrays["pg"][done_rows]).median()
            gprec[prec_g.index.to_numpy()] = prec_g.to_numpy()
        done_peaks = last[arrays["g"]] < hi
        if done_peaks.any():
            pieces.append(_nnet_lib_finish(arrays["m"][done_peaks], arrays["w"][done_peaks], arrays["g"][done_peaks],
                                           gprec, n_groups))
        for k, mask in (("m", done_peaks), ("w", done_peaks), ("g", done_peaks), ("pr", done_rows), ("pg", done_rows)):
            pend[k] = [arrays[k][~mask]] if (~mask).any() else []
        del arrays
    if read != len(codes) or any(pend.values()):
        raise ValueError("library batches did not cover every row")
    _nnet_timecheck(deadline)
    if pieces:
        ug = np.concatenate([p[0] for p in pieces])
        ub = np.concatenate([p[1] for p in pieces])
        amp = np.concatenate([p[2] for p in pieces])
    else:
        ug, ub, amp = np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64), np.zeros(0)
    del pieces
    order = np.lexsort((ub, ug))
    ug, ub, amp = ug[order], ub[order], amp[order]
    counts = np.bincount(ug, minlength=n_groups)
    present = np.flatnonzero(counts > 0)
    first_row = pd.Series(np.arange(len(codes))[codes >= 0]).groupby(codes[codes >= 0]).min()
    frow = np.full(n_groups, -1, dtype=np.int64)
    frow[first_row.index.to_numpy()] = first_row.to_numpy()
    smiles_first = smi.take(pa.array(frow[present])).to_pylist()
    library = {}
    structures = {}
    for adduct in _NNET_ADDUCTS:
        mask = np.array([ad[frow[x]] == adduct for x in present], dtype=bool)
        groups = present[mask]
        sizes = counts[groups]
        ptr = np.concatenate([[0], np.cumsum(sizes)])
        take = np.isin(ug, groups)
        library[adduct] = dict(ptr=ptr, bins=ub[take], amp=amp[take], prec=gprec[groups],
                               ik=np.asarray([ik[frow[x]] for x in groups], dtype=object))
    for x, s in zip(present, smiles_first):
        name = ik[frow[x]]
        if name not in structures and isinstance(s, str) and s:
            structures[name] = s
    return library, structures


def _nnet_sparse(rows, cols, vals, shape):
    from scipy import sparse
    return sparse.csr_matrix((vals, (rows, cols)), shape=shape)


def library_raw_edges(spectra, library, deadline=None, tau=0.6, top_m=_NNET_LIB_M, min_matches=6):
    """Exact modified-cosine edges between query spectra and library entries (lossless prefilter, top_m cap)."""
    import numpy as np
    edges = []
    for adduct in _NNET_ADDUCTS:
        lib = library[adduct]
        n = len(lib["prec"])
        queries = sorted(mid for mid in spectra if adduct in spectra[mid])
        if not n or not queries:
            continue
        sizes = np.diff(lib["ptr"])
        lrow = np.repeat(np.arange(n), sizes)
        lfrag = lib["bins"]
        lnl = np.rint((np.repeat(lib["prec"], sizes) - lfrag / 100.0) * 100).astype(np.int64)
        qb = [np.rint(np.asarray(spectra[q][adduct][0]) * 100).astype(np.int64) for q in queries]
        qn = [np.rint((spectra[q][adduct][2] - np.asarray(spectra[q][adduct][0])) * 100).astype(np.int64) for q in queries]
        width = int(max(lfrag.max(initial=0), lnl.max(initial=0), max((x.max(initial=0) for x in qb), default=0),
                        max((x.max(initial=0) for x in qn), default=0))) + 4
        lnl = np.clip(lnl, 0, width - 1)
        LF = _nnet_sparse(lrow, lfrag, lib["amp"], (n, width)).T.tocsr()
        LN = _nnet_sparse(lrow, lnl, lib["amp"], (n, width)).T.tocsr()
        for start in range(0, len(queries), 64):
            _nnet_timecheck(deadline)
            block = queries[start:start + 64]
            rf, cf, vf, rn, cn, vn = [], [], [], [], [], []
            for k, q in enumerate(block):
                a = np.asarray(spectra[q][adduct][1])
                for off in (-1, 0, 1):
                    rf.append(np.full(len(a), k)); cf.append(qb[start + k] + off); vf.append(a)
                for off in (-2, -1, 0, 1, 2):
                    rn.append(np.full(len(a), k)); cn.append(np.clip(qn[start + k] + off, 0, width - 1)); vn.append(a)
            QF = _nnet_sparse(np.concatenate(rf), np.clip(np.concatenate(cf), 0, width - 1), np.concatenate(vf), (len(block), width))
            QN = _nnet_sparse(np.concatenate(rn), np.concatenate(cn), np.concatenate(vn), (len(block), width))
            S = (QF @ LF + QN @ LN).toarray()
            for k, q in enumerate(block):
                row = S[k]
                idx = np.flatnonzero(row >= tau - 1e-9)
                if len(idx) > top_m:
                    idx = idx[np.lexsort((idx, -row[idx]))[:top_m]]
                qspec = spectra[q][adduct]
                for j in sorted(int(x) for x in idx):
                    _nnet_timecheck(deadline)
                    lspec = (lib["bins"][lib["ptr"][j]:lib["ptr"][j + 1]] / 100.0, lib["amp"][lib["ptr"][j]:lib["ptr"][j + 1]], float(lib["prec"][j]))
                    cosine, matches = modified_cosine(qspec, lspec, deadline)
                    if matches >= min_matches and cosine >= tau:
                        edges.append((q, str(lib["ik"][j]), adduct, float(cosine), int(matches), float(qspec[2]), float(lib["prec"][j]),
                                      int(len(idx)), float(row[j])))
    return edges


def select_library_edges(raw, k=_NNET_LIB_K, exclude=frozenset()):
    """Best cosine per (query, structure) over adducts (first sorted adduct on ties), drop excluded, keep top k."""
    best = {}
    for q, ik, adduct, cosine, matches, qprec, lprec, *_ in sorted(raw, key=lambda e: (e[0], e[1], e[2])):
        if ik in exclude:
            continue
        old = best.get((q, ik))
        if old is None or cosine > old[0]:
            best[(q, ik)] = (cosine, abs(qprec - lprec), qprec, lprec)
    per = {}
    for (q, ik), value in best.items():
        per.setdefault(q, []).append((ik,) + value)
    return {q: sorted(v, key=lambda e: (-e[1], e[0]))[:k] for q, v in per.items()}


# ---- generalised network scores ----
def network_scores_net(lists, graph, deadline=None, evidence_lists=None, lib_edges=None, structures=None,
                       deltas=None, mass_on=False):
    import numpy as np
    from rdkit import Chem, DataStructs
    from rdkit.Chem import rdFingerprintGenerator
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fingerprints = {}
    evidence_lists = lists if evidence_lists is None else evidence_lists
    lib_edges = lib_edges or {}

    def fingerprint(smiles):
        if smiles not in fingerprints:
            molecule = Chem.MolFromSmiles(smiles)
            fingerprints[smiles] = generator.GetFingerprint(molecule) if molecule is not None else None
        return fingerprints[smiles]
    result = {}
    for mid, candidates in lists.items():
        _nnet_timecheck(deadline)
        if (not graph.get(mid) and not lib_edges.get(mid)) or len(candidates) < 2:
            continue
        scores = np.zeros(len(candidates), dtype=float)
        terms = []
        for neighbour, cosine in graph.get(mid, []):
            top = evidence_lists.get(neighbour, [])[:5]
            reciprocal = 1.0 / np.arange(1, len(top) + 1)
            if len(top):
                reciprocal /= reciprocal.sum()
            weight = cosine ** 4
            if mass_on:
                delta, pa, pb = deltas[(min(mid, neighbour), max(mid, neighbour))]
                weight = weight * (1.0 if edit_plausible(delta, pa, pb) else _NNET_OFF_EDIT)
            terms.append((weight, [(fingerprint(smiles), w) for smiles, w in zip(top, reciprocal)]))
        for ik, cosine, delta, qprec, lprec in lib_edges.get(mid, []):
            weight = cosine ** 4
            if mass_on:
                weight = weight * (1.0 if edit_plausible(delta, qprec, lprec) else _NNET_OFF_EDIT)
            terms.append((weight, [(fingerprint(structures[ik]), 1.0)]))
        for weight, evidence in terms:
            for i, smiles in enumerate(candidates):
                _nnet_timecheck(deadline)
                candidate = fingerprint(smiles)
                if candidate is not None:
                    scores[i] += weight * sum(w * DataStructs.TanimotoSimilarity(candidate, other)
                                              for other, w in evidence if other is not None)
        result[mid] = scores
    return result


def rerank_net(lists, graph, lam=0.35, steps=1, deadline=None, lib_edges=None, structures=None, deltas=None, mass_on=False):
    kw = dict(lib_edges=lib_edges, structures=structures, deltas=deltas, mass_on=mass_on)
    scores = network_scores_net(lists, graph, deadline, None, **kw)
    output = rerank_lists(lists, scores, lam)
    for _ in range(int(steps) - 1):
        scores = network_scores_net(lists, graph, deadline, output, **kw)
        output = rerank_lists(lists, scores, lam)
    return output


def lock_rank1(base, arm_lists):
    """Keep base[mid][0] at rank 1; the arm order fills ranks 2-25 (its own leader drops to rank 2 when it differs)."""
    output = {}
    for mid, candidates in arm_lists.items():
        candidates = list(candidates)
        if sorted(base[mid]) != sorted(candidates):
            raise ValueError("lock: base and arm lists are not permutations")
        lead = base[mid][0]
        at = candidates.index(lead)
        output[mid] = [lead] + candidates[:at] + candidates[at + 1:]
    return output


def library_stage(spectra, inp, deadline=None):
    from pathlib import Path
    files = sorted(Path(inp).rglob("train.parquet"),
                   key=lambda path: ("mass-spectra" not in str(path).lower(), len(str(path))))
    if not files:
        raise FileNotFoundError("train.parquet absent")
    library, structures = load_library(files[0], deadline)
    _nnet_memcheck("library edges")
    raw = library_raw_edges(spectra, library, deadline)
    lib_edges = select_library_edges(raw)
    return lib_edges, structures, dict(raw_edges=len(raw), library_entries={a: int(len(library[a]["prec"])) for a in library})


def run(work="/kaggle/working", inp="/kaggle/input", pipeline_done=False, enabled=True,
        cap_s=600.0, lam=0.35, arm="base", lock1=True):
    import csv, io, json, os, time, uuid
    from pathlib import Path
    started = time.monotonic()
    audit = dict(version=_NNET_VERSION, arm=arm, lock1=bool(lock1), lambda_value=lam, cap_s=cap_s, status="started")
    target = Path(work) / "submission.csv"
    try:
        lib_on, mass_on, steps = _NNET_ARMS[arm]
        if not enabled or lam == 0:
            audit["status"] = "disabled"
        elif not pipeline_done:
            audit["status"] = "fast commit: original bytes kept"
        else:
            deadline = started + float(cap_s)
            _nnet_timecheck(deadline)
            before = target.read_bytes()
            lists = _nnet_parse_submission(before)
            files = sorted(Path(inp).rglob("test.parquet"),
                           key=lambda path: ("mass-spectra" not in str(path).lower(), len(str(path))))
            if not files:
                raise FileNotFoundError("test.parquet absent")
            import pandas as pd
            test = pd.read_parquet(files[0])
            if not set(lists).issubset(set(test["molecule_id"].astype(str))):
                raise ValueError("spectrum molecule ids missing")
            spectra = build_spectra(test, set(lists), deadline)
            graph = build_graph(spectra, deadline)
            lib_edges, structures = {}, {}
            if lib_on:
                lib_edges, structures, info = library_stage(spectra, inp, deadline)
                audit.update(library=info, with_library_neighbours=len(lib_edges),
                             library_edges=sum(map(len, lib_edges.values())))
            deltas = edge_deltas(spectra, graph, deadline) if mass_on else None
            output = rerank_net(lists, graph, lam, steps, deadline, lib_edges, structures, deltas, mass_on)
            if lock1:
                base_output = rerank_net(lists, graph, lam, 1, deadline)
                audit.update(lock_moved_to_rank2=sum(output[mid][0] != base_output[mid][0] for mid in lists),
                             base_top1_changed=sum(base_output[mid][0] != lists[mid][0] for mid in lists))
                output = lock_rank1(base_output, output)
            changed = sum(output[mid] != lists[mid] for mid in lists)
            audit.update(molecules=len(lists), with_spectra=len(spectra), with_neighbours=len(graph),
                         edges=sum(map(len, graph.values())) // 2, changed_lists=changed,
                         top1_changed=sum(output[mid][0] != lists[mid][0] for mid in lists))
            _nnet_timecheck(deadline)
            if changed:
                stream = io.StringIO(newline="")
                writer = csv.writer(stream, lineterminator="\n")
                writer.writerow(["molecule_id", "smiles"])
                writer.writerows((mid, ";".join(output[mid])) for mid in lists)
                after = stream.getvalue().encode("utf-8")
                verified = _nnet_parse_submission(after)
                if list(verified) != list(lists) or any(sorted(verified[m]) != sorted(lists[m]) for m in lists):
                    raise ValueError("candidate permutation check failed")
                _nnet_timecheck(deadline)
                if target.read_bytes() != before:
                    raise RuntimeError("submission changed concurrently")
                backup = Path(work) / "submission_pre_n035net.csv"
                with backup.open("xb") as handle:
                    handle.write(before)
                temporary = target.with_name("submission.csv.nnet_" + uuid.uuid4().hex)
                with temporary.open("xb") as handle:
                    handle.write(after)
                    handle.flush()
                    os.fsync(handle.fileno())
                _nnet_timecheck(deadline)
                if target.read_bytes() != before:
                    raise RuntimeError("submission changed before commit")
                _nnet_timecheck(deadline)
                os.replace(temporary, target)
            audit["status"] = "applied"
    except Exception as error:
        # No write to submission.csv happens before all checks succeed.
        audit.update(status="failed closed: original bytes kept", exception=repr(error)[:500])
    audit["elapsed_s"] = time.monotonic() - started
    try:
        with (Path(work) / "n035net_audit.json").open("w", encoding="utf-8", newline="\n") as handle:
            json.dump(audit, handle, ensure_ascii=False, indent=2, default=str)
            handle.write("\n")
    except Exception:
        pass
    return audit


# ---- notebook entry: imports expose functions without running this block ----
if globals().get("_NNET_NOTEBOOK", False):
    _nnet_audit = run(work=str(globals().get("ROOT", "/kaggle/working")),
                      inp=str(globals().get("INPUT", "/kaggle/input")),
                      pipeline_done=bool(globals().get("PIPELINE_DONE")),
                      enabled=_NNET_ON, cap_s=_NNET_CAP_S, lam=_NNET_LAM, arm=_NNET_ARM, lock1=_NNET_LOCK1)
    print("[n035net]", _nnet_audit, flush=True)


In [ ]:
vh_snap('b_network')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== 0.500 APEX OVERLAY: DUAL-BAND EXTERNAL SPECTRAL LIBRARY RANK-1 PROMOTION =====
# Integrates external measured MS/MS reference libraries (Takumu Hata extlib CC0 & MassBank harmonized)
# and heliosli's direct-hit Rank-1 promotion rule.
# For any test query matching an external reference spectrum (precursor <= 10 ppm, same ion mode,
# sqrt-intensity cosine >= 0.85, matched peaks >= 4):
# The verified external structure is promoted directly to Rank 1 (Class 1 queries score 1.00 MRR).
# When lib_max >= LIB_TAU (pipeline already has strong match), existing ranking is protected.
# All operations fail safe: any exception preserves the 0.480 Variant M submission.csv untouched.

def _apex_extlib_overlay(work='/kaggle/working', inp='/kaggle/input'):
    import os, sys, glob, time, json, shutil, math
    from pathlib import Path
    import numpy as np
    import pandas as pd

    t0 = time.monotonic()
    sub_path = os.path.join(work, 'submission.csv')
    if not os.path.exists(sub_path):
        print('[apex-extlib] submission.csv not found; overlay skipped')
        return

    sub = pd.read_csv(sub_path, dtype=str, keep_default_na=False)
    shutil.copy(sub_path, os.path.join(work, 'submission_pre_apex_extlib.csv'))

    # Locate test.parquet
    test_paths = sorted(glob.glob(inp + '/**/test.parquet', recursive=True),
                       key=lambda p: ('mass-spectra' not in p.lower(), len(p)))
    if not test_paths:
        print('[apex-extlib] test.parquet not found; overlay skipped')
        return
    te = pd.read_parquet(test_paths[0])

    # Pre-parse query spectra
    queries = {}
    for r in te.itertuples(index=False):
        mid = str(r.molecule_id)
        m = np.asarray(r.ms2_mzs, dtype=np.float64)
        it = np.asarray(r.ms2_normalized_intensities, dtype=np.float64)
        prec = float(r.precursor_mz)
        mode = 1 if 'pos' in str(getattr(r, 'ionization_mode', 'positive')).lower() else -1
        adduct = str(getattr(r, 'adduct', ''))
        
        ok = (m > 0) & (it > 0) & (m <= prec + 0.5)
        m, it = m[ok], it[ok]
        o = np.argsort(m)
        m, it = m[o], it[o]
        
        # Calculate approximate neutral mass
        nm = prec - 1.007276 if mode == 1 else prec + 1.007276
        if '[M+Na]+' in adduct: nm = prec - 22.989218
        elif '[M+K]+' in adduct: nm = prec - 38.963158
        elif '[M+NH4]+' in adduct: nm = prec - 18.033823
        elif '[M+HCOO]-' in adduct: nm = prec - 44.998201
        
        queries.setdefault(mid, []).append(dict(mz=m, it=it, prec=prec, mode=mode, nm=nm))

    # Read existing pipeline library confidence
    lib_max = {}
    ev_path = os.path.join(work, 'frozen_candidate_evidence.parquet')
    if os.path.exists(ev_path):
        try:
            ev = pd.read_parquet(ev_path, columns=['molecule_id', 'library_max'])
            lib_max = {str(m): float(v) for m, v in ev.groupby('molecule_id', sort=False).library_max.first().items()}
        except Exception:
            pass

    # Find external library sources
    refs = []  # list of tuples: (mode, prec, mz_arr, it_arr, key, smiles, neutral_mass)
    
    # Check Source A: Takumu Hata extlib (ext_meta.parquet + ext_mz.npy / ext_in.npy)
    meta_hits = sorted(glob.glob(inp + '/**/ext_meta.parquet', recursive=True), key=len)
    mz_hits = sorted(glob.glob(inp + '/**/ext_mz.npy', recursive=True), key=len)
    in_hits = sorted(glob.glob(inp + '/**/ext_in.npy', recursive=True), key=len)
    mass_hits = sorted(glob.glob(inp + '/**/ext_mass.parquet', recursive=True), key=len)
    
    if meta_hits and mz_hits and in_hits:
        try:
            print(f'[apex-extlib] Loading Takumu Hata extlib from {meta_hits[0]}...')
            meta_df = pd.read_parquet(meta_hits[0])
            mz_all = np.load(mz_hits[0], mmap_mode='r')
            in_all = np.load(in_hits[0], mmap_mode='r')
            mass_df = pd.read_parquet(mass_hits[0]) if mass_hits else None
            
            # Filter non-empty masses if available
            if mass_df is not None and 'exact_mass' in mass_df.columns:
                meta_df['exact_mass'] = mass_df['exact_mass'].values
                meta_df = meta_df[meta_df['exact_mass'].notna() & (meta_df['exact_mass'] > 0)].copy()

            for idx, row in meta_df.iterrows():
                try:
                    mode = 1 if 'pos' in str(row.get('ionization_mode', 'pos')).lower() else -1
                    prec = float(row['precursor_mz'])
                    nm = float(row.get('exact_mass', prec - (1.007276 * mode)))
                    smi = str(row['normalized_smiles'])
                    key = str(row.get('inchikey14', ''))
                    
                    mz_p = np.asarray(mz_all[idx], dtype=np.float64)
                    it_p = np.asarray(in_all[idx], dtype=np.float64)
                    mask = (mz_p > 0) & (it_p > 0)
                    mz_p, it_p = mz_p[mask], it_p[mask]
                    if len(mz_p) < 3:
                        continue
                    o = np.argsort(mz_p)
                    refs.append((mode, prec, mz_p[o], it_p[o], key, smi, nm))
                except Exception:
                    continue
            print(f'[apex-extlib] Loaded {len(refs)} reference spectra from Source A.')
        except Exception as e:
            print(f'[apex-extlib] Source A loading error: {e}')

    # Check Source B: MassBank harmonized spectra.parquet
    mb_hits = sorted(glob.glob(inp + '/**/spectra.parquet', recursive=True),
                     key=lambda p: ('massbank' not in p.lower(), len(p)))
    if mb_hits and 'massbank' in mb_hits[0].lower():
        try:
            print(f'[apex-extlib] Loading MassBank harmonized from {mb_hits[0]}...')
            mb_df = pd.read_parquet(mb_hits[0], columns=['smiles', 'exact_mass', 'ion_mode', 'mzs', 'intensities', 'is_casmi_subset'])
            mb_df = mb_df[(~mb_df.is_casmi_subset.astype(bool)) & mb_df.smiles.notna()]
            
            for row in mb_df.itertuples(index=False):
                try:
                    mode = 1 if 'pos' in str(row.ion_mode).lower() else -1
                    nm = float(row.exact_mass)
                    prec = nm + (1.007276 * mode)
                    smi = str(row.smiles)
                    mz_p = np.asarray(row.mzs, dtype=np.float64)
                    it_p = np.asarray(row.intensities, dtype=np.float64)
                    mask = (mz_p > 0) & (it_p > 0)
                    mz_p, it_p = mz_p[mask], it_p[mask]
                    if len(mz_p) < 3:
                        continue
                    o = np.argsort(mz_p)
                    refs.append((mode, prec, mz_p[o], it_p[o], '', smi, nm))
                except Exception:
                    continue
            print(f'[apex-extlib] Total references after Source B: {len(refs)} spectra.')
        except Exception as e:
            print(f'[apex-extlib] Source B loading error: {e}')

    if not refs:
        print('[apex-extlib] No external reference spectra available; keeping submission untouched.')
        return

    # Sort references by precursor mass for fast binary search
    order = np.argsort([r[1] for r in refs], kind='stable')
    sorted_refs = [refs[i] for i in order]
    prec_array = np.array([r[1] for r in sorted_refs])

    def calc_cosine(qm, qi, rm, ri, tol=0.015):
        if len(qm) == 0 or len(rm) == 0:
            return 0.0, 0
        qa, ra = np.sqrt(qi), np.sqrt(ri)
        qa = qa / (np.linalg.norm(qa) + 1e-12)
        ra = ra / (np.linalg.norm(ra) + 1e-12)
        
        j0 = np.searchsorted(rm, qm - tol, 'left')
        j1 = np.searchsorted(rm, qm + tol, 'right')
        pairs = [(qa[i] * ra[j], i, j) for i in range(len(qm)) for j in range(j0[i], j1[i])]
        if not pairs:
            return 0.0, 0
        pairs.sort(reverse=True)
        ui, uj, s, n = set(), set(), 0.0, 0
        for p, i, j in pairs:
            if i in ui or j in uj:
                continue
            ui.add(i); uj.add(j)
            s += p; n += 1
        return float(s), n

    TAU = 0.80          # Minimum cosine similarity for direct Rank-1 hit
    TIER2_TAU = 0.72    # ours: [TIER2_TAU, TAU) -> rank 2 (rank 1 kept)
    tier2_hits = tier2_moved = mass_rejected = 0
    _ext_keys = {}
    # ours: molecule neutral mass from every spectrum's own adduct (median), and the matched structure's exact mass
    _ADD = {'[M+H]+': -1.007276, '[M+NH4]+': -18.033823, '[M-H2O+H]+': -1.007276 + 18.010565, '[M-2H2O+H]+': -1.007276 + 36.02113,
            '[M+Na]+': -22.989218, '[M+K]+': -38.963158, '[M-H]-': 1.007276, '[M-H2O-H]-': 1.007276 + 18.010565,
            '[M+CH2O2-H]-': -44.998201, '[M+Cl]-': -34.969402}
    _mol_mass = {}
    for _m, _a, _p in zip(te.molecule_id.astype(str), te.adduct.astype(str), te.precursor_mz.astype(float)):
        if _a in _ADD and _p > 0:
            _mol_mass.setdefault(_m, []).append(_p + _ADD[_a])
    _mol_mass = {k: float(np.median(v)) for k, v in _mol_mass.items()}
    _ext_masses = {}

    def _ext_mass_ok(s, mid):
        if mid not in _mol_mass:
            return True                                    # no usable adduct: keep the original behaviour
        if s not in _ext_masses:
            try:
                from rdkit import Chem as _C
                from rdkit.Chem.Descriptors import ExactMolWt as _W
                _m = _C.MolFromSmiles(s)
                _ext_masses[s] = float(_W(_m)) if _m is not None else None
            except Exception:
                _ext_masses[s] = None
        em = _ext_masses[s]
        if em is None:
            return True
        M = _mol_mass[mid]
        return abs(em - M) <= max(0.003, 10e-6 * M)

    def _ext_key(s):
        if s not in _ext_keys:
            try:
                from rdkit import Chem as _C
                _m = _C.MolFromSmiles(s)
                _ext_keys[s] = _C.MolToInchiKey(_m)[:14] if _m is not None else None
            except Exception:
                _ext_keys[s] = None
        return _ext_keys[s]
    K_PEAKS = 4         # Minimum matching peaks
    LIB_TAU = 0.90      # Protect queries already strong in pipeline library

    hits_count = 0
    promoted_count = 0
    new_smiles_list = []

    for mid, smiles_str in zip(sub['molecule_id'].astype(str), sub['smiles'].astype(str)):
        smis = [s for s in smiles_str.split(';') if s]
        new_smis = list(smis)
        
        try:
            # If the pipeline already matched this molecule with high confidence, protect it
            if lib_max.get(mid, 0.0) >= LIB_TAU:
                new_smiles_list.append(smiles_str)
                continue

            q_list = queries.get(mid, [])
            if not q_list:
                new_smiles_list.append(smiles_str)
                continue

            best_match = (0.0, 0, None)  # (cosine, n_peaks, smi)
            for q in q_list:
                q_mode, q_prec = q['mode'], q['prec']
                q_mz, q_it = q['mz'], q['it']
                
                # Precursor search window: 10 ppm + 0.002 Da
                tol = q_prec * 10e-6 + 0.002
                idx_lo = np.searchsorted(prec_array, q_prec - tol, 'left')
                idx_hi = np.searchsorted(prec_array, q_prec + tol, 'right')
                
                for ri in range(idx_lo, idx_hi):
                    ref = sorted_refs[ri]
                    if ref[0] != q_mode:
                        continue
                    cos, n_matched = calc_cosine(q_mz, q_it, ref[2], ref[3])
                    if n_matched >= K_PEAKS and cos > best_match[0]:
                        best_match = (cos, n_matched, ref[5])

            best_cos, best_k, match_smi = best_match
            if match_smi is not None and best_cos >= min(TAU, TIER2_TAU) and not _ext_mass_ok(match_smi, mid):
                mass_rejected += 1                         # ours: the matched structure's exact mass contradicts the molecule
            elif match_smi is not None and best_cos >= min(TAU, TIER2_TAU):
                # ours: structure identity by InChIKey first block (the metric's key), not the SMILES string
                mk = _ext_key(match_smi)
                pos = next((i for i, s in enumerate(smis) if mk is not None and _ext_key(s) == mk), -1)
                if best_cos >= TAU:
                    hits_count += 1
                    if pos >= 0:
                        new_smis = [smis[pos]] + [s for i, s in enumerate(smis) if i != pos]
                        promoted_count += int(pos != 0)
                    else:
                        new_smis = ([match_smi] + smis)[:25]
                        promoted_count += 1
                elif smis:
                    # ours: tier 2 (idea: zz3t3rn41 v9) - a weaker match is placed at rank 2, the pipeline's rank 1 is kept
                    tier2_hits += 1
                    if pos > 1:
                        new_smis = [smis[0], smis[pos]] + [s for i, s in enumerate(smis) if i not in (0, pos)]
                        tier2_moved += 1
                    elif pos < 0:
                        new_smis = ([smis[0], match_smi] + smis[1:])[:25]
                        tier2_moved += 1

        except Exception as exc:
            new_smis = smis

        new_smiles_list.append(';'.join(new_smis) if new_smis else smiles_str)

    res_df = sub.copy()
    res_df['smiles'] = new_smiles_list
    assert len(res_df) == len(sub)
    assert res_df['smiles'].str.split(';').map(len).max() <= 25

    # Atomically save updated submission
    tmp_path = os.path.join(work, 'submission.tmp.csv')
    res_df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, sub_path)

    elapsed = round(time.monotonic() - t0, 1)
    audit = dict(molecules=len(sub), hits=hits_count, promoted=promoted_count, tier2_hits=tier2_hits, tier2_moved=tier2_moved, mass_rejected=mass_rejected, elapsed_s=elapsed)
    with open(os.path.join(work, 'apex_extlib_audit.json'), 'w') as f:
        json.dump(audit, f, indent=2)

    print(f'[apex-extlib] SUCCESS: {hits_count} library matches (TAU={TAU}, K={K_PEAKS}), '
          f'{promoted_count} Rank-1 promotions applied in {elapsed}s. tier2 hits {tier2_hits}, moved {tier2_moved}, mass rejected {mass_rejected}.')

try:
    _apex_extlib_overlay()
except Exception:
    import traceback
    traceback.print_exc()
    print('[apex-extlib] overlay failed; existing submission preserved.')


In [ ]:
vh_snap('c_extlib')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# What each layer did in this run, read from the files the stages wrote.
if not PIPELINE_DONE and not IS_RERUN and COMMIT_MODE == 'fast':
    print('Fast commit: skipping layer audit plot.')
elif SHOW:
    rep = {}
    for name in ('spectral_parallax_report.json', 'provenance_tournament_report.json', 'merge_report.json'):
        p = ROOT / name
        if p.is_file():
            try:
                rep[name] = json.loads(p.read_text())
            except Exception as exc:  # noqa: BLE001 - a report is only shown, never used
                rep[name] = {'unreadable': repr(exc)}
    base = pd.read_csv(BASELINE, dtype=str, keep_default_na=False) if BASELINE.is_file() else None
    final = pd.read_csv(ROOT / 'submission.csv', dtype=str, keep_default_na=False)
    layers = [('0.465 pipeline', BASELINE), ('V1, neural 0.25', VARIANTS_DIR / 'submission_V1.csv'),
              ('V2, ranks 2-25 at 0.50', VARIANTS_DIR / 'submission_V2.csv'),
              (f'{VARIANT}, ranks 2-25 at {CUSTOM["w_mid"]:.2f}', VARIANTS_DIR / f'submission_{VARIANT}.csv')]
    if globals().get('FRAG_ON_V2'):
        layers += [('submitted: CUSTOM + the fragment check\'s leader moves on the V2 order', ROOT / 'submission.csv')]
    elif PROVENANCE:
        layers += [('after the fragment check', ROOT / 'submission_physical0472_control.csv'),
                   ('after the provenance reranker (submitted)', ROOT / 'submission.csv')]
    else:
        layers += [('after the fragment check (submitted)' if PHYSICAL else 'submitted', ROOT / 'submission.csv')]
    prev, out = None, []
    for label, path in layers:
        if not path.is_file():
            continue
        f = pd.read_csv(path, dtype=str, keep_default_na=False).set_index('molecule_id').smiles
        if prev is not None:
            common = f.index.intersection(prev.index)
            moved = int((f[common] != prev[common]).sum())
            top1 = int(sum(a.split(';')[0] != b.split(';')[0] for a, b in zip(f[common], prev[common])))
            out.append({'layer': label, 'lists re-ordered vs previous': moved, 'rank 1 changed': top1})
        else:
            out.append({'layer': label, 'lists re-ordered vs previous': '-', 'rank 1 changed': '-'})
        prev = f
    table_html(pd.DataFrame(out).set_index('layer', drop=False), numeric=('lists re-ordered vs previous', 'rank 1 changed'))
    print('Commit run: the counts cover the 12 visible smoke molecules; the hidden scoring rerun is not shown to anyone.')
    for k, v in rep.items():
        print(k, json.dumps(v)[:600])


In [ ]:
# ===== Arsokan fpn: n5 (NetGen) and p12 (12 parents / 300 products) together in one P5 runner; runner sha256 re-pinned.
# ── H-A-130 s5: P5 tail fix - Engine 1 parents "transformable first", new generated candidates on ranks 3-25 only ──
# Appended after the parent's last cell (FR 3-25 re-ranking) by scripts/build_p5tail_v1.py (cell version 1.0.0).
# The parent pipeline above runs unchanged; its final submission.csv is the parent list, and ranks 1-2 stay byte for byte.
# This cell reruns Engine 1 alone in its own process (scripts/p5tail_runner_v1.py, embedded below: the pinned frozen pipeline
# cells 2-38 without Engine 2 / PubChem / the read-only observer, then the unchanged cell 39) with one change: the analog list
# given to the generate step puts first the analogs whose mass gap to the query has a transform combination (H-A-130 s4 arm
# P5; 6 parents and the cap of 150 unchanged; the analog features are not reordered). Engine 1 rows generated from a parent
# outside the parent run's own six parents ("new") are inserted into the parent's final list at their tail rank = 2 + their
# position among the Engine 1 rows whose metric identity is not one of the final top 2 (the H-A-130 s4 tail-only rule, with
# the Engine 1 position used as the final position), only when that rank is <= 25 and the identity is computable and not yet
# in the list. The parent's guesses keep their order; the list is cut at 25.
# Safety: per-molecule checks (ranks 1-2 = parent bytes, parent order kept, inserted = new P5 rows, no duplicate identity), a
# time budget (the rerun's process group is killed at its deadline; the clock is checked again right before the file replace)
# and a session guard; any exception or failed check keeps the parent's submission.csv byte-identical and prints
# 'P5TAIL FAILED'. Fast commit (no pipeline answer): self-test, then the Engine 1 rerun on the pipeline's own 12-molecule smoke
# subset and a dry-run selection against the placeholder; submission.csv unchanged. Audit /kaggle/working/p5tail_audit.json.
# Names use the prefix _p5_.
import csv as _p5_csv, hashlib as _p5_hashlib, json as _p5_json, os as _p5_os, shutil as _p5_shutil, signal as _p5_signal
import subprocess as _p5_sp, sys as _p5_sys, time as _p5_time, traceback as _p5_tb
from pathlib import Path as _P5_Path

_P5_VERSION = '1.0.0'
_P5_RDKIT = '2026.03.3'         # the metric's pinned RDKit (solution key); any other version keeps the parent file
_P5_BUDGET_S = 5400.0           # whole step in the scoring run (rerun + merge); past it the parent file is kept
_P5_FAST_BUDGET_S = 1800.0      # fast commit: rerun on the smoke subset
_P5_SESSION_S = 8.6 * 3600      # session guard (seconds after NB_T0)
_P5_MERGE_S = 900.0             # kept free after the rerun's deadline for the merge and the write
_P5_CAP = 25
_P5_RUNNER = '"""P5 tail runner (H-A-130 s5), version 1.0.0 - the Engine 1 rerun behind the p5tail cell (scripts/build_p5tail_v1.py embeds\nthis file byte for byte; the cell writes it next to the notebook and runs it in its own process).\nIt executes the pinned frozen 0.465 pipeline (the parent\'s p2s_code/frozen_0465.ipynb, sha256 checked) code cells 2..38 in one\nscope from a fresh working directory - skipping cell 28 (read-only observer), 32 (Engine 2 subprocess) and 34 (PubChem\nsubprocess), whose outputs Engine 1 does not read (the H-A-130 s3/s4 replays) - and then the unchanged Engine 1 cell 39 once,\nwith ONE change: the analog list handed to Engine.generate is reordered \'transformable first\' (H-A-130 s4 arm P5 =\nscripts/h130_s4_e1_container_v1.py 1.1.0 reorder): analogs with sim >= gen_min_sim whose mass gap to the query has a\ncasmi.derive.matching_transforms combination (<= 2 steps, tol = max(gen_tol_da, target * ppm_win * 1e-6)) first, then the other\neligible analogs, then the rest, original order kept inside each group. gen_n_analog 6 and gen_max_total 150 are unchanged\n(asserted, config compared before/after) and the analog list used for features is not reordered (the engine keeps its own list).\nOutput (JSON, temp file + replace): per molecule the P5 Engine 1 BASE list (top 60 after the pool-popularity re-rank) as rows\n[smiles, engine key, generated flag (form \'|gen\'), new flag], the P6 parent set (first 6 eligible analogs in the original order),\nthe P5 parents, and the generated products [key, smiles, parent sid]. \'new\' = a BASE row whose key was generated from a parent\noutside the P6 parent set; the P6 generated list is the prefix of the P5 one (H-A-130 s4 verify: 173/173, 400/400), so these\nare exactly the candidates the parent never generated.\nUsage: python -B p5tail_runner.py <frozen.ipynb> <frozen sha256> <fresh work dir> <out.json>\nEnv (local replays only): P5TAIL_THREADS=n sets os.cpu_count and torch threads like the s3/s4 containers.\n"""\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nVERSION = \'1.0.0\'\nP12_N, P12_TOTAL = 12, 300   # Arsokan p12\nN5_PER_PARENT, N5_MAX = 10, 16   # Arsokan n5: products per network parent / per molecule\nN5_PAR = {}\nif os.environ.get(\'P5N_PARENTS\') and os.path.isfile(os.environ[\'P5N_PARENTS\']):\n    N5_PAR = json.load(open(os.environ[\'P5N_PARENTS\'], encoding=\'utf-8\'))\nSKIP = (28, 32, 34)\nLAST = 39\n\n\ndef p5_order(analogs, min_sim, n_parent, transformable):\n    """\'Transformable first\' order of the analog list (P5). Returns (new list, P6 parents, P5 parents, n transformable)."""\n    el = [a for a in analogs if a[1] >= min_sim]\n    rest = [a for a in analogs if a[1] < min_sim]\n    tr = [bool(transformable(a)) for a in el]\n    new = [a for a, t in zip(el, tr) if t] + [a for a, t in zip(el, tr) if not t] + rest\n    p6 = [int(a[0]) for a in el][:n_parent]\n    p5 = [int(a[0]) for a in new if a[1] >= min_sim][:n_parent]\n    return new, p6, p5, int(sum(tr))\n\n\ndef new_keys(gen, p6):\n    """Keys generated from a parent outside the P6 parent set."""\n    s = set(int(x) for x in p6)\n    return {str(k) for k, _, p in gen if int(p) not in s}\n\n\ndef sha(p):\n    h = hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef main(argv):\n    t_start = time.time()\n    src, expected, wd, out = Path(argv[1]), argv[2], Path(argv[3]), Path(argv[4])\n    assert sha(src) == expected, \'P5TAIL: frozen pipeline source differs\'\n    assert not wd.exists() and not out.exists(), \'P5TAIL: fresh work dir and output required\'\n    threads = os.environ.get(\'P5TAIL_THREADS\')\n    if threads:\n        n_t = int(threads)\n        os.cpu_count = lambda: n_t\n    wd.mkdir(parents=True)\n    os.chdir(wd)\n    import torch\n    if threads:\n        torch.set_num_threads(int(threads))\n    from IPython.display import display\n    scope = {\'__name__\': \'__main__\', \'display\': display}\n    cells = json.loads(src.read_text(encoding=\'utf-8\'))[\'cells\']\n    code39, timings = None, {}\n    for i, c in enumerate(cells):\n        if c[\'cell_type\'] != \'code\' or i > LAST or i in SKIP:\n            continue\n        code = \'\'.join(c[\'source\'])\n        if i == LAST:\n            code39 = code\n            break\n        t = time.time()\n        print(f\'P5TAIL CELL {i} START\', flush=True)\n        exec(compile(code, f\'pipeline_cell_{i}\', \'exec\'), scope)\n        timings[str(i)] = round(time.time() - t, 1)\n        print(f\'P5TAIL CELL {i} DONE {timings[str(i)]}\', flush=True)\n    assert code39 is not None and \'BASE[mid] = (smis, keys, lib_max, scs, forms)\' in code39, \'P5TAIL: Engine 1 cell differs\'\n    import dataclasses\n    E, L, V0 = scope[\'E\'], scope[\'L\'], scope[\'V\']\n    ecfg = E.cfg\n    cfg0 = dataclasses.asdict(ecfg)\n    assert ecfg.generate and (ecfg.gen_n_analog, ecfg.gen_max_total) == (6, 150), (\'P5TAIL: generator setting\', cfg0)\n    from casmi import derive as dmod\n    orig_generate = E.generate\n    G = {}\n\n    def p5_generate(analogs, target, exclude_keys):\n        tol = max(ecfg.gen_tol_da, target * ecfg.ppm_win * 1e-6)\n\n        def transformable(a):\n            return dmod.matching_transforms(target - float(L.struct_mass[a[0]]), tol, 2)\n        new, p6, p5, n_tr = p5_order(analogs, ecfg.gen_min_sim, ecfg.gen_n_analog, transformable)\n        # Arsokan p12: up to P12_N transformable-first parents and P12_TOTAL products for this call only; the engine\n        # config is restored right after, so cfg0 and the P6 parent set (first 6 eligible analogs) are unchanged.\n        p5 = [int(a[0]) for a in new if a[1] >= ecfg.gen_min_sim][:P12_N]\n        _n0, _t0 = ecfg.gen_n_analog, ecfg.gen_max_total\n        ecfg.gen_n_analog, ecfg.gen_max_total = P12_N, P12_TOTAL\n        try:\n            res = orig_generate(new, target, exclude_keys)\n        finally:\n            ecfg.gen_n_analog, ecfg.gen_max_total = _n0, _t0\n        npar = N5_PAR.get(str(scope.get(\'mid\')), []) if N5_PAR else []\n        if npar:\n            res = list(res) + n5_products(npar, target, set(exclude_keys) | {g[\'key\'] for g in res}, tol)\n        G.update(called=True, p6=p6, p5=p5, n_tr=n_tr, gen=[[str(g[\'key\']), str(g[\'smiles\']), int(g[\'parent\'])] for g in res])\n        return res\n    def n5_products(npar, target, seen, tol):\n        """Arsokan n5: derivatives of network neighbours\' top candidates whose mass gap to the query is a casmi.derive\n        transform combination (<= 2 steps), built exactly like Engine.generate\'s rows; parent id -2-j marks them new."""\n        import numpy as _np\n        from casmi import chem as _c, frag as _fm\n        out = []\n        for j, (psmi, psim) in enumerate(npar):\n            try:\n                pp = _c.mol_props(psmi)\n                if pp is None:\n                    continue\n                delta = target - float(pp[1])\n                if abs(delta) < tol or not dmod.matching_transforms(delta, tol, 2):\n                    continue\n                pfp = _c.raw_fingerprint(psmi)\n                if pfp is None:\n                    continue\n                pfp = pfp[E.pool.bits].astype(_np.float32)\n                prods = dmod.derive(psmi, delta, tol=tol, max_steps=2, max_per_analog=N5_PER_PARENT)\n            except Exception:\n                continue\n            for s in prods:\n                try:\n                    std = _c.standardize_smiles(s)\n                    if std is None:\n                        continue\n                    k = _c.score_key(std)\n                    if k is None or k in seen:\n                        continue\n                    p = _c.mol_props(std)\n                    if p is None or abs(p[1] - target) > tol:\n                        continue\n                    fp = _c.raw_fingerprint(std)\n                    if fp is None:\n                        continue\n                    fp = fp[E.pool.bits].astype(_np.float32)\n                    inter = float((fp * pfp).sum()); tan = inter / (fp.sum() + pfp.sum() - inter + 1e-9)\n                    seen.add(k)\n                    out.append(dict(smiles=std, key=k, formula=p[0], mass=p[1], n_heavy=p[2], fp=fp,\n                                    frags=_fm.fragments_for_smiles(std), sim=float(psim), steps=1, tan=tan, parent=-2 - j))\n                except Exception:\n                    continue\n                if len(out) >= N5_MAX:\n                    return out\n        return out\n    E.generate = p5_generate\n    REC = {}\n\n    class _V:\n        """Pass-through around the engine wrapper: same return value, records the generate step of each molecule."""\n        def __init__(self, v):\n            object.__setattr__(self, \'_v\', v)\n\n        def __getattr__(self, n):\n            return getattr(object.__getattribute__(self, \'_v\'), n)\n\n        def run(self, *a, **k):\n            G.clear()\n            try:\n                return object.__getattribute__(self, \'_v\').run(*a, **k)\n            finally:\n                REC[str(scope.get(\'mid\'))] = dict(G)\n    scope[\'V\'] = _V(V0)\n    t = time.time()\n    print(\'P5TAIL CELL 39 START (P5 order)\', flush=True)\n    exec(compile(code39, \'pipeline_cell_39_p5\', \'exec\'), scope)\n    timings[\'39\'] = round(time.time() - t, 1)\n    print(f\'P5TAIL CELL 39 DONE {timings["39"]}\', flush=True)\n    assert dataclasses.asdict(ecfg) == cfg0, \'P5TAIL: engine config changed\'\n    BASE = scope[\'BASE\']\n    mols, n_new_rows, n_changed = {}, 0, 0\n    for mid, v in BASE.items():\n        smis, keys, _, _, forms = v\n        r = REC.get(str(mid), {})\n        nk = new_keys(r.get(\'gen\', []), r.get(\'p6\', []))\n        rows = []\n        for s, k, f in zip(smis, keys, forms):\n            isg = str(f).endswith(\'|gen\')\n            rows.append([str(s), str(k), int(isg), int(str(k) in nk)])\n        n_new_rows += sum(x[3] for x in rows)\n        n_changed += int(r.get(\'p6\', [])[:6] != r.get(\'p5\', [])[:6])\n        mols[str(mid)] = dict(base=rows, p6=r.get(\'p6\', []), p5=r.get(\'p5\', []), n_tr=r.get(\'n_tr\'), gen=r.get(\'gen\', []),\n                              called=bool(r.get(\'called\')))\n    res = dict(version=VERSION, runner_sha256=sha(__file__), frozen_sha256=expected, is_rerun=bool(scope.get(\'IS_RERUN\')),\n               dev=str(scope.get(\'dev\')), n_err=int(scope.get(\'n_err\', -1)), molecules=len(mols), new_rows=n_new_rows,\n               parents_changed=n_changed, timings=timings, wall_s=round(time.time() - t_start, 1),\n               cfg={k: cfg0[k] for k in (\'generate\', \'gen_n_analog\', \'gen_max_total\', \'gen_min_sim\', \'gen_tol_da\', \'ppm_win\')},\n               molecules_data=mols)\n    tmp = Path(str(out) + \'.tmp\')\n    tmp.write_text(json.dumps(res), encoding=\'utf-8\')\n    os.replace(tmp, out)\n    print(\'P5TAIL RUNNER DONE\', json.dumps({k: res[k] for k in (\'molecules\', \'new_rows\', \'parents_changed\', \'n_err\', \'is_rerun\',\n                                                                \'dev\', \'wall_s\')}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main(sys.argv)\n'
_P5_RUNNER_SHA256 = '3ae21c01440793ae1befcd1437b97216be019e1c4ac4835383b9d709b2037704'
_p5_ids = {}                    # stripped guess -> metric identity or None (cache)


def _p5_rdkit():
    import rdkit
    from rdkit import Chem, RDLogger
    from rdkit.Chem.MolStandardize import rdMolStandardize
    RDLogger.DisableLog('rdApp.*')
    if rdkit.__version__ != _P5_RDKIT:
        raise RuntimeError(f'p5tail: RDKit {rdkit.__version__} is not the metric version {_P5_RDKIT}')
    return Chem, rdMolStandardize.TautomerEnumerator(), rdkit.__version__


def _p5_identity(s, chem, taut):
    """Metric identity of one guess: (heavy-atom composition, InChIKey14 after stereo removal and RDKit tautomer
    canonicalisation) - the key-hygiene logic of the dedup cell (D-A-457, D-A-542); None when it cannot be computed."""
    s = s.strip()
    if s in _p5_ids:
        return _p5_ids[s]
    v = None
    try:
        m = chem.MolFromSmiles(s)
        if m is not None:
            comp = {}
            for a in m.GetAtoms():
                if a.GetSymbol() != 'H':
                    comp[a.GetSymbol()] = comp.get(a.GetSymbol(), 0) + 1
            chem.RemoveStereochemistry(m)
            k = chem.MolToInchiKey(taut.Canonicalize(m))[:14]
            if k:
                v = (tuple(sorted(comp.items())), k)
    except Exception:
        v = None
    _p5_ids[s] = v
    return v


def _p5_tail(raw, b5, ident, cap=_P5_CAP):
    """One molecule: parent semicolon list + P5 Engine 1 rows [(smiles, is_new)] in Engine 1 order -> (new list string,
    [(final 1-based position, smiles)] inserted). Tail rank of a row = 2 + its position among the rows whose identity is not
    one of the parent's top 2; a new row is inserted at that rank when it is <= cap, its identity is computable and not in the
    list yet. Parent guesses keep their order, the list is cut at cap, ranks 1-2 never move."""
    toks = raw.split(';')
    if len(toks) < 2 or any(not t.strip() for t in toks):
        return raw, []
    last = max((i for i, (_, n) in enumerate(b5) if n), default=-1)
    if last < 0:                      # no new row: nothing to insert (identities not needed)
        return raw, []
    top = {ident(t) for t in toks[:2]} - {None}
    have = None                       # identities of the parent list, computed at the first possible insertion
    out, ins, j = list(toks), [], 0
    for smi, is_new in b5[:last + 1]:
        k = ident(smi)
        if k is not None and k in top:
            continue
        j += 1
        rank = 2 + j
        if rank > cap:
            break
        if not is_new or k is None:
            continue
        if have is None:
            have = {ident(t) for t in toks} - {None}
        if k in have:
            continue
        pos = min(rank, len(out) + 1)
        out.insert(pos - 1, smi)
        have.add(k)
        ins.append((pos, smi))
    if not ins:
        return raw, []
    return ';'.join(out[:cap]), ins


def _p5_check(old, new, ins, allowed, ident, cap=_P5_CAP):
    """Per-molecule rules: ranks 1-2 = parent bytes; inserted guesses at their positions (>= 3, increasing), each a new P5 row
    with a computable identity found once in the list; removing them gives a prefix of the parent list; length rule."""
    og, ng = old.split(';'), new.split(';')
    assert len(og) >= 2 and ng[:2] == og[:2], 'p5tail: rank 1-2 changed'
    assert 0 < len(ng) <= cap and all(t.strip() for t in ng), 'p5tail: list length or empty guess'
    pos = [p for p, _ in ins]
    assert ins and pos == sorted(set(pos)) and pos[0] >= 3 and pos[-1] <= len(ng), 'p5tail: insertion positions'
    assert all(ng[p - 1] == s and s in allowed for p, s in ins), 'p5tail: inserted guess is not a new P5 row at its position'
    rest = [t for i, t in enumerate(ng, 1) if i not in set(pos)]
    assert rest == og[:len(rest)], 'p5tail: parent guesses reordered'
    assert len(ng) == min(cap, len(og) + len(ins)), 'p5tail: list length differs'
    ks = [ident(t) for t in ng]
    for _, s in ins:
        k = ident(s)
        assert k is not None and ks.count(k) == 1, 'p5tail: inserted identity duplicated'


def _p5_read(path):
    with open(path, newline='', encoding='utf-8') as f:
        rows = list(_p5_csv.DictReader(f))
    assert rows and all(list(r) == ['molecule_id', 'smiles'] and r['smiles'] is not None for r in rows), \
        'p5tail: unexpected submission columns or a malformed row'
    ids = [r['molecule_id'] for r in rows]
    assert len(ids) == len(set(ids)), 'p5tail: duplicate molecule_id'
    assert all(r['smiles'].strip() for r in rows), 'p5tail: empty guess list'
    return ids, {r['molecule_id']: r['smiles'] for r in rows}


def _p5_write(path, ids, lists):
    """Same writer as the FR 3-25 cell (csv, header molecule_id,smiles, LF); returns the temp path."""
    tmp = str(path) + '.p5'
    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        wr = _p5_csv.writer(f, lineterminator='\n')
        wr.writerow(['molecule_id', 'smiles'])
        for m in ids:
            wr.writerow([m, lists[m]])
    return tmp


def _p5_runner_defs():
    """The embedded runner's definitions (its main() only runs as __main__)."""
    ns = {'__name__': 'p5tail_runner_defs'}
    exec(compile(_P5_RUNNER, 'p5tail_runner', 'exec'), ns)
    return ns


def _p5_selftest(ident):
    """Merge, check and P5-order invariants on fixed inputs (identities = official_metric with RDKit 2026.03.3)."""
    assert ident('CCO') == ident('OCC') == ((('C', 2), ('O', 1)), 'LFQSCWFLJHTTHZ'), 'p5tail selftest: identity'
    raw = 'CCO;CCN;CCC;CCCC;CCCCC'
    b5 = [('CCN', False), ('c1ccccc1', True), ('OCC', True), ('CCC', False), ('CC(C)C', True), ('CCCl', False)]
    new, ins = _p5_tail(raw, b5, ident)
    assert (new, ins) == ('CCO;CCN;c1ccccc1;CCC;CC(C)C;CCCC;CCCCC', [(3, 'c1ccccc1'), (5, 'CC(C)C')]), (new, ins)
    _p5_check(raw, new, ins, {'c1ccccc1', 'OCC', 'CC(C)C'}, ident)
    full = ';'.join('C' * i for i in range(1, 26))
    new, ins = _p5_tail(full, [('c1ccccc1', True)], ident)
    assert ins == [(3, 'c1ccccc1')] and new.split(';')[:3] == ['C', 'CC', 'c1ccccc1'] and len(new.split(';')) == 25 \
        and new.split(';')[-1] == 'C' * 24, 'p5tail selftest: cap'
    _p5_check(full, new, ins, {'c1ccccc1'}, ident)
    far = [('C' * i + 'O', False) for i in range(1, 24)] + [('c1ccccc1', True)]
    assert _p5_tail(full, far, ident) == (full, []), 'p5tail selftest: rank beyond 25'
    assert _p5_tail(raw, [('CCCC', True), ('C1CC', True), ('[CH2]', False)], ident) == (raw, []), 'present / no identity'
    assert _p5_tail('CCO', [('c1ccccc1', True)], ident) == ('CCO', []), 'p5tail selftest: short list'
    assert _p5_tail('CCO;CCN', [('c1ccccc1', True)], ident) == ('CCO;CCN;c1ccccc1', [(3, 'c1ccccc1')]), 'short append'
    for o, n, i, a in ((raw, 'CCN;CCO;c1ccccc1;CCC;CCCC', [(3, 'c1ccccc1')], {'c1ccccc1'}),
                       (raw, 'CCO;CCN;c1ccccc1;CCC;CCCC;CCCCC', [(3, 'c1ccccc1')], set()),
                       (raw, 'CCO;CCN;c1ccccc1;CCCC;CCC;CCCCC', [(3, 'c1ccccc1')], {'c1ccccc1'}),
                       (raw, 'CCO;CCN;OCC;CCC;CCCC;CCCCC', [(3, 'OCC')], {'OCC'}),
                       (raw, 'CCO;c1ccccc1;CCN;CCC;CCCC;CCCCC', [(2, 'c1ccccc1')], {'c1ccccc1'})):
        try:
            _p5_check(o, n, i, a, ident)
        except AssertionError:
            continue
        raise AssertionError('p5tail selftest: a broken merge passed the check')
    rd = _p5_runner_defs()
    an = [(1, .9, 0, 0), (2, .8, 0, 0), (3, .7, 0, 0), (4, .6, 0, 0), (5, .5, 0, 0), (6, .4, 0, 0), (7, .35, 0, 0), (8, .32, 0, 0),
          (9, .2, 0, 0)]
    order, p6, p5, n_tr = rd['p5_order'](an, 0.3, 6, lambda a: a[0] in (3, 7, 8, 9))
    assert [a[0] for a in order] == [3, 7, 8, 1, 2, 4, 5, 6, 9] and p6 == [1, 2, 3, 4, 5, 6] and p5 == [3, 7, 8, 1, 2, 4] \
        and n_tr == 3, 'p5tail selftest: P5 order'
    assert rd['new_keys']([['a', 's', 3], ['b', 's', 7], ['c', 's', 8]], p6) == {'b', 'c'}, 'p5tail selftest: new keys'
    return 'passed'


def _p5_rerun(deadline, root):
    """Engine 1 P5 rerun in its own process (group); killed at the deadline. Returns the runner's JSON."""
    root = _P5_Path(root)
    code, out, log = root / 'p5tail_runner.py', root / 'p5tail_e1.json', root / 'p5tail_runner.log'
    work = _P5_Path('/tmp') / f'p5tail_e1_{_p5_os.getpid()}_{int(_p5_time.time())}'
    assert not out.exists() and not work.exists(), 'p5tail: Engine 1 output or work folder exists already'
    code.write_bytes(_P5_RUNNER.encode('utf-8'))
    assert _p5_hashlib.sha256(code.read_bytes()).hexdigest() == _P5_RUNNER_SHA256, 'p5tail: runner source differs'
    left = deadline - _p5_time.monotonic()
    if left < 120:
        raise TimeoutError('p5tail: no time left for the Engine 1 rerun')
    env = {**_p5_os.environ, 'PYTHONHASHSEED': '0', 'PYTHONUNBUFFERED': '1'}
    env.pop('INC_PANEL', None)
    try:
        with open(log, 'w') as fh:
            kw = dict(start_new_session=True) if _p5_os.name == 'posix' else {}
            proc = _p5_sp.Popen([_p5_sys.executable, '-B', str(code), str(FROZEN), FROZEN_SHA256, str(work), str(out)],
                                cwd=str(root), env=env, stdout=fh, stderr=_p5_sp.STDOUT, **kw)
            try:
                rc = proc.wait(timeout=left)
            except _p5_sp.TimeoutExpired:
                try:
                    if hasattr(_p5_os, 'killpg'):
                        _p5_os.killpg(proc.pid, _p5_signal.SIGKILL)
                    else:
                        proc.kill()
                finally:
                    proc.wait()
                raise TimeoutError('p5tail: Engine 1 rerun passed its deadline (process group killed)')
    finally:
        _p5_shutil.rmtree(work, ignore_errors=True)
    if rc != 0:
        raise RuntimeError(f'p5tail: Engine 1 rerun exit code {rc} (log {log})')
    res = _p5_json.loads(out.read_text(encoding='utf-8'))
    assert res.get('runner_sha256') == _P5_RUNNER_SHA256 and res.get('frozen_sha256') == FROZEN_SHA256, \
        'p5tail: runner output provenance differs'
    assert res.get('molecules') == len(res.get('molecules_data') or {}) and \
        (res['cfg'].get('gen_n_analog'), res['cfg'].get('gen_max_total')) == (6, 150), 'p5tail: runner output incomplete'
    return res


def _p5_select(mids, raw, e1, ident, deadline=None):
    """Merge every molecule with its Engine 1 rows (molecules without rows stay as they are); returns (lists, counts)."""
    data = e1['molecules_data']
    out, n = {}, dict(molecules=len(mids), with_rerun=0, new_rows=0, parents_changed=0, molecules_changed=0, inserted=0,
                      inserted_rank3=0, inserted_rank4_10=0, inserted_rank11_25=0, max_inserted=0)
    for m in mids:
        if deadline is not None and _p5_time.monotonic() > deadline:
            raise TimeoutError('p5tail: time budget exceeded in the merge')
        d = data.get(m)
        if d is None:
            out[m] = raw[m]
            continue
        rows = d['base']
        assert isinstance(rows, list) and all(isinstance(r, list) and len(r) == 4 for r in rows), 'p5tail: malformed Engine 1 rows'
        b5 = [(str(r[0]), bool(r[3])) for r in rows]
        allowed = {str(r[0]) for r in rows if r[3]}
        n['with_rerun'] += 1
        n['new_rows'] += len(allowed)
        n['parents_changed'] += int(list(d.get('p6') or [])[:6] != list(d.get('p5') or [])[:6])
        x, ins = _p5_tail(raw[m], b5, ident)
        if ins:
            _p5_check(raw[m], x, ins, allowed, ident)
            n['molecules_changed'] += 1
            n['inserted'] += len(ins)
            n['inserted_rank3'] += sum(1 for p, _ in ins if p == 3)
            n['inserted_rank4_10'] += sum(1 for p, _ in ins if 4 <= p <= 10)
            n['inserted_rank11_25'] += sum(1 for p, _ in ins if p >= 11)
            n['max_inserted'] = max(n['max_inserted'], len(ins))
        else:
            assert x == raw[m], 'p5tail: unchanged list differs'
        out[m] = x
    n['identities'] = len(_p5_ids)
    return out, n


# ===== Arsokan n5 (NetGen): network neighbours' top candidates become extra generation parents in the P5 Engine 1
# rerun. Neighbours = the D-L1 cell's N-1c graph (same adduct, modified cosine >= 0.6, >= 6 matched peaks); parents = each
# neighbour's top 2 guesses, ordered by edge cosine; a parent is used only when the mass gap to the query is a casmi.derive
# transform combination (<= 2 steps). Products enter Engine 1 as generated rows (parent id < 0 = 'new') and reach the list
# only through P5's unchanged tail rule (ranks 3-25 by Engine 1 position). Any failure: no parent file, plain P5.
_N5_TOP, _N5_MAXPAR = 2, 12
def _n5_write_parents(raw, root):
    import os as _o, json as _j
    _o.environ.pop('P5N_PARENTS', None)
    try:
        import pandas as _pd
        comp = globals().get('COMP')
        te = _pd.read_parquet(_o.path.join(str(comp), 'test.parquet'))
        spectra = build_spectra(te, set(raw))
        graph = build_graph(spectra)
        par = {}
        for m, nbrs in graph.items():
            cand = []
            for n, cos in sorted(nbrs, key=lambda x: -x[1]):
                toks = [t for t in str(raw.get(n, '')).split(';') if t][:_N5_TOP]
                cand += [(t, float(cos)) for t in toks]
            seen, keep = set(), []
            for t, c in cand:
                if t not in seen:
                    seen.add(t); keep.append([t, c])
            if keep:
                par[str(m)] = keep[:_N5_MAXPAR]
        path = _o.path.join(str(root), 'p5n_parents.json')
        _j.dump(par, open(path, 'w', encoding='utf-8'))
        _o.environ['P5N_PARENTS'] = path
        print('[n5] network parents for', len(par), 'molecules, edges', sum(len(v) for v in graph.values()) // 2, flush=True)
    except Exception as _e:
        print('[n5] parent file skipped:', repr(_e)[:200], flush=True)

# ---- run (everything above is definitions) ----
_p5_audit = dict(item='H-A-130 s5 P5 tail fix (new generated candidates on ranks 3-25 only)', cell_version=_P5_VERSION,
                 status='not started', budget_s=_P5_BUDGET_S, rdkit=None, runner_sha256=_P5_RUNNER_SHA256)
globals().pop('_p5_parent_bytes', None)
_p5_t0 = _p5_time.monotonic()
_p5_root = _P5_Path(globals().get('ROOT', '/kaggle/working'))
_p5_sub = _p5_root / 'submission.csv'
try:
    _p5_f3 = globals().get('_f3_audit')
    _p5_audit['fr3to25_status'] = _p5_f3.get('status') if isinstance(_p5_f3, dict) else None
    _p5_chem, _p5_taut, _p5_audit['rdkit'] = _p5_rdkit()
    _p5_ident = lambda s: _p5_identity(s, _p5_chem, _p5_taut)
    _p5_audit['selftest'] = _p5_selftest(_p5_ident)
    _p5_mids, _p5_raw = _p5_read(_p5_sub)
    _p5_audit['molecules'] = len(_p5_mids)
    _p5_session_left = None
    if isinstance(globals().get('NB_T0'), (int, float)):
        _p5_session_left = _P5_SESSION_S - (_p5_time.time() - NB_T0)
    if globals().get('PIPELINE_DONE'):
        _p5_parent_bytes = _p5_sub.read_bytes()
        _p5_audit['input_sha256'] = _p5_hashlib.sha256(_p5_parent_bytes).hexdigest()
        _p5_allow = _P5_BUDGET_S if _p5_session_left is None else min(_P5_BUDGET_S, _p5_session_left)
        _p5_audit['seconds_allowed'] = round(_p5_allow, 1)
        if _p5_allow < 600:
            raise TimeoutError('p5tail: session-time guard')
        _p5_deadline = _p5_t0 + _p5_allow
        _n5_write_parents(_p5_raw, _p5_root)
        _p5_e1 = _p5_rerun(_p5_deadline - _P5_MERGE_S, _p5_root)
        _p5_audit['rerun'] = {k: _p5_e1.get(k) for k in ('molecules', 'new_rows', 'parents_changed', 'n_err', 'is_rerun', 'dev',
                                                         'wall_s', 'timings')}
        assert set(_p5_e1['molecules_data']) == set(_p5_mids), 'p5tail: Engine 1 rerun molecules differ from the submission'
        assert bool(_p5_e1.get('is_rerun')) == bool(globals().get('IS_RERUN')), 'p5tail: Engine 1 rerun mode differs'
        _p5_new, _p5_audit['counts'] = _p5_select(_p5_mids, _p5_raw, _p5_e1, _p5_ident, _p5_deadline)
        if _p5_audit['counts']['molecules_changed']:
            _p5_tmp = _p5_write(_p5_sub, _p5_mids, _p5_new)
            _p5_mids2, _p5_back = _p5_read(_p5_tmp)
            assert _p5_mids2 == _p5_mids and _p5_back == _p5_new, 'p5tail: written file differs'
            _p5_tmp_bytes = _P5_Path(_p5_tmp).read_bytes()
            if _p5_time.monotonic() > _p5_deadline:
                raise TimeoutError('p5tail: time budget exceeded before the file replace')
            _p5_os.replace(_p5_tmp, _p5_sub)
            assert _p5_sub.read_bytes() == _p5_tmp_bytes, 'p5tail: replaced file differs'
        _p5_audit.update(status='applied' if _p5_audit['counts']['molecules_changed'] else 'applied: nothing to insert, file untouched',
                         output_sha256=_p5_hashlib.sha256(_p5_sub.read_bytes()).hexdigest())
    elif not globals().get('IS_RERUN'):
        _p5_before = _p5_sub.read_bytes()
        _p5_allow = _P5_FAST_BUDGET_S if _p5_session_left is None else min(_P5_FAST_BUDGET_S, _p5_session_left)
        _p5_audit['seconds_allowed'] = round(_p5_allow, 1)
        if _p5_allow < 600:
            raise TimeoutError('p5tail: session-time guard (fast commit)')
        _p5_deadline = _p5_t0 + _p5_allow
        _n5_write_parents(_p5_raw, _p5_root)
        _p5_e1 = _p5_rerun(_p5_deadline - 60, _p5_root)
        _p5_audit['rerun'] = {k: _p5_e1.get(k) for k in ('molecules', 'new_rows', 'parents_changed', 'n_err', 'is_rerun', 'dev',
                                                         'wall_s', 'timings')}
        _p5_sm = [m for m in _p5_mids if m in _p5_e1['molecules_data']]
        assert _p5_sm and len(_p5_sm) == _p5_e1['molecules'], 'p5tail: smoke molecules not in the placeholder'
        _p5_x, _p5_audit['selection'] = _p5_select(_p5_sm, _p5_raw, _p5_e1, _p5_ident, _p5_deadline)
        assert _p5_sub.read_bytes() == _p5_before, 'p5tail: fast commit changed submission.csv'
        _p5_audit.update(status='fast commit: self-test and selection only, submission.csv unchanged',
                         output_sha256=_p5_hashlib.sha256(_p5_before).hexdigest())
    else:
        _p5_audit['status'] = 'parent file kept: no pipeline answer in a scoring run'
except Exception as _p5_exc:
    _p5_tb.print_exc()
    _p5_audit.update(status='parent file kept after an error', exception=repr(_p5_exc)[:500])
    try:  # the parent's exact bytes if anything was replaced before the failure; never leave the temp file
        if '_p5_parent_bytes' in globals() and _p5_sub.read_bytes() != _p5_parent_bytes:
            _p5_sub.write_bytes(_p5_parent_bytes)
            _p5_audit['restored_parent_bytes'] = True
        if _p5_os.path.exists(str(_p5_sub) + '.p5'):
            _p5_os.remove(str(_p5_sub) + '.p5')
    except Exception as _p5_exc2:
        _p5_audit['restore_error'] = repr(_p5_exc2)[:300]
    print('P5TAIL FAILED', repr(_p5_exc)[:300], flush=True)
_p5_audit['total_s'] = round(_p5_time.monotonic() - _p5_t0, 1)
try:
    (_p5_root / 'p5tail_audit.json').write_text(_p5_json.dumps(_p5_audit, indent=1, default=str), encoding='utf-8')
except Exception:
    pass
try:
    print('[p5tail]', _p5_json.dumps({k: _p5_audit.get(k) for k in ('status', 'rdkit', 'fr3to25_status', 'selftest', 'rerun',
                                                                    'counts', 'selection', 'total_s', 'exception')}, default=str))
except Exception:
    pass


In [ ]:
vh_snap('d_p5')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== Arsokan fp5: this FR 3-25 cell runs AFTER the P5 tail (moved, otherwise unchanged), so the learned fragment
# re-ranker also orders the generated candidates P5 inserted into ranks 3-25. Ranks 1-2 stay locked. Session guard: past
# 8.2 h after NB_T0 the re-rank is skipped (P5's output is kept as is).
import time as _fp5_time
_FP5_SKIP = isinstance(globals().get('NB_T0'), (int, float)) and (_fp5_time.time() - NB_T0) > 8.2 * 3600
if _FP5_SKIP:
    print('[fp5] session guard: FR 3-25 after P5 skipped', flush=True)
# ── H-A-111 s2c follow-up: FR 3-25 rerank, own weights (W = 3.0): learned fragment re-ranker on ranks 3-25 only, ranks 1-2 locked ──
# Appended after the parent's last cell by scripts/build_fr325_clean_v1.py (cell template fr3to25_cell_clean_v1.py 1.0.0).
# Same scoring as scripts/fr2to25_cell_v3.py 3.0.0 (the W 3.0 FR 2-25 kernel casmi26-487-fr2to25own-w300): FRNet fragment
# re-ranker (architecture model.py and features fragfeat.py of the public FR code, unchanged) with weight files trained by
# our own code (fr_train_clean_v1.py, shipped with the weights) on the competition train set (spectra and true structures;
# negatives also from public structure databases); every file is SHA-256 pinned. Features once per list, every model scores the same batch (one model: raw
# logits). The only change from the v3 cell: ranks 1 AND 2 are locked; per list the score is z-scored over the WHOLE list
# (ranks 1..n, as in v3) and ranks 3..n are re-ordered by key = -rank + W * z (stable, unscored -> z 0); no slot rule is
# needed because rank 3 is the best rank a moved candidate can reach. No candidate is added or dropped, and any problem
# keeps the parent's list: per molecule on a scoring error, for the whole file on a load/self-test/check failure. CPU
# fp32. In a fast commit (no pipeline answer) the cell only loads, self-tests and times a short dry run; submission.csv is
# left byte-identical. Names use the prefix _f3_ so the cell never collides with a FR 2-25 cell.
import csv as _f3_csv, glob as _f3_glob, hashlib as _f3_hashlib, importlib.util as _f3_ilu, json as _f3_json
import math as _f3_math, os as _f3_os, shutil as _f3_shutil, sys as _f3_sys, tempfile as _f3_tempfile, time as _f3_time
import traceback as _f3_tb
from collections import Counter as _F3_Counter
from pathlib import Path as _F3_Path
import numpy as _f3_np
import pandas as _f3_pd

_F3_VERSION = '1.0.0'
_F3_W = float('3.0')
_F3_MAX_W = max(1.0, _F3_W)     # W is used as given (H-A-111 s2c measured W above the author's 1.0 clamp locally)
_F3_KEEP = 2                    # parent ranks 1 and 2 never move
_F3_MIN_PEAKS = 3               # merged peaks needed before FR may reorder
_F3_MAX_CAND = 25
_F3_BUDGET_S = 2400.0           # scoring stops after this many seconds; later molecules keep the parent list
_F3_DATASET_SLUG = 'casmi26-fr3to25-clean'
_F3_PIN = {'fragfeat.py': '7b54023d89617fafebbd052e500dba30a7825d8e6893e8bf6567fe11ab7f8ecf',
           'model.py': 'b6276a285903181455df40bfe1215788f31f3ed6466066152efa68d2f21f596d'}
_F3_MODELS = (('fr_model.pt', '4888123625bfff8b56ee631b4048a402c5620ab5e7d403132de39814687443df'),)


def _f3_sha(path):
    h = _f3_hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()


def _f3_find_assets(inp):
    """Folder under /kaggle/input holding the pinned code files and every pinned model file (our private dataset)."""
    inp = _F3_Path(inp)
    first = _F3_MODELS[0][0]
    fixed = [inp / 'datasets/taeyangg4' / _F3_DATASET_SLUG, inp / _F3_DATASET_SLUG]
    cands = [p for p in fixed if (p / first).is_file()]
    cands += sorted(_F3_Path(p).parent for p in _f3_glob.glob(str(inp / '**' / first), recursive=True))
    need = list(_F3_PIN.items()) + [tuple(x) for x in _F3_MODELS]
    for d in cands:
        if all((d / n).is_file() and _f3_sha(d / n) == s for n, s in need):
            return d
    raise FileNotFoundError('fr3to25: no input folder with the pinned fragfeat.py, model.py and model files')


def _f3_load(asset_dir, device='cpu'):
    """Import the pinned modules from a private temp copy (numba cache stays out of /kaggle/working) and load every
    pinned checkpoint with torch.load(weights_only=True)."""
    import torch
    code = _F3_Path(_f3_tempfile.mkdtemp(prefix='fr3to25_code_'))
    mods = {}
    for n in ('fragfeat.py', 'model.py'):
        _f3_shutil.copyfile(_F3_Path(asset_dir) / n, code / n)
        assert _f3_sha(code / n) == _F3_PIN[n], n
        name = 'fr3to25_' + n[:-3]
        spec = _f3_ilu.spec_from_file_location(name, code / n)
        m = _f3_ilu.module_from_spec(spec)
        _f3_sys.modules[name] = m
        spec.loader.exec_module(m)
        mods[n[:-3]] = m
    ff, md = mods['fragfeat'], mods['model']
    if not ff.HAVE_NUMBA:
        raise RuntimeError('fr3to25: numba unavailable (pure-Python fragment enumeration is too slow)')
    nets, info = [], []
    for n, s in _F3_MODELS:
        p = _F3_Path(asset_dir) / n
        assert _f3_sha(p) == s, n
        ck = torch.load(p, map_location='cpu', weights_only=True)
        a = ck.get('args', {})
        net = md.FRNet(d=int(a.get('d', 128)), layers=int(a.get('layers', 5)), no_frag=bool(a.get('no_frag', False)))
        net.load_state_dict(ck['state'])
        net.to(device).eval()
        nets.append(net)
        info.append(dict(file=n, seed=a.get('seed'), epoch=ck.get('ep'), params=int(sum(q.numel() for q in net.parameters()))))
    assert ff.warmup(), 'fr3to25: fragfeat warmup failed'
    return ff, nets, dict(models=info, n_models=len(nets), code_dir=str(code))


def _f3_spectra(test_df):
    """molecule_id -> list of spectrum dicts in fragfeat.merge_spectra's format (base peak = 1 in the competition data)."""
    out = {}
    for r in test_df.itertuples(index=False):
        add = str(r.adduct)
        ion = str(getattr(r, 'ionization_mode', '') or '').lower()
        mode = 1 if ion.startswith('pos') else (-1 if ion.startswith('neg') else (1 if add.endswith('+') else -1))
        out.setdefault(str(r.molecule_id), []).append(dict(
            mz=_f3_np.asarray(r.ms2_mzs, _f3_np.float64), it=_f3_np.asarray(r.ms2_normalized_intensities, _f3_np.float64),
            adduct=add, mode=mode, prec=float(r.precursor_mz)))
    return out


def _f3_batch(feats, device='cpu'):
    """agentili FRNetScorer._build_batch, unchanged in effect (graphs of the featurised candidates packed in order)."""
    import torch
    fs = [f for f in feats if f is not None]
    nn_ = _f3_np.array([f['node_cat'].shape[0] for f in fs], dtype=_f3_np.int32)
    ne = _f3_np.array([f['edge'].shape[0] for f in fs], dtype=_f3_np.int32)
    npk = _f3_np.array([f['peak_x'].shape[0] for f in fs], dtype=_f3_np.int32)
    node_shift = _f3_np.concatenate([[0], _f3_np.cumsum(nn_)[:-1]])
    edge = _f3_np.concatenate([f['edge'] for f in fs], axis=0).astype(_f3_np.int64)
    edge = edge + _f3_np.repeat(node_shift, ne)[:, None]
    P = int(npk.max())
    peak_x = _f3_np.zeros((len(fs), P, 7), dtype=_f3_np.float32)
    peak_mask = _f3_np.zeros((len(fs), P), dtype=bool)
    rr = _f3_np.repeat(_f3_np.arange(len(fs)), npk)
    cc = _f3_np.arange(int(npk.sum())) - _f3_np.repeat(_f3_np.concatenate([[0], _f3_np.cumsum(npk[:-1])]), npk)
    peak_x[rr, cc] = _f3_np.concatenate([f['peak_x'] for f in fs], axis=0)
    peak_mask[rr, cc] = True

    def t(a, dt=torch.float32):
        return torch.as_tensor(_f3_np.ascontiguousarray(a)).to(device, dt)
    return dict(node_cat=t(_f3_np.concatenate([f['node_cat'] for f in fs]).astype(_f3_np.int64), torch.long),
                node_x=t(_f3_np.concatenate([f['node_x'] for f in fs]).astype(_f3_np.float32)),
                edge=t(edge, torch.long),
                edge_cat=t(_f3_np.concatenate([f['edge_cat'] for f in fs]).astype(_f3_np.int64), torch.long),
                edge_x=t(_f3_np.concatenate([f['edge_x'] for f in fs]).astype(_f3_np.float32)),
                peak_x=t(peak_x), peak_mask=t(peak_mask, torch.bool),
                glob_x=t(_f3_np.stack([f['glob_x'] for f in fs]).astype(_f3_np.float32)),
                graph=t(_f3_np.repeat(_f3_np.arange(len(fs)), nn_).astype(_f3_np.int64), torch.long), n_graphs=len(fs))


def _f3_score_list(ff, nets, spectra, smiles_list, device='cpu'):
    """Per-model FR logits aligned with smiles_list (NaN = not scored), features computed once:
    ([scores of model 1, ...], n_peaks, n_feat_errors)."""
    import torch
    out = [[float('nan')] * len(smiles_list) for _ in nets]
    spec = ff.merge_spectra(spectra) if spectra else None
    npk = 0 if spec is None else int(len(spec['mz']))
    if spec is None or npk < _F3_MIN_PEAKS:
        return out, npk, 0
    qtab = ff.query_table(spec)
    feats, errs = [], 0
    for s in smiles_list:
        try:
            feats.append(ff.featurize(spec, qtab, s) if isinstance(s, str) and s else None)
        except Exception:
            feats.append(None)
            errs += 1
    pos = [i for i, f in enumerate(feats) if f is not None]
    if not pos:
        return out, npk, errs
    with torch.no_grad():
        b = _f3_batch(feats, device)
        for k, net in enumerate(nets):
            logits = net(b).float().cpu().numpy()
            for i, v in zip(pos, logits):
                out[k][i] = float(v) if _f3_math.isfinite(float(v)) else float('nan')
    return out, npk, errs


def _f3_z(scores):
    """Within-list z of the finite scores (population std; std < 1e-8 -> 0); NaN stays NaN."""
    sc = _f3_np.array([float(s) for s in scores], dtype=_f3_np.float64)
    ok = ~_f3_np.isnan(sc)
    z = _f3_np.full_like(sc, _f3_np.nan)
    if ok.any():
        v = sc[ok]
        mu, sd = float(_f3_np.mean(v)), float(_f3_np.std(v))
        z[ok] = 0.0 if sd < 1e-8 else (v - mu) / sd
    return z


def _f3_ens(per_model):
    """Ensemble score per candidate: one model -> its raw logits (identical to the single-model cell); several models ->
    mean of the finite per-model within-list z values (NaN if no model scored the candidate)."""
    if len(per_model) == 1:
        return [float(x) for x in per_model[0]]
    zs = _f3_np.vstack([_f3_z(s) for s in per_model])
    ok = ~_f3_np.isnan(zs)
    cnt = ok.sum(0)
    tot = _f3_np.where(ok, zs, 0.0).sum(0)
    return [float(t / c) if c > 0 else float('nan') for t, c in zip(tot, cnt)]


def _f3_blend(lst, scores, w, clamp=True):
    """Ranks 1-2 locked; ranks 3..n re-ordered by key = -rank + W * z, stable (ties keep the parent order). z = within-list
    z of the finite scores over the WHOLE list (ranks 1..n; population std; std < 1e-8 -> 0; unscored -> 0), computed
    exactly as in the v3 cell's _fr_blend. Returns the new order (same items)."""
    w = float(max(0.0, min(_F3_MAX_W, w))) if clamp else float(w)
    if len(lst) <= _F3_KEEP + 1:
        return list(lst)
    sc = _f3_np.array([float(s) for s in scores[:len(lst)]], dtype=_f3_np.float64)
    ok = ~_f3_np.isnan(sc)
    if not ok.any():
        return list(lst)
    z = _f3_np.zeros_like(sc)
    v = sc[ok]
    mu, sd = float(_f3_np.mean(v)), float(_f3_np.std(v))
    z[ok] = 0.0 if sd < 1e-8 else (v - mu) / sd
    tail = [(-float(j + 1) + w * float(z[j]), j) for j in range(_F3_KEEP, len(lst))]
    tail.sort(key=lambda x: x[0], reverse=True)
    return list(lst[:_F3_KEEP]) + [lst[j] for _k, j in tail]


def _f3_same(old, new):
    return (len(new) == len(old) and new[:_F3_KEEP] == old[:_F3_KEEP] and _F3_Counter(new) == _F3_Counter(old)
            and len(new) <= _F3_MAX_CAND)


def _f3_read(path):
    with open(path, newline='', encoding='utf-8') as f:
        rows = list(_f3_csv.DictReader(f))
    assert rows and list(rows[0]) == ['molecule_id', 'smiles'], 'fr3to25: unexpected submission columns'
    ids = [r['molecule_id'] for r in rows]
    assert len(ids) == len(set(ids)), 'fr3to25: duplicate molecule_id'
    return ids, {r['molecule_id']: r['smiles'] for r in rows}


def _f3_write(path, ids, lists):
    tmp = str(path) + '.fr'
    with open(tmp, 'w', newline='', encoding='utf-8') as f:
        wr = _f3_csv.writer(f, lineterminator='\n')
        wr.writerow(['molecule_id', 'smiles'])
        for m in ids:
            wr.writerow([m, lists[m]])
    _f3_os.replace(tmp, path)


def _f3_rerank(ids, raw, spectra, ff, nets, w, budget_s=_F3_BUDGET_S, device='cpu', score_store=None):
    """New semicolon lists for every molecule (parent list where FR did not run); audit counters."""
    t0 = _f3_time.monotonic()
    out, n = {}, _F3_Counter()
    for m in ids:
        lst = raw[m].split(';')
        out[m] = raw[m]
        if _f3_time.monotonic() - t0 > budget_s:
            n['skipped_budget'] += 1
            continue
        if len(lst) <= _F3_KEEP + 1 or not all(lst):
            n['skipped_short_or_empty'] += 1
            continue
        try:
            per, npk, errs = _f3_score_list(ff, nets, spectra.get(m), lst, device)
            n['feature_errors'] += errs
            if score_store is not None:
                score_store[m] = per
            sc = _f3_ens(per)
            if not any(_f3_math.isfinite(x) for x in sc):
                n['skipped_no_scores' if npk >= _F3_MIN_PEAKS else 'skipped_few_peaks'] += 1
                continue
            new = _f3_blend(lst, sc, w)
            if not _f3_same(lst, new):
                n['check_failed'] += 1
                continue
            n['scored'] += 1
            if new != lst:
                n['changed'] += 1
                n['moved_candidates'] += sum(a != b for a, b in zip(lst, new))
                n['rank3_changed'] += int(new[_F3_KEEP] != lst[_F3_KEEP])
                out[m] = ';'.join(new)
        except Exception:
            n['molecule_errors'] += 1
    n['seconds'] = round(_f3_time.monotonic() - t0, 1)
    return out, dict(n)


def _f3_selftest(ff=None, nets=None):
    """Lock and blend invariants on fixed score vectors; optional model smoke on a toy spectrum."""
    nan = float('nan')
    lst = ['L', 'a', 'b', 'c', 'd', 'e']
    assert _f3_blend(lst, [0.0] * 6, 3.0) == lst                           # constant scores: no change
    assert _f3_blend(lst, [nan] * 6, 3.0) == lst                           # no scores: no change
    assert _f3_blend(lst, [0, 9, -9, 1, 2, 3], 0.0) == lst                 # W = 0: no change
    assert _f3_blend(lst, [0, -3.0, 3.0, 0, 0, 0], 3.0)[:2] == ['L', 'a']   # rank 3 outscores rank 2: rank 2 stays
    up = _f3_blend(lst, [9.0, -1, -1, -1, -1, 5.0], 3.0, clamp=False)      # big score at rank 6 -> rank 3
    assert up == ['L', 'a', 'e', 'b', 'c', 'd'], up
    lo = _f3_blend(lst, [9.0, -1, -1, -1, -1, 5.0], 1.0, clamp=False)      # W 1: key -6 + 0.845 is below -4.676
    assert lo == ['L', 'a', 'b', 'c', 'e', 'd'], lo
    st = _f3_blend(['L', 'a', 'b', 'c', 'd'], [0, 5.0, nan, 1.0, 1.0], 3.0) # ties and unscored keep the parent order
    assert st[:2] == ['L', 'a'] and sorted(st) == sorted(['L', 'a', 'b', 'c', 'd']), st
    assert _f3_blend(['L', 'a', 'b'], [0, 1, 9], 3.0) == ['L', 'a', 'b']   # 3-lists cannot change
    big = _f3_blend(lst, [9.0, -1, -1, -1, -1, 5.0], 50.0, clamp=True)     # clamp to _F3_MAX_W
    assert big == _f3_blend(lst, [9.0, -1, -1, -1, -1, 5.0], _F3_MAX_W, clamp=False), big
    assert _f3_same(lst, up) and not _f3_same(lst, ['a', 'L', 'b', 'c', 'd', 'e']) and not _f3_same(lst, ['L', 'b', 'a', 'c', 'd', 'e'])
    v = [0.3, -1.0, 2.0, nan, 0.5, -0.2]
    assert _f3_ens([v])[:3] == v[:3] and _f3_math.isnan(_f3_ens([v])[3])  # one model: raw scores
    res = dict(lock_checks='passed', blend_checks='passed', max_w=_F3_MAX_W, keep=_F3_KEEP)
    if ff is not None and nets is not None:
        spec = [dict(mz=_f3_np.array([77.0386, 105.0335, 121.0284]), it=_f3_np.array([0.4, 1.0, 0.6]), adduct='[M+H]+',
                     mode=1, prec=167.0703)]
        per, npk, errs = _f3_score_list(ff, nets, spec, ['COC(=O)c1ccccc1O', 'CCOC(=O)c1ccccc1', 'CC(=O)Oc1ccccc1'])
        assert npk == 3 and errs == 0 and len(per) == len(nets) and all(_f3_math.isfinite(x) for sc in per for x in sc), (per, npk, errs)
        res['toy_scores'] = [[round(x, 4) for x in sc] for sc in per]
    return res


# ---- run (everything above is definitions) ----
_f3_audit = dict(item='H-A-111 s2c follow-up FR 3-25 rerank (own weights, ranks 1-2 locked)', cell_version=_F3_VERSION, w=_F3_W,
                 keep=_F3_KEEP, n_models=len(_F3_MODELS), status='not started')
globals().pop('_f3_parent_bytes', None)
try:
    _f3_t0 = _f3_time.monotonic()
    _f3_work = _F3_Path(globals().get('ROOT', '/kaggle/working'))
    _f3_inp = _F3_Path(globals().get('INPUT', '/kaggle/input'))
    _f3_dir = _f3_find_assets(_f3_inp)
    _f3_ff, _f3_nets, _f3_audit['model'] = _f3_load(_f3_dir)
    _f3_audit.update(assets=str(_f3_dir), selftest=_f3_selftest(_f3_ff, _f3_nets), load_s=round(_f3_time.monotonic() - _f3_t0, 1))
    _f3_sub = _f3_work / 'submission.csv'
    _f3_ids, _f3_raw = _f3_read(_f3_sub)
    _f3_comp = _F3_Path(globals()['COMP']) if 'COMP' in globals() else next(
        _F3_Path(p).parent for p in _f3_glob.glob(str(_f3_inp / '**/test.parquet'), recursive=True))
    _f3_test = _f3_pd.read_parquet(_f3_comp / 'test.parquet', columns=[
        'molecule_id', 'ms2_mzs', 'ms2_normalized_intensities', 'adduct', 'ionization_mode', 'precursor_mz'])
    _f3_specs = _f3_spectra(_f3_test)
    if globals().get('PIPELINE_DONE') and not _FP5_SKIP:
        _f3_parent_bytes = _f3_sub.read_bytes()
        _f3_audit['input_sha256'] = _f3_hashlib.sha256(_f3_parent_bytes).hexdigest()
        _f3_new, _f3_audit['counts'] = _f3_rerank(_f3_ids, _f3_raw, _f3_specs, _f3_ff, _f3_nets, _F3_W)
        for _f3_m in _f3_ids:
            assert _f3_same(_f3_raw[_f3_m].split(';'), _f3_new[_f3_m].split(';')), _f3_m
        if any(_f3_new[_f3_m] != _f3_raw[_f3_m] for _f3_m in _f3_ids):
            _f3_write(_f3_sub, _f3_ids, _f3_new)
            _f3_ids2, _f3_back = _f3_read(_f3_sub)
            assert _f3_ids2 == _f3_ids and _f3_back == _f3_new, 'fr3to25: written file differs'
        _f3_audit.update(status='applied', output_sha256=_f3_sha(_f3_sub))
    else:
        _f3_dry = _f3_ids[:20]
        _f3_new, _f3_audit['dry_counts'] = _f3_rerank(_f3_dry, _f3_raw, _f3_specs, _f3_ff, _f3_nets, _F3_W, budget_s=300)
        _f3_audit['status'] = 'fast commit: dry run only, submission.csv unchanged'
except Exception as _f3_exc:
    _f3_tb.print_exc()
    _f3_audit.update(status='parent list kept after an error', exception=repr(_f3_exc)[:500])
    try:  # restore the parent's exact bytes if anything was written before the failure
        if '_f3_parent_bytes' in globals() and _f3_sub.read_bytes() != _f3_parent_bytes:
            _f3_sub.write_bytes(_f3_parent_bytes)
            _f3_audit['restored_parent_bytes'] = True
    except Exception as _f3_exc2:
        _f3_audit['restore_error'] = repr(_f3_exc2)[:300]
_f3_audit['total_s'] = round(_f3_time.monotonic() - _f3_t0, 1) if '_f3_t0' in globals() else None
try:
    (_F3_Path(globals().get('ROOT', '/kaggle/working')) / 'fr3to25_audit.json').write_text(
        _f3_json.dumps(_f3_audit, indent=1, default=str), encoding='utf-8')
except Exception:
    pass
print('[fr3to25]', _f3_json.dumps({k: _f3_audit.get(k) for k in ('status', 'w', 'keep', 'n_models', 'counts', 'dry_counts', 'total_s', 'exception')}, default=str))


In [ ]:
vh_snap('e_fr')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== Ours: APRIOR - adduct-pattern prior (which adducts each molecule was observed with) =====
# The test molecules were measured in the same timsTOF campaign as the ~180k "enveda-180" molecules of the competition
# train set. Whether a molecule shows an [M-H]- / formate / [M+Na]+ / [M+H]+ spectrum at all depends on its structure
# (acids ionise in negative mode, basic nitrogens in positive mode, sugars as sodium adducts...). This cell learns
# P(adduct observed | structure) from enveda-180 (gradient boosting on Morgan counts, acid/base SMARTS counts and a few
# descriptors), scores every candidate by the log-likelihood of the adduct pattern the molecule actually shows, and
# re-orders each list by  -GAMMA * ln(rank) + LAM * loglik  (stable). Fails closed: any error keeps the original file.
_AP_ON = True
_AP_LAM = 0.5            # weight of the pattern log-likelihood
_AP_GAMMA = 1.5          # position prior of the incoming list: -GAMMA * ln(rank)
_AP_KEEP1 = True        # True: rank 1 never changes (ranks 2-25 re-ordered only)
_AP_PROTECT = True       # with KEEP1 False: leaders backed by library evidence keep rank 1
_AP_PROTECT_LIB = 0.90   # pipeline library match (frozen_candidate_evidence.library_max) that protects a leader
_AP_N_TRAIN = 60000      # enveda-180 molecules used for training (seeded sample)
_AP_TREES = 300
_AP_CAP_S = 1500.0
_AP_CLIP = 1e-3
_AP_VERSION = "1.0.0"
_AP_TA = ("[M+H]+", "[M-H]-", "[M+CH2O2-H]-", "[M+Na]+", "[M+NH4]+", "[M+K]+", "[M+Cl]-")
_AP_OBS = ("[M+H]+", "[M-H]-", "[M+CH2O2-H]-", "[M+Na]+")
_AP_SMARTS = {
    "cooh": "[CX3](=O)[OX2H1]", "phenol": "c[OX2H1]", "sulfo": "S(=O)(=O)[OH]", "phos": "P(=O)[OH]",
    "amine_basic": "[NX3;H2,H1,H0;!$(NC=O);!$(N-a);!$(N-S(=O)=O);!$(N-C=N)]", "aniline": "[NX3;H2,H1;$(N-a)]",
    "pyridine": "[nX2]", "amide": "C(=O)[NX3]", "imidazole": "c1ncc[nH]1", "guan": "NC(=N)N",
    "ester": "[CX3](=O)[OX2][#6]", "lactone": "[C;R](=O)[O;R]", "glyco": "[OX2H][CX4][CX4][OX2H]", "enol": "[OX2H]C=C",
    "oh_any": "[OX2H]", "nh_any": "[#7;!H0]", "quat": "[N+;!$(N=O);!$(N-[O-])]", "nitro": "[N+](=O)[O-]", "thiol": "[SX2H]",
    "carbonyl": "[CX3]=O", "halogen": "[Cl,Br,I]",
    "acid_any": "[$([CX3](=O)[OX2H1]),$(S(=O)(=O)[OH]),$(P(=O)[OH]),$(c[OX2H1])]",
}


def _ap_featurizer():
    import numpy as np
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, Descriptors, rdMolDescriptors
    RDLogger.DisableLog("rdApp.*")
    pats = [Chem.MolFromSmarts(v) for v in _AP_SMARTS.values()]
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

    def feats(smi):
        try:
            m = Chem.MolFromSmiles(smi)
        except Exception:
            m = None
        if m is None:
            return None
        fp = np.asarray(gen.GetCountFingerprintAsNumPy(m), dtype=np.float32)
        sm = np.array([len(m.GetSubstructMatches(p)) for p in pats], np.float32)
        d = np.array([Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m), rdMolDescriptors.CalcNumHBD(m),
                      rdMolDescriptors.CalcNumHBA(m), rdMolDescriptors.CalcNumAromaticRings(m), m.GetNumHeavyAtoms(),
                      sum(a.GetSymbol() == "N" for a in m.GetAtoms()), sum(a.GetSymbol() == "O" for a in m.GetAtoms()),
                      sum(a.GetSymbol() == "S" for a in m.GetAtoms()), Chem.GetFormalCharge(m)], np.float32)
        return np.concatenate([fp, sm, d])
    return feats


def ap_train(train_path, n_train=_AP_N_TRAIN, trees=_AP_TREES, deadline=None, seed=0):
    """Fit one classifier per observed-adduct label on enveda-180 molecules. Returns (models, info)."""
    import time
    import numpy as np
    import pyarrow.compute as pc
    import pyarrow.parquet as pq
    t0 = time.monotonic()
    tbl = pq.read_table(train_path, columns=["ingest_lib", "inchikey14", "normalized_smiles", "adduct"])
    tbl = tbl.filter(pc.equal(tbl["ingest_lib"], "enveda-180"))
    tbl = tbl.filter(pc.is_in(tbl["adduct"], value_set=__import__("pyarrow").array(list(_AP_TA))))
    df = tbl.select(["inchikey14", "adduct"]).to_pandas().drop_duplicates()
    keys = np.array(sorted(df["inchikey14"].unique()))
    rng = np.random.default_rng(seed)
    keys = keys[rng.permutation(len(keys))[:n_train]]
    chosen = set(keys)
    df = df[df["inchikey14"].isin(chosen)]
    has = {a: set(df.loc[df["adduct"] == a, "inchikey14"]) for a in _AP_OBS}
    sm = tbl.select(["inchikey14", "normalized_smiles"]).to_pandas()
    sm = sm[sm["inchikey14"].isin(chosen)].drop_duplicates("inchikey14")
    smi = dict(zip(sm["inchikey14"], sm["normalized_smiles"]))
    feats = _ap_featurizer()
    X, Y = [], []
    for k in keys:
        if deadline is not None and time.monotonic() > deadline:
            raise TimeoutError("APRIOR featurisation over budget")
        f = feats(smi[k])
        if f is None:
            continue
        X.append(f)
        Y.append([k in has[a] for a in _AP_OBS])
    X, Y = np.stack(X), np.asarray(Y, np.float32)
    models = []
    try:
        import lightgbm as lgb
        for j in range(len(_AP_OBS)):
            models.append(lgb.LGBMClassifier(n_estimators=trees, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, random_state=seed, n_jobs=4,
                                             verbose=-1).fit(X, Y[:, j]))
        engine = "lightgbm"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        for j in range(len(_AP_OBS)):
            models.append(HistGradientBoostingClassifier(max_iter=trees, learning_rate=0.05, max_leaf_nodes=63,
                                                         min_samples_leaf=40, random_state=seed).fit(X, Y[:, j]))
        engine = "sklearn"
    info = dict(train_molecules=int(len(X)), label_rates=[round(float(v), 3) for v in Y.mean(0)], engine=engine,
                train_s=round(time.monotonic() - t0, 1))
    return models, info


def ap_loglik(models, feats, smiles_list, observed):
    """Log-likelihood of the observed pattern (dict adduct -> bool over _AP_OBS) for each candidate SMILES."""
    import numpy as np
    rows, ok = [], []
    for s in smiles_list:
        f = feats(s)
        ok.append(f is not None)
        rows.append(f if f is not None else None)
    if not any(ok):
        return [None] * len(smiles_list)
    dim = len(next(r for r in rows if r is not None))
    X = np.stack([r if r is not None else np.zeros(dim, np.float32) for r in rows])
    P = np.clip(np.stack([m.predict_proba(X)[:, 1] for m in models], 1), _AP_CLIP, 1 - _AP_CLIP)
    y = np.array([float(observed[a]) for a in _AP_OBS], np.float32)
    ll = (y * np.log(P) + (1 - y) * np.log(1 - P)).sum(1)
    return [float(v) if o else None for v, o in zip(ll, ok)]


def ap_rerank(candidates, ll, lam=_AP_LAM, gamma=_AP_GAMMA, keep1=_AP_KEEP1):
    """Stable re-order by -gamma*ln(rank) + lam*loglik. Candidates without a log-likelihood get the list's median."""
    import math
    known = sorted(v for v in ll if v is not None)
    if not known:
        return list(candidates)
    med = known[len(known) // 2]
    score = [-gamma * math.log(i + 1) + lam * (v if v is not None else med) for i, v in enumerate(ll)]
    order = sorted(range(len(candidates)), key=lambda i: (-score[i], i))
    if keep1:
        order = [0] + [i for i in order if i != 0]
    return [candidates[i] for i in order]


def ap_run(work, inp, pipeline_done, enabled=True, cap_s=_AP_CAP_S):
    import csv, io, json, os, time, uuid
    from pathlib import Path
    started = time.monotonic()
    audit = dict(version=_AP_VERSION, status="started", lam=_AP_LAM, gamma=_AP_GAMMA, keep1=_AP_KEEP1, n_train=_AP_N_TRAIN)
    target = Path(work) / "submission.csv"
    try:
        if not enabled:
            audit["status"] = "disabled"
        elif not pipeline_done:
            audit["status"] = "fast commit: original bytes kept"
        else:
            deadline = started + float(cap_s)
            before = target.read_bytes()
            rows = list(csv.reader(io.StringIO(before.decode("utf-8"), newline="")))
            if not rows or rows[0] != ["molecule_id", "smiles"]:
                raise ValueError("invalid submission header")
            lists = {r[0]: r[1].split(";") for r in rows[1:]}

            def locate(name):
                files = sorted(Path(inp).rglob(name), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))
                if not files:
                    raise FileNotFoundError(name + " absent")
                return files[0]
            import pandas as pd
            test = pd.read_parquet(locate("test.parquet"), columns=["molecule_id", "adduct"])
            seen = test.groupby(test["molecule_id"].astype(str))["adduct"].agg(lambda s: set(map(str, s)))
            models, info = ap_train(locate("train.parquet"), deadline=deadline)
            audit.update(info)
            feats = _ap_featurizer()
            cache = {}

            def cfeats(s):
                if s not in cache:
                    cache[s] = feats(s)
                return cache[s]
            protected = set()
            if _AP_PROTECT and not _AP_KEEP1:
                # leaders backed by library evidence keep rank 1: the external-library overlay changed rank 1 (backup
                # written by that cell), or the pipeline's own library match of the molecule is >= _AP_PROTECT_LIB
                pre = Path(work) / "submission_pre_apex_extlib.csv"
                if pre.is_file():
                    prow = list(csv.reader(io.StringIO(pre.read_text(encoding="utf-8"), newline="")))[1:]
                    pre_lead = {r[0]: r[1].split(";")[0] for r in prow if len(r) == 2}
                    apex = {m for m in lists if m in pre_lead and pre_lead[m] != lists[m][0]}
                    protected |= apex
                    audit["protected_apex"] = len(apex)
                else:
                    audit["protected_apex"] = "file absent"
                ev = Path(work) / "frozen_candidate_evidence.parquet"
                if ev.is_file():
                    lib = pd.read_parquet(ev, columns=["molecule_id", "library_max"])
                    lib = lib.groupby(lib["molecule_id"].astype(str))["library_max"].max()
                    strong = {m for m, v in lib.items() if m in lists and float(v) >= _AP_PROTECT_LIB}
                    protected |= strong
                    audit["protected_lib"] = len(strong)
                else:
                    audit["protected_lib"] = "file absent"
                audit["protected"] = len(protected)
            output, moved_top1, scored = {}, 0, 0
            for mid, cands in lists.items():
                if time.monotonic() > deadline:
                    raise TimeoutError("APRIOR time budget exhausted")
                if mid not in seen or not (seen[mid] & set(_AP_TA)):
                    output[mid] = cands
                    continue
                observed = {a: (a in seen[mid]) for a in _AP_OBS}
                ll = ap_loglik(models, cfeats, cands, observed)
                output[mid] = ap_rerank(cands, ll, keep1=_AP_KEEP1 or mid in protected)
                scored += 1
                moved_top1 += output[mid][0] != cands[0]
            changed = sum(output[m] != lists[m] for m in lists)
            audit.update(molecules=len(lists), scored=scored, changed_lists=changed, top1_changed=moved_top1)
            if changed:
                stream = io.StringIO(newline="")
                writer = csv.writer(stream, lineterminator="\n")
                writer.writerow(["molecule_id", "smiles"])
                writer.writerows((m, ";".join(output[m])) for m in lists)
                after = stream.getvalue().encode("utf-8")
                check = {r[0]: r[1].split(";") for r in list(csv.reader(io.StringIO(after.decode("utf-8"), newline="")))[1:]}
                if list(check) != list(lists) or any(sorted(check[m]) != sorted(lists[m]) for m in lists):
                    raise ValueError("candidate permutation check failed")
                if target.read_bytes() != before:
                    raise RuntimeError("submission changed concurrently")
                (Path(work) / "submission_pre_aprior.csv").write_bytes(before)
                tmp = target.with_name("submission.csv.aprior_" + uuid.uuid4().hex)
                with tmp.open("xb") as handle:
                    handle.write(after)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.replace(tmp, target)
            audit["status"] = "applied"
    except Exception as error:
        audit.update(status="failed closed: original bytes kept", exception=repr(error)[:500])
    audit["elapsed_s"] = round(time.monotonic() - started, 1)
    try:
        (Path(work) / "aprior_audit.json").write_text(json.dumps(audit, indent=2, default=str) + "\n")
    except Exception:
        pass
    return audit


if "ROOT" in globals():
    import time as _ap_t, json as _ap_json
    _ap_left = 8.7 * 3600 - (_ap_t.time() - float(globals().get("NB_T0", _ap_t.time())))
    if bool(globals().get("PIPELINE_DONE")):
        if _ap_left < 600:
            print("[aprior] skipped: session time guard, %.0f s left" % _ap_left, flush=True)
        else:
            _ap_audit = ap_run(str(ROOT), str(INPUT), True, enabled=_AP_ON, cap_s=min(_AP_CAP_S, _ap_left - 300))
            print("[aprior]", _ap_json.dumps(_ap_audit), flush=True)
    else:
        # commit run: small training + a dry run on the visible test file (submission.csv is not touched)
        try:
            _ap_t0 = _ap_t.monotonic()
            assert ap_rerank(["A", "B", "C"], [-3.0, -0.1, -0.2], lam=1.0, gamma=1.5, keep1=False) == ["B", "C", "A"]
            assert ap_rerank(["A", "B", "C"], [-3.0, -0.1, -0.2], lam=1.0, gamma=1.5, keep1=True) == ["A", "B", "C"]
            assert ap_rerank(["A", "B", "C"], [-0.5, -0.1, -0.2], lam=0.5, gamma=1.5, keep1=False) == ["A", "B", "C"]
            assert ap_rerank(["A", "B"], [None, None]) == ["A", "B"]
            _ap_train = sorted(INPUT.rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            _ap_models, _ap_info = ap_train(_ap_train, n_train=3000, trees=50)
            import pandas as _ap_pd
            _ap_test = _ap_pd.read_parquet(COMP / "test.parquet", columns=["molecule_id", "adduct"])
            _ap_seen = _ap_test.groupby(_ap_test["molecule_id"].astype(str))["adduct"].agg(lambda s: set(map(str, s)))
            _ap_f = _ap_featurizer()
            _ap_ll = [ap_loglik(_ap_models, _ap_f, ["CC(=O)O", "CCOC(C)=O", "OCC(O)CO", "c1ccccc1O"],
                                {a: (a in _ap_seen[m]) for a in _AP_OBS}) for m in list(_ap_seen.index)[:20]]
            assert all(len(v) == 4 and all(x is not None for x in v) for v in _ap_ll), _ap_ll[:2]
            print("[aprior] commit dry run ok:", _ap_json.dumps(_ap_info), "| %d molecules scored | %.1f s"
                  % (len(_ap_ll), _ap_t.monotonic() - _ap_t0), flush=True)
        except Exception as _ap_exc:
            print("[aprior] commit dry run FAILED:", repr(_ap_exc)[:400], flush=True)


In [ ]:
vh_snap('f_aprior')  # VH: copy of submission.csv at this stage boundary


In [ ]:
_UNIQ_NOTEBOOK = True
"""UNIQ: one structure, one molecule (test-set uniqueness overlay).

Idea. Every molecule_id of the test set is a different structure (one InChIKey first block per molecule, as in the
training libraries built by the same hosts). The pipeline ranks each molecule on its own, and the neighbour network even
pulls similar molecules towards the same answer, so the same structure X can end up as rank 1 of several test molecules.
At most one of them can be right.

Rule. For a structure X that is rank 1 of two or more molecules ("claimants") and has reference spectra in train.parquet:
  e(m, X) = best cosine between a spectrum of molecule m and a reference spectrum of X with the same adduct
            (0.01 Da bins, +-1 bin, square-root intensities, one-to-one peak matching, peaks below precursor - 1.5).
  The claimant with the highest e keeps X (the "owner") when e_owner >= TAU_OWN.
  Every other claimant m with e_owner - e(m, X) >= DELTA moves X from rank 1 to rank 2; its old rank 2 becomes rank 1
  (X stays at rank 2 there as a hedge).
Fingerprint fallback (v1.1): a group whose structure has no usable reference spectra is decided by the public FPNet instead.
  g(m, X) = z(f.z of X) - max z(f.z of the molecule's other candidates), over its +-10 ppm pool window and its own list.
  The claimant with the largest g keeps X; a claimant at least DELTA_FP below moves X to rank 2 (validated on 307 duplicate
  pairs of held-out isomers with the fold-safe FPNet: owner right 80%, mean gain +0.22 reciprocal rank per pair).
Optional exclusion below rank 1 (SINK): a structure whose owner has e_owner >= TAU_SINK is moved to the end of every other
molecule's list where it sits at rank 2-25 and e_owner - e(m, X) >= DELTA_SINK; the candidates below it move up one place.

Nothing is added or dropped: every list keeps exactly its candidates. Any error, time-out or failed check keeps the input
bytes. Inputs: submission.csv, test.parquet, train.parquet. Outputs: submission.csv (atomic), uniq_audit.json.
"""
_UNIQ_ON = True
_UNIQ_TAU_OWN = 0.60       # the owner's own match must reach this
_UNIQ_DELTA = 0.05         # owner's lead over a claimant that has to give X up
_UNIQ_SINK = True          # exclusion below rank 1
_UNIQ_TAU_SINK = 0.85      # owner match needed before X is excluded from other lists
_UNIQ_DELTA_SINK = 0.10
_UNIQ_MIN_MATCH = 4        # matched peaks needed for a cosine to count
_UNIQ_MAX_REFS = 48        # reference spectra kept per (structure, adduct)
_UNIQ_ROUNDS = 3           # a swap can create a new duplicate: repeat
_UNIQ_CAP_S = 1800.0
_UNIQ_FP = True            # fingerprint fallback for duplicate groups whose structure has no usable reference spectra
_UNIQ_DELTA_FP = 0.0  # FPNet gap lead (z units) the owner needs over a claimant that gives the structure up
_UNIQ_FP_TOP = 3           # candidates per involved molecule that get fingerprint evidence
_UNIQ_FP_CAP_S = 1200.0
_UNIQ_FP_RUNNER = '"""UNIQ fingerprint evidence: for each requested (molecule, candidate SMILES), how far the candidate leads the molecule\'s other\ncandidates under the public FPNet score f.z.\n\nFor a molecule (neutral mass M = median over its spectra, logits z = the PubChem channel\'s molecule_logits) the reference set is\nits pool window (+-10 ppm) plus its own answer list. Scores are z-scored over that set; the evidence of candidate X is\ngap(X) = z(X) - max z(c) over the candidates c whose metric key differs from X\'s (negative when X is not the FPNet favourite).\n\nusage: python uniq_fp_runner.py <casmi_code_dir> <fpnet_models_dir or -> <fold0.pt or -> <pool_dir> <test.parquet>\n                                <request.json: {molecule_id: [smiles, ...]}> <lists.json: {molecule_id: [smiles, ...]}> <out.json>\n"""\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\n\nPPM = 10.0\n\n\ndef molecule_logits(bank, spectra):\n    """The PubChem channel\'s probe_core2.molecule_logits (public pipeline), unchanged."""\n    from casmi import chem, fpnet\n    from casmi.spectra import merge_spectra\n    items_s = []\n    for s in spectra:\n        pm, pi = fpnet.prep_peaks(s[\'mz\'], s[\'it\'], s[\'prec\'])\n        ce_ok = s[\'ce\'] is not None and not np.isnan(s[\'ce\'])\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\'prec\']), adduct_ix=chem.adduct_index(s[\'adduct\']),\n                            ce=float(s[\'ce\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\n                            n_merged=min(int(s.get(\'ce_n\', 1)), 8), mode=float(s[\'mode\'])))\n    items_m = []\n    for md in (1, -1):\n        grp = [s for s in spectra if s[\'mode\'] == md]\n        if not grp:\n            continue\n        mm, ii = merge_spectra([(s[\'mz\'], s[\'it\']) for s in grp])\n        prec = float(np.median([s[\'prec\'] for s in grp]))\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\n        ces = [s[\'ce\'] for s in grp if s[\'ce\'] is not None and not np.isnan(s[\'ce\'])]\n        adducts = [s[\'adduct\'] for s in grp]\n        ad = max(adducts, key=adducts.count)\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                            n_merged=min(sum(max(1, int(s.get(\'ce_n\', 1))) for s in grp), 8), mode=float(md)))\n    if not items_s:\n        return None\n    zs, _ = bank.logits_el(items_s)\n    zm, _ = bank.logits_el(items_m)\n    return (1.5 * 0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\n\n\ndef main():\n    code_dir, models_dir, fold0, pool_dir, test_path, req_path, lists_path, out_path = sys.argv[1:9]\n    sys.path.insert(0, code_dir)\n    import glob\n    import pandas as pd\n    import torch\n    from casmi import chem, fpnet\n    t0 = time.time()\n    paths = (sorted(glob.glob(os.path.join(models_dir, \'fpnet_*.pt\'))) if models_dir not in (\'\', \'-\') else []) + \\\n            ([fold0] if fold0 not in (\'\', \'-\') else [])\n    assert paths, \'no FPNet checkpoint\'\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    bank = fpnet.ModelBank(paths, device=dev)\n    bits = np.load(os.path.join(pool_dir, \'fp_bits.npy\'))\n    assert bank.nbits == len(bits), (bank.nbits, len(bits))\n    meta = pd.read_parquet(os.path.join(pool_dir, \'pool_meta.parquet\'), columns=[\'key\', \'mass\'])\n    pmass = meta[\'mass\'].to_numpy(dtype=np.float64)\n    pkey = meta[\'key\'].to_numpy()\n    pfp = np.load(os.path.join(pool_dir, \'pool_fp.npy\'), mmap_mode=\'r\')\n    req = json.load(open(req_path))\n    lists = json.load(open(lists_path))\n    te = pd.read_parquet(test_path)\n    te = te[te.molecule_id.astype(str).isin(set(req))]\n    te[\'nm\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\n    fpcache, keycache = {}, {}\n\n    def fp_of(s):\n        if s not in fpcache:\n            f = chem.raw_fingerprint(s)\n            fpcache[s] = None if f is None else f[bits].astype(np.float32)\n        return fpcache[s]\n\n    def key_of(s):\n        if s not in keycache:\n            try:\n                keycache[s] = chem.score_key(s)\n            except Exception:\n                keycache[s] = None\n        return keycache[s]\n\n    out, diag = {}, dict(molecules=0, no_logits=0, no_window=0)\n    for mid, sub in te.groupby(te.molecule_id.astype(str), sort=False):\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\n        if not len(nms):\n            continue\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == \'positive\' else -1, adduct=r.adduct, prec=float(r.precursor_mz),\n                                ce=float(ce_arr.mean()) if len(ce_arr) else np.nan, ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        try:\n            z = molecule_logits(bank, spectra[:16])\n        except Exception:\n            z = None\n        if z is None:\n            diag[\'no_logits\'] += 1\n            continue\n        z = z.astype(np.float32)\n        M = float(np.median(nms))\n        tol = M * PPM * 1e-6\n        a, b = np.searchsorted(pmass, M - tol, \'left\'), np.searchsorted(pmass, M + tol, \'right\')\n        scores, skeys = [], []\n        if b > a:\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :len(bits)].astype(np.float32)\n            scores = list(F @ z)\n            skeys = list(pkey[a:b])\n        else:\n            diag[\'no_window\'] += 1\n        for s in lists.get(mid, []) + req[mid]:\n            f, k = fp_of(s), key_of(s)\n            if f is not None and k is not None:\n                scores.append(float(f @ z))\n                skeys.append(k)\n        if len(scores) < 2:\n            continue\n        sc = np.asarray(scores, dtype=np.float64)\n        zz = (sc - sc.mean()) / (sc.std() + 1e-9)\n        best = {}\n        for k, v in zip(skeys, zz):\n            if k not in best or v > best[k]:\n                best[k] = float(v)\n        res = {}\n        for s in req[mid]:\n            k = key_of(s)\n            if k is None or k not in best:\n                continue\n            others = [v for kk, v in best.items() if kk != k]\n            if others:\n                res[s] = best[k] - max(others)\n        out[mid] = res\n        diag[\'molecules\'] += 1\n    diag[\'seconds\'] = round(time.time() - t0, 1)\n    diag[\'models\'] = len(paths)\n    tmp = out_path + \'.tmp\'\n    json.dump(dict(evidence=out, diag=diag), open(tmp, \'w\'))\n    os.replace(tmp, out_path)\n    print(\'UNIQ_FP done\', json.dumps(diag), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'  # source of uniq_fp_runner.py (embedded by the notebook build)
_UNIQ_VERSION = "1.1.0"


def _uniq_vec(mz, it, prec):
    import numpy as np
    mz = np.asarray(mz, dtype=float)
    it = np.asarray(it, dtype=float)
    if mz.ndim != 1 or it.ndim != 1 or len(mz) != len(it) or len(mz) == 0:
        return None
    keep = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz < prec - 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) < 3:
        return None
    it = it / it.max()
    bins = np.rint(mz * 100).astype(np.int64)
    ub, inverse = np.unique(bins, return_inverse=True)
    top = np.zeros(len(ub))
    np.maximum.at(top, inverse, it)
    order = np.argsort(-top, kind="stable")[:120]
    ub, amp = ub[order], np.sqrt(top[order])
    order = np.argsort(ub, kind="stable")
    ub, amp = ub[order], amp[order]
    return ub, amp / max(float(np.linalg.norm(amp)), 1e-12)


def _uniq_cos(a, b):
    """Exact greedy cosine of two binned spectra: +-1 bin, each peak used once. Returns (cosine, matched peaks)."""
    import numpy as np
    ab, aa = a
    bb, ba = b
    lo = np.searchsorted(bb, ab - 1, side="left")
    hi = np.searchsorted(bb, ab + 1, side="right")
    pairs = [(aa[i] * ba[j], i, j) for i in range(len(ab)) for j in range(lo[i], hi[i])]
    pairs.sort(key=lambda t: (-t[0], t[1], t[2]))
    used_a, used_b = set(), set()
    total, matches = 0.0, 0
    for weight, i, j in pairs:
        if i not in used_a and j not in used_b:
            used_a.add(i)
            used_b.add(j)
            total += weight
            matches += 1
    return float(total), matches


def _uniq_parse(raw):
    import csv, io
    rows = list(csv.reader(io.StringIO(raw.decode("utf-8"), newline="")))
    if not rows or rows[0] != ["molecule_id", "smiles"] or len(rows) < 2:
        raise ValueError("invalid submission header or no rows")
    lists = {}
    for row in rows[1:]:
        if len(row) != 2 or not row[0] or row[0] in lists:
            raise ValueError("invalid or repeated molecule id")
        candidates = row[1].split(";")
        if not 1 <= len(candidates) <= 25 or any(not x.strip() for x in candidates):
            raise ValueError("invalid candidate list")
        lists[row[0]] = candidates
    return lists


def _uniq_keys(lists):
    """InChIKey first block of every candidate (None when RDKit cannot read the SMILES)."""
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog("rdApp.*")
    cache = {}

    def key(smiles):
        if smiles not in cache:
            try:
                mol = Chem.MolFromSmiles(smiles)
                cache[smiles] = Chem.MolToInchiKey(mol)[:14] if mol is not None else None
            except Exception:
                cache[smiles] = None
            if not cache[smiles]:
                cache[smiles] = None
        return cache[smiles]
    return {mid: [key(s) for s in candidates] for mid, candidates in lists.items()}


def _uniq_queries(test, wanted):
    import numpy as np
    spectra = {}
    ids = test["molecule_id"].astype(str).to_numpy()
    for mid, adduct, prec, mz, it in zip(ids, test["adduct"].astype(str), test["precursor_mz"], test["ms2_mzs"],
                                         test["ms2_normalized_intensities"]):
        if mid not in wanted:
            continue
        try:
            prec = float(prec)
        except Exception:
            continue
        if not np.isfinite(prec):
            continue
        v = _uniq_vec(mz, it, prec)
        if v is not None:
            spectra.setdefault(mid, {}).setdefault(adduct, []).append(v)
    return spectra


def _uniq_references(train_path, keys, adducts, max_refs, deadline):
    """Reference spectra of the wanted structures: {inchikey14: {adduct: [vector, ...]}} (first max_refs rows each)."""
    import time
    import numpy as np
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq
    refs = {}
    if not keys:
        return refs
    key_set = pa.array(sorted(keys))
    adduct_set = pa.array(sorted(adducts))
    source = pq.ParquetFile(train_path)
    for batch in source.iter_batches(batch_size=65536, use_threads=False,
                                     columns=["inchikey14", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"]):
        if time.monotonic() >= deadline:
            raise TimeoutError("UNIQ time budget exhausted while reading train.parquet")
        table = pa.Table.from_batches([batch])
        mask = pc.and_(pc.is_in(table.column("inchikey14"), value_set=key_set),
                       pc.is_in(table.column("adduct"), value_set=adduct_set))
        table = table.filter(pc.fill_null(mask, False))
        if table.num_rows == 0:
            continue
        for ik, adduct, prec, mz, it in zip(table.column("inchikey14").to_pylist(), table.column("adduct").to_pylist(),
                                            table.column("precursor_mz").to_pylist(), table.column("ms2_mzs").to_pylist(),
                                            table.column("ms2_normalized_intensities").to_pylist()):
            if prec is None or mz is None or it is None or not np.isfinite(prec):
                continue
            bucket = refs.setdefault(ik, {}).setdefault(adduct, [])
            if len(bucket) >= max_refs:
                continue
            v = _uniq_vec(mz, it, float(prec))
            if v is not None:
                bucket.append(v)
    return refs


def _uniq_evidence(query, reference, min_match, deadline):
    """Best same-adduct cosine between a molecule's spectra and a structure's references (0.0 when nothing compares)."""
    import time
    best = 0.0
    for adduct, qs in query.items():
        for r in reference.get(adduct, ()):
            if time.monotonic() >= deadline:
                raise TimeoutError("UNIQ time budget exhausted while scoring")
            for q in qs:
                cosine, matches = _uniq_cos(q, r)
                if matches >= min_match and cosine > best:
                    best = cosine
    return best


def uniq_resolve(lists, keys, evidence, tau_own=_UNIQ_TAU_OWN, delta=_UNIQ_DELTA, sink=_UNIQ_SINK,
                 tau_sink=_UNIQ_TAU_SINK, delta_sink=_UNIQ_DELTA_SINK, rounds=_UNIQ_ROUNDS, fp_evidence=None,
                 delta_fp=_UNIQ_DELTA_FP):
    """Pure ranking step. lists/keys: {mid: [...]} aligned; evidence(mid, key) -> float or None (no references).
    Returns (new lists, audit dict)."""
    out = {mid: list(c) for mid, c in lists.items()}
    kk = {mid: list(k) for mid, k in keys.items()}
    audit = dict(duplicate_structures=0, duplicate_claimants=0, with_references=0, decided=0, demoted=0, sunk=0,
                 rounds=0, demoted_ids=[], sunk_ids=[], decided_fp=0, demoted_fp=0, demoted_fp_ids=[], no_evidence=0)
    hedged = set()   # (molecule, structure) moved from rank 1 to rank 2: kept there, never sunk
    for round_no in range(int(rounds)):
        claims = {}
        for mid in out:
            if kk[mid] and kk[mid][0] is not None:
                claims.setdefault(kk[mid][0], []).append(mid)
        moved = 0
        for key in sorted(k for k, v in claims.items() if len(v) > 1):
            claimants = sorted(claims[key])
            if round_no == 0:
                audit["duplicate_structures"] += 1
                audit["duplicate_claimants"] += len(claimants)
            scores = {mid: evidence(mid, key) for mid in claimants}
            lib_ok = not any(v is None for v in scores.values())
            if lib_ok and round_no == 0:
                audit["with_references"] += 1
            if lib_ok:
                owner = max(claimants, key=lambda mid: (scores[mid], mid))
                lib_ok = scores[owner] >= tau_own
            if not lib_ok:
                fps = {mid: (fp_evidence(mid, key) if fp_evidence is not None else None) for mid in claimants}
                if any(v is None for v in fps.values()):
                    audit["no_evidence"] += int(round_no == 0)
                    continue
                owner = max(claimants, key=lambda mid: (fps[mid], mid))
                audit["decided_fp"] += 1
                for mid in claimants:
                    if mid == owner or fps[owner] - fps[mid] < delta_fp or len(out[mid]) < 2:
                        continue
                    out[mid][0], out[mid][1] = out[mid][1], out[mid][0]
                    kk[mid][0], kk[mid][1] = kk[mid][1], kk[mid][0]
                    audit["demoted_fp"] += 1
                    audit["demoted_fp_ids"].append(mid)
                    hedged.add((mid, key))
                    moved += 1
                continue
            audit["decided"] += 1
            for mid in claimants:
                if mid == owner or scores[owner] - scores[mid] < delta or len(out[mid]) < 2:
                    continue
                out[mid][0], out[mid][1] = out[mid][1], out[mid][0]
                kk[mid][0], kk[mid][1] = kk[mid][1], kk[mid][0]
                audit["demoted"] += 1
                audit["demoted_ids"].append(mid)
                hedged.add((mid, key))
                moved += 1
        audit["rounds"] = round_no + 1
        if not moved:
            break
    if sink:
        owners = {}
        for mid in out:
            key = kk[mid][0] if kk[mid] else None
            if key is not None:
                owners.setdefault(key, []).append(mid)
        strong = {}
        for key, mids in owners.items():
            if len(mids) != 1:
                continue
            value = evidence(mids[0], key)
            if value is not None and value >= tau_sink:
                strong[key] = (mids[0], value)
        for mid in out:
            if len(out[mid]) < 3:
                continue
            drop = []
            for position in range(1, len(out[mid])):
                key = kk[mid][position]
                if key in strong and strong[key][0] != mid and (mid, key) not in hedged:
                    mine = evidence(mid, key)
                    if mine is not None and strong[key][1] - mine >= delta_sink:
                        drop.append(position)
            if drop:
                keep = [i for i in range(len(out[mid])) if i not in set(drop)]
                order = keep + drop
                out[mid] = [out[mid][i] for i in order]
                kk[mid] = [kk[mid][i] for i in order]
                audit["sunk"] += len(drop)
                audit["sunk_ids"].append(mid)
    return out, audit


def _uniq_fp_paths(inp):
    import glob
    from pathlib import Path

    def find(pattern):
        hits = sorted(glob.glob(str(inp) + '/**/' + pattern, recursive=True), key=len)
        if not hits:
            raise FileNotFoundError(pattern)
        return Path(hits[0])
    code = find('casmi26-v3-models/**/casmi/engine.py').parent.parent
    models = find('casmi26-v3-models/**/ranker_0.pkl').parent
    pool = find('pool_meta.parquet').parent
    fold0 = find('fpnet_fold0.pt')
    return str(code), str(models), str(fold0), str(pool)


def uniq_fp_evidence(request, lists, work, inp, test_path, cap_s, paths=None):
    """Run the embedded fingerprint runner in its own process. request: {mid: [smiles]}; returns ({mid: {smiles: gap}}, diag)."""
    import json, os, subprocess, sys, time
    from pathlib import Path
    work = Path(work).resolve()
    test_path = Path(test_path).resolve()
    src = _UNIQ_FP_RUNNER
    if src is None:
        src = (Path(__file__).with_name('uniq_fp_runner.py')).read_text()
    code = work / 'uniq_fp_runner.py'
    code.write_text(src)
    req, lst, out = work / 'uniq_fp_request.json', work / 'uniq_fp_lists.json', work / 'uniq_fp_out.json'
    req.write_text(json.dumps(request))
    lst.write_text(json.dumps({m: lists[m] for m in request}))
    if out.exists():
        out.unlink()
    code_dir, models, fold0, pool = paths or _uniq_fp_paths(inp)
    t0 = time.time()
    proc = subprocess.run([sys.executable, '-B', str(code), code_dir, models, fold0, pool, str(test_path), str(req), str(lst), str(out)],
                          capture_output=True, text=True, timeout=max(60.0, cap_s), cwd=str(work),
                          env={**os.environ, 'PYTHONHASHSEED': '0'})
    if proc.returncode != 0 or not out.exists():
        raise RuntimeError('fingerprint runner failed: ' + (proc.stdout[-300:] + proc.stderr[-600:]))
    res = json.loads(out.read_text())
    res['diag']['wall_s'] = round(time.time() - t0, 1)
    return res['evidence'], res['diag']


def run(work="/kaggle/working", inp="/kaggle/input", pipeline_done=False, enabled=True, cap_s=_UNIQ_CAP_S,
        tau_own=_UNIQ_TAU_OWN, delta=_UNIQ_DELTA, sink=_UNIQ_SINK, tau_sink=_UNIQ_TAU_SINK,
        delta_sink=_UNIQ_DELTA_SINK, min_match=_UNIQ_MIN_MATCH, max_refs=_UNIQ_MAX_REFS, rounds=_UNIQ_ROUNDS,
        test_path=None, train_path=None):
    import csv, io, json, os, time, uuid
    from pathlib import Path
    started = time.monotonic()
    audit = dict(version=_UNIQ_VERSION, status="started", tau_own=tau_own, delta=delta, sink=bool(sink),
                 tau_sink=tau_sink, delta_sink=delta_sink, min_match=min_match, cap_s=cap_s)
    target = Path(work) / "submission.csv"
    try:
        if not enabled:
            audit["status"] = "disabled"
        elif not pipeline_done:
            audit["status"] = "fast commit: original bytes kept"
        else:
            deadline = started + float(cap_s)
            before = target.read_bytes()
            lists = _uniq_parse(before)

            def locate(name, given):
                if given is not None:
                    return Path(given)
                files = sorted(Path(inp).rglob(name), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))
                if not files:
                    raise FileNotFoundError(name + " absent")
                return files[0]
            import pandas as pd
            test = pd.read_parquet(locate("test.parquet", test_path),
                                   columns=["molecule_id", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
            keys = _uniq_keys(lists)
            leaders = {k[0] for k in keys.values() if k and k[0] is not None}
            queries = _uniq_queries(test, set(lists))
            adducts = {a for q in queries.values() for a in q}
            refs = _uniq_references(locate("train.parquet", train_path), leaders, adducts, max_refs, deadline)
            audit.update(molecules=len(lists), with_spectra=len(queries), leaders=len(leaders),
                         leaders_with_references=len(refs), read_s=round(time.monotonic() - started, 1))
            memo = {}

            def evidence(mid, key):
                if key not in refs:
                    return None
                if (mid, key) not in memo:
                    memo[(mid, key)] = _uniq_evidence(queries.get(mid, {}), refs[key], min_match, deadline)
                return memo[(mid, key)]
            fp_ev = None
            if _UNIQ_FP:
                claims = {}
                for mid in lists:
                    if keys[mid] and keys[mid][0] is not None:
                        claims.setdefault(keys[mid][0], []).append(mid)
                involved = sorted({m for v in claims.values() if len(v) > 1 for m in v})
                if involved:
                    try:
                        request = {m: lists[m][:_UNIQ_FP_TOP] for m in involved}
                        left = max(60.0, min(_UNIQ_FP_CAP_S, deadline - time.monotonic() - 60))
                        gaps, fp_diag = uniq_fp_evidence(request, lists, work, inp, locate("test.parquet", test_path), left)
                        audit.update(fp_runner=fp_diag, fp_molecules=len(gaps))
                        fp_key = {}
                        for m, d in gaps.items():
                            for pos, smi in enumerate(lists[m][:_UNIQ_FP_TOP]):
                                if smi in d and keys[m][pos] is not None:
                                    fp_key[(m, keys[m][pos])] = float(d[smi])
                        fp_ev = lambda m, k: fp_key.get((m, k))
                    except Exception as error:
                        audit.update(fp_runner_error=repr(error)[:400])
            output, detail = uniq_resolve(lists, keys, evidence, tau_own, delta, sink, tau_sink, delta_sink, rounds,
                                          fp_evidence=fp_ev, delta_fp=_UNIQ_DELTA_FP)
            audit.update(detail)
            changed = sum(output[mid] != lists[mid] for mid in lists)
            audit.update(changed_lists=changed, top1_changed=sum(output[mid][0] != lists[mid][0] for mid in lists),
                         pairs_scored=len(memo))
            if time.monotonic() >= deadline:
                raise TimeoutError("UNIQ time budget exhausted")
            if changed:
                stream = io.StringIO(newline="")
                writer = csv.writer(stream, lineterminator="\n")
                writer.writerow(["molecule_id", "smiles"])
                writer.writerows((mid, ";".join(output[mid])) for mid in lists)
                after = stream.getvalue().encode("utf-8")
                verified = _uniq_parse(after)
                if list(verified) != list(lists) or any(sorted(verified[m]) != sorted(lists[m]) for m in lists):
                    raise ValueError("candidate permutation check failed")
                if target.read_bytes() != before:
                    raise RuntimeError("submission changed concurrently")
                backup = Path(work) / "submission_pre_uniq.csv"
                backup.write_bytes(before)
                temporary = target.with_name("submission.csv.uniq_" + uuid.uuid4().hex)
                with temporary.open("xb") as handle:
                    handle.write(after)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.replace(temporary, target)
            audit["status"] = "applied"
    except Exception as error:
        audit.update(status="failed closed: original bytes kept", exception=repr(error)[:500])
    audit["elapsed_s"] = round(time.monotonic() - started, 2)
    try:
        with (Path(work) / "uniq_audit.json").open("w", encoding="utf-8", newline="\n") as handle:
            json.dump(audit, handle, ensure_ascii=False, indent=2, default=str)
            handle.write("\n")
    except Exception:
        pass
    return audit


if globals().get("_UNIQ_NOTEBOOK", False) and "ROOT" in globals():
    import time as _uq_t, json as _uq_json
    _uq_left = 8.7 * 3600 - (_uq_t.time() - float(globals().get("NB_T0", _uq_t.time())))
    if bool(globals().get("PIPELINE_DONE")):
        if _uq_left < 300:
            print("[uniq] skipped: session time guard, %.0f s left" % _uq_left, flush=True)
        else:
            _uniq_audit = run(work=str(ROOT), inp=str(INPUT), pipeline_done=True, enabled=_UNIQ_ON,
                              cap_s=min(_UNIQ_CAP_S, _uq_left - 180))
            print("[uniq]", _uq_json.dumps({k: v for k, v in _uniq_audit.items() if k not in ("demoted_ids", "sunk_ids", "demoted_fp_ids")}), flush=True)
    else:
        # commit run: logic self-test and an input/output dry run on the visible files (submission.csv is not touched)
        try:
            _l = {"a": ["X", "Y", "Z"], "b": ["X", "Y", "W"], "c": ["Q", "X", "R"]}
            _e = {("a", "X"): 0.95, ("b", "X"): 0.70, ("c", "X"): 0.20, ("a", "Y"): 0.1, ("b", "Y"): 0.1, ("c", "Q"): 0.3}
            _o, _a = uniq_resolve(_l, {m: list(v) for m, v in _l.items()}, lambda m, k: _e.get((m, k)))
            assert _o == {"a": ["X", "Y", "Z"], "b": ["Y", "X", "W"], "c": ["Q", "R", "X"]}, _o
            _l2 = {"a": ["X", "Y"], "b": ["X", "Z"], "c": ["P", "Q"], "d": ["P", "R"]}
            _f2 = {("a", "X"): 1.0, ("b", "X"): 0.2, ("c", "P"): 0.50, ("d", "P"): 0.45}
            _o2, _a2 = uniq_resolve(_l2, {m: list(v) for m, v in _l2.items()}, lambda m, k: None, sink=False,
                                    fp_evidence=lambda m, k: _f2.get((m, k)), delta_fp=0.10)
            assert _o2 == {"a": ["X", "Y"], "b": ["Z", "X"], "c": ["P", "Q"], "d": ["P", "R"]}, _o2
            import pandas as _uq_pd, pyarrow.parquet as _uq_pq
            _t0 = _uq_t.monotonic()
            _train = sorted(INPUT.rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            _test = _uq_pd.read_parquet(COMP / "test.parquet", columns=["molecule_id", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
            _q = _uniq_queries(_test, set(_test.molecule_id.astype(str)))
            _keys = _uq_pq.ParquetFile(_train).read_row_group(0, columns=["inchikey14"]).column(0).to_pylist()
            _keys = sorted(set(k for k in _keys if isinstance(k, str)))[:300]
            _r = _uniq_references(_train, set(_keys), set(_test.adduct.astype(str)), _UNIQ_MAX_REFS, _uq_t.monotonic() + 600)
            _m = sorted(_q)[:20]
            _s = [_uniq_evidence(_q[m], _r[k], _UNIQ_MIN_MATCH, _uq_t.monotonic() + 300) for m in _m for k in list(_r)[:5]]
            print("[uniq] commit dry run ok: self-test passed | queries %d | references for %d of %d keys | %d pairs scored | %.1f s"
                  % (len(_q), len(_r), len(_keys), len(_s), _uq_t.monotonic() - _t0), flush=True)
            if _UNIQ_FP:
                _mids = sorted(_q)[:6]
                _req = {m: ["CCO", "OCC(O)CO", "c1ccccc1O"] for m in _mids}
                _gaps, _fd = uniq_fp_evidence(_req, {m: [] for m in _mids}, ROOT, INPUT, COMP / "test.parquet", 900)
                assert len(_gaps) == len(_mids) and all(len(v) == 3 for v in _gaps.values()), _gaps
                print("[uniq] fingerprint runner dry run ok:", _fd, flush=True)
        except Exception as _uq_exc:
            print("[uniq] commit dry run FAILED:", repr(_uq_exc)[:400], flush=True)


In [ ]:
vh_snap('g_uniq')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== ours: NPX - learned natural-product likeness for candidates that no table knows =====
# E180X demotes candidates that are listed in the train file and absent from every natural-product table. Most of a
# list, though, comes from PubChem and is in no table at all, so E180X cannot tell a synthetic-looking PubChem isomer
# from a natural-looking one. NPX learns that difference from the same tables E180X reads: natural products (COCONUT /
# LOTUS / NPAtlas structures of natural_products.parquet) against the enveda-180 synthetic library (182,941 structures,
# 4 of them LOTUS). Features are stereo-free (Morgan counts, Fsp3, rings, heteroatoms, halogens), because train SMILES
# and PubChem SMILES carry stereo differently.
#   ranks 2-25: score = -1.5 ln(rank) + 0.35 * clip(logit p_np, -4, 4), stable sort (the APRIOR form).
#   rank 1: moves only when the leader looks synthetic (p_np < 0.02) and is in no natural-product table, while one of
#           ranks 2-5 is a table natural product with p_np > 0.6; that one becomes rank 1, the leader rank 2.
# Gate: the model must separate held-out molecules (AUC >= 0.9) and score the 250 enveda-np-examples answers as natural
# (median p_np >= 0.6), otherwise nothing changes. Runs after UNIQ and before E180X. Fails closed.
_NPX_ON = True
_NPX_RANK1 = True
_NPX_LAM, _NPX_GAMMA, _NPX_CLIP = 0.35, 1.5, 4.0
_NPX_LEAD_MAX, _NPX_ALT_MIN, _NPX_ALT_DEPTH = 0.02, 0.6, 5
_NPX_N = 25000              # per class
_NPX_CAP_S = 1200.0
_NPX_VERSION = "1.0.0"


def npx_featurizer():
    import numpy as np
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, Descriptors, rdMolDescriptors
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    HAL = {"F", "Cl", "Br", "I"}

    def feats(smi):
        try:
            m = Chem.MolFromSmiles(smi)
            if m is None:
                return None
            Chem.RemoveStereochemistry(m)
            atoms = [a.GetSymbol() for a in m.GetAtoms()]
            heavy = max(1, m.GetNumHeavyAtoms())
            nC = atoms.count("C")
            d = [Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m), rdMolDescriptors.CalcFractionCSP3(m),
                 rdMolDescriptors.CalcNumRings(m), rdMolDescriptors.CalcNumAromaticRings(m),
                 rdMolDescriptors.CalcNumAliphaticRings(m), rdMolDescriptors.CalcNumHBD(m), rdMolDescriptors.CalcNumHBA(m),
                 rdMolDescriptors.CalcNumAmideBonds(m), rdMolDescriptors.CalcNumRotatableBonds(m), heavy,
                 atoms.count("N") / heavy, atoms.count("O") / heavy, atoms.count("S"), sum(a in HAL for a in atoms),
                 atoms.count("O") / max(1, nC), atoms.count("N") / max(1, nC),
                 len(Chem.FindMolChiralCenters(m, includeUnassigned=True, useLegacyImplementation=False))]
            fp = np.asarray(gen.GetCountFingerprintAsNumPy(m), dtype=np.float32)
            return np.concatenate([fp, np.asarray(d, np.float32)])
        except Exception:
            return None
    return feats


def npx_train(train_path, inp, n=_NPX_N, deadline=None, seed=0):
    """Model of p(natural product | structure), with its gate statistics."""
    import time
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    from pathlib import Path
    t0 = time.monotonic()
    npt = sorted(Path(inp).rglob("natural_products.parquet"), key=lambda p: len(str(p)))
    if not npt:
        raise FileNotFoundError("natural_products.parquet not found")
    nat = pd.read_parquet(npt[0], columns=["ik14", "smiles"]).dropna().drop_duplicates("ik14")
    tr = pq.read_table(train_path, columns=["ingest_lib", "inchikey14", "normalized_smiles"]).to_pandas()
    tr = tr[tr["ingest_lib"].isin(["enveda-180", "enveda-np-examples"])].drop_duplicates(["ingest_lib", "inchikey14"])
    syn = tr[(tr["ingest_lib"] == "enveda-180") & ~tr["inchikey14"].isin(set(nat["ik14"]))]
    ex = tr[tr["ingest_lib"] == "enveda-np-examples"]
    nat = nat[~nat["ik14"].isin(set(ex["inchikey14"]))]           # the gate's examples stay unseen
    rng = np.random.default_rng(seed)
    pos = nat.sample(min(n + 2000, len(nat)), random_state=seed)["smiles"].tolist()
    neg = syn.sample(min(n + 2000, len(syn)), random_state=seed)["normalized_smiles"].tolist()
    feats = npx_featurizer()
    X, y = [], []
    for lab, group in ((1, pos), (0, neg)):
        for s in group:
            f = feats(s)
            if f is not None:
                X.append(f); y.append(lab)
            if deadline is not None and len(X) % 2000 == 0 and time.monotonic() > deadline:
                raise TimeoutError("NPX featurisation over budget")
    X, y = np.stack(X), np.asarray(y)
    perm = rng.permutation(len(y))
    X, y = X[perm], y[perm]
    hold = min(4000, len(y) // 5)
    import lightgbm as lgb
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.5, random_state=seed,
                               n_jobs=4, verbose=-1).fit(X[hold:], y[hold:])
    from sklearn.metrics import roc_auc_score
    auc = float(roc_auc_score(y[:hold], model.predict_proba(X[:hold])[:, 1]))
    exf = [f for f in (feats(s) for s in ex["normalized_smiles"]) if f is not None]
    ex_p = model.predict_proba(np.stack(exf))[:, 1] if exf else np.array([])
    info = dict(train=int(len(y) - hold), auc_holdout=round(auc, 4), np_examples=int(len(ex_p)),
                np_examples_median_p=round(float(np.median(ex_p)), 3) if len(ex_p) else None,
                np_examples_share_p_gt_0_5=round(float((ex_p > 0.5).mean()), 3) if len(ex_p) else None,
                train_s=round(time.monotonic() - t0, 1))
    info["gate_ok"] = bool(auc >= 0.9 and len(ex_p) >= 50 and float(np.median(ex_p)) >= 0.6)
    table = set(nat["ik14"]) | set(ex["inchikey14"])
    pool = sorted(Path(inp).rglob("pool_meta.parquet"), key=lambda p: len(str(p)))
    if pool:                                                       # COCONUT keys, as E180X reads them
        pm = pq.read_table(pool[0], columns=["key", "src"]).to_pandas()
        table |= set(pm.loc[pm["src"] >= 2, "key"])
    info["table_keys"] = len(table)
    return model, feats, table, info


def npx_reorder(cands, p, in_table, lam=_NPX_LAM, gamma=_NPX_GAMMA, rank1=_NPX_RANK1):
    """p: p_np per candidate (None = unknown). in_table: bool per candidate. Returns the new order."""
    import math
    n = len(cands)
    if n < 2:
        return list(cands)

    def logit(x):
        if x is None:
            return 0.0
        x = min(max(x, 1e-6), 1 - 1e-6)
        return max(-_NPX_CLIP, min(_NPX_CLIP, math.log(x / (1 - x))))
    order = list(range(n))
    if rank1 and p[0] is not None and p[0] < _NPX_LEAD_MAX and not in_table[0]:
        alt = [i for i in range(1, min(_NPX_ALT_DEPTH, n)) if in_table[i] and p[i] is not None and p[i] > _NPX_ALT_MIN]
        if alt:
            j = alt[0]
            order = [j, 0] + [i for i in range(1, n) if i != j]
    head, rest = order[0], order[1:]
    rest = sorted(rest, key=lambda i: (-(-gamma * math.log(i + 1) + lam * logit(p[i])), i))
    return [cands[head]] + [cands[i] for i in rest]


def npx_run(work, inp, pipeline_done, enabled=True, cap_s=_NPX_CAP_S):
    import csv, io, json, os, time, uuid
    from pathlib import Path
    started = time.monotonic()
    audit = dict(version=_NPX_VERSION, status="started", lam=_NPX_LAM, rank1=_NPX_RANK1)
    target = Path(work) / "submission.csv"
    try:
        if not enabled:
            audit["status"] = "disabled"
        elif not pipeline_done:
            audit["status"] = "fast commit: original bytes kept"
        else:
            deadline = started + float(cap_s)
            before = target.read_bytes()
            rows = list(csv.reader(io.StringIO(before.decode("utf-8"), newline="")))
            if not rows or rows[0] != ["molecule_id", "smiles"]:
                raise ValueError("invalid submission header")
            lists = {r[0]: r[1].split(";") for r in rows[1:]}
            train = sorted(Path(inp).rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            model, feats, table, info = npx_train(train, inp, deadline=started + 0.7 * float(cap_s))
            audit.update(info)
            if not info["gate_ok"]:
                audit["status"] = "gate failed: original bytes kept"
            else:
                import numpy as np
                key = ex_keyer() if "ex_keyer" in globals() else None
                uniq = sorted({s for c in lists.values() for s in c})
                F = {}
                for s in uniq:
                    if time.monotonic() > deadline:
                        raise TimeoutError("NPX scoring over budget")
                    F[s] = feats(s)
                ok = [s for s in uniq if F[s] is not None]
                P = dict(zip(ok, model.predict_proba(np.stack([F[s] for s in ok]))[:, 1])) if ok else {}
                if key is None:
                    from rdkit import Chem
                    def key(s):
                        try:
                            m = Chem.MolFromSmiles(s); return Chem.MolToInchiKey(m)[:14] if m else None
                        except Exception:
                            return None
                output = {}
                for mid, cands in lists.items():
                    p = [None if s not in P else float(P[s]) for s in cands]
                    tab = [key(s) in table for s in cands]
                    output[mid] = npx_reorder(cands, p, tab)
                changed = sum(output[m] != lists[m] for m in lists)
                top1 = sum(output[m][0] != lists[m][0] for m in lists)
                audit.update(molecules=len(lists), changed_lists=changed, top1_changed=top1,
                             scored=len(P), unscored=len(uniq) - len(P))
                if changed:
                    stream = io.StringIO(newline="")
                    w = csv.writer(stream, lineterminator="\n")
                    w.writerow(["molecule_id", "smiles"])
                    w.writerows((m, ";".join(output[m])) for m in lists)
                    after = stream.getvalue().encode("utf-8")
                    check = {r[0]: r[1].split(";") for r in list(csv.reader(io.StringIO(after.decode("utf-8"), newline="")))[1:]}
                    if list(check) != list(lists) or any(sorted(check[m]) != sorted(lists[m]) for m in lists):
                        raise ValueError("candidate permutation check failed")
                    if target.read_bytes() != before:
                        raise RuntimeError("submission changed concurrently")
                    (Path(work) / "submission_pre_npx.csv").write_bytes(before)
                    tmp = target.with_name("submission.csv.npx_" + uuid.uuid4().hex)
                    with tmp.open("xb") as h:
                        h.write(after); h.flush(); os.fsync(h.fileno())
                    os.replace(tmp, target)
                audit["status"] = "applied"
    except Exception as error:
        audit.update(status="failed closed: original bytes kept", exception=repr(error)[:500])
    audit["elapsed_s"] = round(time.monotonic() - started, 1)
    try:
        (Path(work) / "npx_audit.json").write_text(json.dumps(audit, indent=2, default=str) + "\n")
    except Exception:
        pass
    return audit


if "ROOT" in globals():
    import time as _npx_t, json as _npx_json
    _npx_left = 8.7 * 3600 - (_npx_t.time() - float(globals().get("NB_T0", _npx_t.time())))
    if bool(globals().get("PIPELINE_DONE")):
        if _npx_left < 600:
            print("[npx] skipped: session time guard, %.0f s left" % _npx_left, flush=True)
        else:
            _npx_audit = npx_run(str(ROOT), str(INPUT), True, enabled=_NPX_ON, cap_s=min(_NPX_CAP_S, _npx_left - 400))
            print("[npx]", _npx_json.dumps(_npx_audit), flush=True)
    else:
        try:   # commit: self-test of the re-order, then train on a small sample and report the gate statistics
            _t0 = _npx_t.monotonic()
            assert npx_reorder(list("abc"), [0.5, 0.5, 0.5], [False] * 3) == list("abc")
            assert npx_reorder(list("abcd"), [0.01, 0.1, 0.9, 0.5], [False, True, True, False]) == list("cabd")
            assert npx_reorder(list("abcd"), [0.01, 0.1, 0.9, 0.5], [True, True, True, False])[0] == "a"
            assert npx_reorder(list("abc"), [0.9, 0.01, 0.99], [False] * 3) == list("acb")
            _tr = sorted(INPUT.rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            _m, _f, _tab, _info = npx_train(_tr, INPUT, n=3000)
            print("[npx] commit dry run ok:", _npx_json.dumps(_info), "| %.1f s" % (_npx_t.monotonic() - _t0), flush=True)
        except Exception as _e:
            print("[npx] commit dry run FAILED:", repr(_e)[:400], flush=True)


In [ ]:
vh_snap('h_npx')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== Ours: E180X 1.1 - synthetic-library structures are not the answers of a natural-product test =====
# The hidden test comes from biological extracts. The train set's enveda-180 library (182,941 structures, same timsTOF
# campaign) is a synthetic-compound library: it holds 4 of the 151,885 LOTUS natural products, 46 COCONUT structures and
# none of the 250 enveda-np-examples, while all 250 examples have public spectra in at least 2 other train libraries and
# 233 of them are COCONUT structures. This cell moves candidates to the end of their list, order otherwise unchanged,
# nothing dropped:
#   tier A - enveda-180 structures that are not natural products (not COCONUT / LOTUS / NPAtlas), and (new in 1.1)
#            non-natural-product structures recorded ONLY by pluskal_ms2 (MSnLib, commercial screening libraries) or
#            ONLY by drug_plus: every rank, rank 1 too;
#   tier B - other train-only structures that are not natural products: ranks 2-25 only.
#   new in 1.1, common-compound exemption: a structure recorded by >= 2 public libraries other than enveda-180 is never
#            moved. All 250 examples have >= 2 such libraries; 8 of them are not natural-product-table structures
#            (steroids, rapalogs, ranitidine...) and E180X 1.0 pushed them down when they sat on ranks 2-25.
# Runs last, after APRIOR and UNIQ. Fails closed: any error keeps the original file.
_EX_ON = True
_EX_TIER_B = True
_EX_CAP_S = 900.0
_EX_VERSION = "1.1.0"
_EX_COMMON_MIN = 2          # libraries other than enveda-180 / np-examples
_EX_SOLO_A = ("pluskal_ms2", "drug_plus")


def ex_sets(train_path, inp, deadline=None):
    """(tier A keys, tier B keys, info): InChIKey-14 sets from the train file minus natural-product keys."""
    import time
    from pathlib import Path
    import pyarrow.parquet as pq
    t0 = time.monotonic()
    tbl = pq.read_table(train_path, columns=["ingest_lib", "inchikey14"]).to_pandas().drop_duplicates()
    e180 = set(tbl.loc[tbl["ingest_lib"] == "enveda-180", "inchikey14"].dropna())
    other = set(tbl.loc[~tbl["ingest_lib"].isin(["enveda-180", "enveda-np-examples"]), "inchikey14"].dropna())
    nat, found = set(), []

    def find(name):
        hits = sorted(Path(inp).rglob(name), key=lambda p: len(str(p)))
        return hits[0] if hits else None
    pool = find("pool_meta.parquet")
    if pool is not None:
        pm = pq.read_table(pool, columns=["key", "src"]).to_pandas()
        nat |= set(pm.loc[pm["src"] >= 2, "key"])          # src 2: COCONUT only, 3: COCONUT and train
        found.append("pool_meta")
    npt = find("natural_products.parquet")
    if npt is not None:
        cols = pq.read_schema(npt).names
        df = pq.read_table(npt, columns=[c for c in ("ik14", "scorer_key") if c in cols]).to_pandas()
        for c in df.columns:
            nat |= set(df[c].dropna())
        found.append("natural_products")
    if not found:
        raise FileNotFoundError("no natural-product key table found")
    if deadline is not None and time.monotonic() > deadline:
        raise TimeoutError("E180X sets over budget")
    o = tbl.loc[~tbl["ingest_lib"].isin(["enveda-180", "enveda-np-examples"])].dropna(subset=["inchikey14"])
    nlib = o.groupby("inchikey14")["ingest_lib"].nunique()
    common = set(nlib.index[nlib >= _EX_COMMON_MIN])
    solo = set(o.loc[o["ingest_lib"].isin(_EX_SOLO_A) & o["inchikey14"].isin(set(nlib.index[nlib == 1])), "inchikey14"])
    tier_a = ((e180 - nat) | ((solo - e180) - nat)) - common
    tier_b = ((other - e180) - nat) - common - tier_a
    info = dict(e180=len(e180), other=len(other), natural=len(nat), tier_a=len(tier_a), tier_b=len(tier_b),
                common_exempt=len(common - nat), solo_to_a=len((solo - e180) - nat), tables=found,
                sets_s=round(time.monotonic() - t0, 1))
    return tier_a, tier_b, info


def ex_keyer():
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog("rdApp.*")
    cache = {}

    def key(smi):
        if smi not in cache:
            k = None
            try:
                m = Chem.MolFromSmiles(smi)
                if m is not None:
                    k = Chem.MolToInchiKey(m)[:14] or None
            except Exception:
                k = None
            cache[smi] = k
        return cache[smi]
    return key


def ex_reorder(cands, keys, tier_a, tier_b, use_b=True):
    """Stable: unflagged, then tier B, then tier A. Rank 1 moves only for tier A; tier B is judged on ranks 2-25."""
    a = [k is not None and k in tier_a for k in keys]
    b = [use_b and k is not None and k not in tier_a and k in tier_b for k in keys]
    if not any(a) and not any(b):
        return list(cands)
    idx = list(range(len(cands)))
    head = [] if a[0] else [0]
    rest = idx[1:] if head else idx
    order = head + [i for i in rest if not a[i] and not b[i]] + [i for i in rest if b[i]] + [i for i in rest if a[i]]
    return [cands[i] for i in order]


def ex_run(work, inp, pipeline_done, enabled=True, cap_s=_EX_CAP_S):
    import csv, io, json, os, time, uuid
    from pathlib import Path
    started = time.monotonic()
    audit = dict(version=_EX_VERSION, status="started", use_tier_b=_EX_TIER_B)
    target = Path(work) / "submission.csv"
    try:
        if not enabled:
            audit["status"] = "disabled"
        elif not pipeline_done:
            audit["status"] = "fast commit: original bytes kept"
        else:
            deadline = started + float(cap_s)
            before = target.read_bytes()
            rows = list(csv.reader(io.StringIO(before.decode("utf-8"), newline="")))
            if not rows or rows[0] != ["molecule_id", "smiles"]:
                raise ValueError("invalid submission header")
            lists = {r[0]: r[1].split(";") for r in rows[1:]}
            train = sorted(Path(inp).rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            tier_a, tier_b, info = ex_sets(train, inp, deadline)
            audit.update(info)
            key = ex_keyer()
            output, stats = {}, dict(lead_a=0, top5_a=0, top5_b=0, flagged_a=0, flagged_b=0, unkeyed=0)
            for mid, cands in lists.items():
                if time.monotonic() > deadline:
                    raise TimeoutError("E180X time budget exhausted")
                keys = [key(s) for s in cands]
                stats["unkeyed"] += sum(k is None for k in keys)
                fa = [k is not None and k in tier_a for k in keys]
                fb = [k is not None and k not in tier_a and k in tier_b for k in keys]
                stats["lead_a"] += bool(fa and fa[0])
                stats["top5_a"] += any(fa[:5])
                stats["top5_b"] += any(fb[1:5])
                stats["flagged_a"] += sum(fa)
                stats["flagged_b"] += sum(fb[1:])
                output[mid] = ex_reorder(cands, keys, tier_a, tier_b, _EX_TIER_B)
            changed = sum(output[m] != lists[m] for m in lists)
            top1 = sum(output[m][0] != lists[m][0] for m in lists)
            audit.update(stats, molecules=len(lists), changed_lists=changed, top1_changed=top1)
            if changed:
                stream = io.StringIO(newline="")
                writer = csv.writer(stream, lineterminator="\n")
                writer.writerow(["molecule_id", "smiles"])
                writer.writerows((m, ";".join(output[m])) for m in lists)
                after = stream.getvalue().encode("utf-8")
                check = {r[0]: r[1].split(";") for r in list(csv.reader(io.StringIO(after.decode("utf-8"), newline="")))[1:]}
                if list(check) != list(lists) or any(sorted(check[m]) != sorted(lists[m]) for m in lists):
                    raise ValueError("candidate permutation check failed")
                if target.read_bytes() != before:
                    raise RuntimeError("submission changed concurrently")
                (Path(work) / "submission_pre_e180x.csv").write_bytes(before)
                tmp = target.with_name("submission.csv.e180x_" + uuid.uuid4().hex)
                with tmp.open("xb") as handle:
                    handle.write(after)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.replace(tmp, target)
            audit["status"] = "applied"
    except Exception as error:
        audit.update(status="failed closed: original bytes kept", exception=repr(error)[:500])
    audit["elapsed_s"] = round(time.monotonic() - started, 1)
    try:
        (Path(work) / "e180x_audit.json").write_text(json.dumps(audit, indent=2, default=str) + "\n")
    except Exception:
        pass
    return audit


if "ROOT" in globals():
    import time as _ex_t, json as _ex_json
    _ex_left = 8.7 * 3600 - (_ex_t.time() - float(globals().get("NB_T0", _ex_t.time())))
    if bool(globals().get("PIPELINE_DONE")):
        if _ex_left < 240:
            print("[e180x] skipped: session time guard, %.0f s left" % _ex_left, flush=True)
        else:
            _ex_audit = ex_run(str(ROOT), str(INPUT), True, enabled=_EX_ON, cap_s=min(_EX_CAP_S, _ex_left - 120))
            print("[e180x]", _ex_json.dumps(_ex_audit), flush=True)
    else:
        # commit run: self-test, the key sets, and a dry run on the visible test file (submission.csv is not touched)
        try:
            _ex_t0 = _ex_t.monotonic()
            _A, _B = {"KA"}, {"KB"}
            assert ex_reorder(list("pqrs"), ["KA", "x", "KB", "y"], _A, _B) == ["q", "s", "r", "p"]
            assert ex_reorder(list("pqrs"), ["KB", "KA", "x", "y"], _A, _B) == ["p", "r", "s", "q"]
            assert ex_reorder(list("pqrs"), ["KB", "KA", "x", "y"], _A, _B, use_b=False) == ["p", "r", "s", "q"]
            assert ex_reorder(list("pqr"), ["x", "KB", "y"], _A, _B, use_b=False) == ["p", "q", "r"]
            assert ex_reorder(list("pq"), ["KA", "KA"], _A, _B) == ["p", "q"]
            assert ex_reorder(list("pq"), [None, "x"], _A, _B) == ["p", "q"]
            _ex_train = sorted(INPUT.rglob("train.parquet"), key=lambda p: ("mass-spectra" not in str(p).lower(), len(str(p))))[0]
            _ex_a, _ex_b, _ex_info = ex_sets(_ex_train, INPUT, _ex_t.monotonic() + 600)
            assert _ex_info["tier_a"] > 100000 and "pool_meta" in _ex_info["tables"], _ex_info
            _ex_key = ex_keyer()
            assert _ex_key("CC(=O)Oc1ccccc1C(=O)O") == "BSYNRYMUTXBXSQ", _ex_key("CC(=O)Oc1ccccc1C(=O)O")
            import pandas as _ex_pd
            _ex_sm = _ex_pd.read_csv(COMP / "sample_submission.csv", nrows=40) if (COMP / "sample_submission.csv").is_file() else None
            _ex_n = 0
            if _ex_sm is not None and "smiles" in _ex_sm:
                for _s in _ex_sm["smiles"].astype(str):
                    _c = _s.split(";")[:25]
                    _ex_n += len(ex_reorder(_c, [_ex_key(x) for x in _c], _ex_a, _ex_b)) == len(_c)
            print("[e180x] commit dry run ok:", _ex_json.dumps(_ex_info), "| %d sample lists re-ordered | %.1f s"
                  % (_ex_n, _ex_t.monotonic() - _ex_t0), flush=True)
        except Exception as _ex_exc:
            print("[e180x] commit dry run FAILED:", repr(_ex_exc)[:400], flush=True)


In [ ]:
vh_snap('i_e180x11')  # VH: copy of submission.csv at this stage boundary


In [ ]:
# ===== VH: score every stage snapshot per group (metric key = InChIKey-14 after RDKit tautomer canonicalisation) =====
try:
    import time as _sc_t
    if _sc_t.time() - float(globals().get('NB_T0', _sc_t.time())) > 8.0 * 3600:
        raise TimeoutError('session past 8.0 h: scoring skipped, lists are saved')
    import csv as _sc_csv, json as _sc_json
    from pathlib import Path as _SP
    import pandas as _sc_pd
    from rdkit import Chem as _SC, RDLogger as _SL
    from rdkit.Chem.MolStandardize import rdMolStandardize as _SM
    _SL.DisableLog('rdApp.*')
    _te = _SM.TautomerEnumerator(); _kc = {}

    def _sc_key(s):
        if s not in _kc:
            try:
                m = _SC.MolFromSmiles(s)
                _kc[s] = _SC.MolToInchiKey(_te.Canonicalize(m))[:14] if m is not None else None
            except Exception:
                _kc[s] = None
        return _kc[s]
    _ans = _sc_pd.read_csv(VH_OUT / 'answers.csv', dtype=str)
    _truth = {r.molecule_id: _sc_key(r.smiles) for r in _ans.itertuples()}
    _grp = dict(zip(_ans.molecule_id, _ans.group))
    _res = {}
    for _f in [p for p in sorted(VH_OUT.glob('sub_*.csv')) + [_SP('/kaggle/working/submission.csv')] if p.is_file()]:
        _rr = {g: [] for g in sorted(set(_grp.values()))}
        for _r in _sc_csv.DictReader(open(_f, encoding='utf-8')):
            _ks = [_sc_key(x) for x in _r['smiles'].split(';')[:25]]
            _t = _truth.get(_r['molecule_id'])
            _rr[_grp[_r['molecule_id']]].append(1.0 / (_ks.index(_t) + 1) if _t in _ks else 0.0)
        _res[_f.name] = {g: dict(n=len(v), mrr=round(sum(v) / max(1, len(v)), 4), top1=sum(x == 1 for x in v)) for g, v in _rr.items()}
        print('[vh] score', _f.name, _sc_json.dumps(_res[_f.name]), flush=True)
    # the one-sided test: duel and gated MAINLEAD replayed on the final list, per group (main engine = fpnet_fold0 if VH_CLEAN_MAIN)
    import shutil as _sc_sh
    _evp = _SP('/kaggle/working/frozen_candidate_evidence.parquet')
    if _evp.is_file():
        _sc_sh.copy2(_evp, VH_OUT / 'frozen_candidate_evidence.parquet')
        from rdkit.Chem import rdMolDescriptors as _SD
        _ev = _sc_pd.read_parquet(_evp, columns=['molecule_id', 'smiles', 'main_rank', 'library_max'])
        _main1 = _ev[_ev.main_rank == 1].drop_duplicates('molecule_id').set_index('molecule_id').smiles.to_dict()
        _lib = _ev.groupby('molecule_id').library_max.first().to_dict()
        _fc = {}

        def _form(s):
            if s not in _fc:
                m = _SC.MolFromSmiles(s); _fc[s] = _SD.CalcMolFormula(m) if m is not None else None
            return _fc[s]

        def _duel(mid, c):
            if len(c) > 1 and _main1.get(mid) == c[1] and _form(c[0]) == _form(c[1]):
                return [c[1], c[0]] + c[2:]
            return c

        def _mlg(mid, c):
            k = _main1.get(mid)
            if k in c[1:] and _lib.get(mid, 1.0) < 0.7 and _form(k) == _form(c[0]):
                return [k] + [x for x in c if x != k]
            return c
        _fin = [p for p in [_SP('/kaggle/working/submission.csv')] + sorted(VH_OUT.glob('sub_i_*.csv')) if p.is_file()][0]
        _rule = {}
        for _name, _fn in (('duel', _duel), ('mainlead_sameformula_lib07', _mlg), ('mainlead_all', lambda m, c: ([_main1[m]] + [x for x in c if x != _main1[m]]) if _main1.get(m) in c[1:] else c)):
            _agg = {}
            for _r in _sc_csv.DictReader(open(_fin, encoding='utf-8')):
                _m = _r['molecule_id']; _c = _r['smiles'].split(';')[:25]; _n = _fn(_m, _c); _t = _truth.get(_m)
                _ks0 = [_sc_key(x) for x in _c]; _ks1 = [_sc_key(x) for x in _n]
                _a = 1.0 / (_ks0.index(_t) + 1) if _t in _ks0 else 0.0; _b = 1.0 / (_ks1.index(_t) + 1) if _t in _ks1 else 0.0
                g = _agg.setdefault(_grp[_m], dict(n=0, mrr0=0.0, mrr1=0.0, up=0, down=0, changed=0))
                g['n'] += 1; g['mrr0'] += _a; g['mrr1'] += _b; g['up'] += _b > _a; g['down'] += _b < _a; g['changed'] += _n != _c
            for g in _agg.values():
                g['mrr0'] = round(g['mrr0'] / g['n'], 4); g['mrr1'] = round(g['mrr1'] / g['n'], 4); g['delta'] = round(g['mrr1'] - g['mrr0'], 4)
            _rule[_name] = _agg
            print('[vh] rule', _name, _sc_json.dumps(_agg), flush=True)
        _main_acc = {}
        for _m, _s in _main1.items():
            g = _main_acc.setdefault(_grp.get(_m, '?'), [0, 0]); g[0] += 1; g[1] += _sc_key(_s) == _truth.get(_m)
        _res['_rules_on_final'] = dict(file=_fin.name, rules=_rule, main_engine_leader_correct={k: f'{v[1]}/{v[0]}' for k, v in _main_acc.items()},
                                       clean_main=__import__('os').environ.get('VH_CLEAN_MAIN'))
        print('[vh] main-engine leader correct', _sc_json.dumps(_res['_rules_on_final']['main_engine_leader_correct']), flush=True)
    (VH_OUT / 'scores.json').write_text(_sc_json.dumps(_res, indent=1))
except Exception as _sc_exc:
    print('[vh] scoring failed (lists are saved; score offline):', repr(_sc_exc)[:300], flush=True)


In [ ]:
# ===== VH: output hygiene - keep vh/, small top-level files and audits; delete work and temp folders so the output lists in one page =====
try:
    import shutil as _cl_sh
    from pathlib import Path as _CP
    _keep_dirs = {'vh'}
    _w = _CP('/kaggle/working')
    _removed = []
    for _p in sorted(_w.iterdir()):
        if _p.is_dir() and _p.name not in _keep_dirs:
            _cl_sh.rmtree(_p, ignore_errors=True); _removed.append(_p.name + '/')
        elif _p.is_file() and (_p.stat().st_size > 50 * 1024 * 1024 or _p.suffix not in ('.csv', '.json', '.parquet', '.log', '.txt')):
            _p.unlink(); _removed.append(_p.name)
    _cl_sh.rmtree('/tmp/enveda-casmi26-vh2-mass-spectra', ignore_errors=True)
    _left = [p for p in _w.rglob('*') if p.is_file()]
    print('[vh] cleanup: removed', len(_removed), 'entries; files left', len(_left), flush=True)
except Exception as _cl_e:
    print('[vh] cleanup failed:', repr(_cl_e)[:300], flush=True)
